# PowerGenome Settings Builder for Switch-USA-PG-ReEDS
**Melek Ben-Ayed**

---
## What this notebook does


The [PowerGenome GUI](https://gschivley.github.io/PowerGenome-tools/web/) is a convenient way to design an energy system model (selecting regions, technologies, fuels, and policies) and export YAML settings files. However, the GUI does not produce all the files needed to run [`pg_to_switch.py`](https://github.com/switch-model/Switch-USA-PG-ReEDS/blob/main/pg_to_switch.py) in [Switch-USA-PG-ReEDS](https://github.com/switch-model/Switch-USA-PG-ReEDS).

This notebook bridges that gap. It:
1. **Reads** the GUI-produced settings from `pg/powergenome_settings/settings/`
2. **Walks through** each missing or incomplete settings file, explaining what every parameter controls
3. **Writes** a complete, ready-to-use settings folder to a new directory (`pg/my_settings/`) without modifying the GUI output

## How PowerGenome settings work

PowerGenome doesn't care about filenames. It loads every `.yml` file in a settings folder, sorts them alphabetically, and merges them into one big dictionary. If the same key appears in two files, the last file (alphabetically) wins. The split into separate files (`demand.yml`, `fuels.yml`, etc.) is purely for our "human" organization.

For `pg_to_switch.py` to run, the merged dictionary must contain all the keys PowerGenome expects: database paths, region definitions, generator clusters, fuel prices, demand profiles, transmission constraints, and more.

## Files the GUI produces (7 files)


| File | Controls |
|---|---|
| `model_definition.yml` | Planning years, target USD year, model regions, region aggregations |
| `resources.yml` | New-build technologies (from ATB), existing generator clustering |
| `fuels.yml` | Fuel price scenarios, emission factors |
| `transmission.yml` | Inter-regional transmission limits, expansion costs, losses |
| `distributed_gen.yml` | Rooftop solar profiles and capacity (from NREL Cambium) |
| `startup_costs.yml` | Generator startup cost parameters |
| `resource_tags.yml` | Dispatch behavior tags (THERM, VRE, STOR, HYDRO, etc.) |



## Files missing from GUI output (8 files)


| File | Controls | Why it matters |
|---|---|---|
| `env.yml` | Database paths (PUDL, PG), data file locations | Without this, PowerGenome can't find any data |
| `demand.yml` | Load profiles, demand growth, electrification | No electricity demand = no model |
| `scenario_management.yml` | Multi-scenario definitions, parameter swaps | Defines what cases actually get run |
| `switch.yml` | Switch-model-specific parameters | Glue between PowerGenome and Switch |
| `extra_inputs.yml` | Paths to supplementary CSVs (misc gen values, etc.) | Custom data overrides |
| `time_clustering.yml` | Representative period selection | Controls temporal resolution |
| `flexible_load.yml` | Flexible demand (EVs, heat pumps) | Demand-side flexibility |
| `regional_resource_tags.yml` | Region-specific tag overrides | Per-region dispatch rules |

---

## Strategy: Reference skeleton + GUI choices



Our installed PowerGenome is **v0.7** (bundled with Switch-USA-PG-ReEDS). The GUI targets **v0.8** and uses parameter names that v0.7 doesn't recognize. Copying GUI files directly will silently drop critical settings.

**Approach:** Start from the reference settings (which are v0.7-compatible), then overwrite values with the GUI's actual choices (your regions, technologies, fuels, etc.), translating parameter names as needed.

### Translation map (GUI v0.8 → PG v0.7)

| GUI key (v0.8) | Reference key (v0.7) | Found in PG source? | Notes |
|---|---|---|---|
| `new_resources` | `atb_new_gen` | Yes (`generators.py`) | Both exist in v0.7; `new_resources` might work but `atb_new_gen` is safer |
| `resource_modifiers` | `atb_modifiers` | Only `atb_modifiers` | Must translate |
| `resource_data_year` | `atb_data_year` | Only `atb_data_year` | Must translate |
| `resource_financial_case` | `atb_financial_case` | Only `atb_financial_case` | Must translate |
| `resource_cap_recovery_years` | `atb_cap_recovery_years` | Only `atb_cap_recovery_years` | Must translate |
| `fuel_data_year` | `fuel_eia_aeo_year` | Only `fuel_eia_aeo_year` | Must translate |
| `fuel_scenarios` | `aeo_fuel_scenarios` | Both exist | Check behavior; safer to use `aeo_fuel_scenarios` |
| `model_periods` | `model_year` + `model_first_planning_year` | Only old names | Must unpack |
| `cache_resource_clusters` | (drop) | Not found | v0.8 caching flag, irrelevant |
| `use_resource_clusters_cache` | (drop) | Not found | v0.8 caching flag, irrelevant |
| `interconnect_capex_mw` | `interconnect_capex_mw` | Yes (`renewables.py`) | Same name, keep as-is |

### What the notebook does for each settings file

1. Load the reference version (v0.7 skeleton with correct key names)
2. Load the GUI version (your actual model choices)
3. Show both side by side so you can see what's different
4. Translate GUI keys → v0.7 keys
5. Merge: GUI values override reference values where they represent your choices
6. Let you review and edit before saving to `my_settings/`

---

## Setup

In [45]:
from pathlib import Path
import yaml
import shutil
import copy
import subprocess
yaml.SafeDumper.ignore_aliases = lambda *args: True
from ruamel.yaml import YAML

yaml_rt = YAML(typ="rt")
yaml_rt.version = (1, 2)           # yes/no are strings, not bools
yaml_rt.preserve_quotes = True
yaml_rt.indent(mapping=2, sequence=4, offset=2)
def _none_as_tilde(representer, _):
    return representer.represent_scalar("tag:yaml.org,2002:null", "~")

yaml_rt.representer.add_representer(type(None), _none_as_tilde)
# ============================================================
# EDIT THESE PATHS to match local setup
# ============================================================
REPO_ROOT = Path("/Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS").expanduser()
GUI_SETTINGS_DIR = REPO_ROOT / "pg" / "powergenome_settings_46regions" / "settings"
REF_SETTINGS_DIR = REPO_ROOT / "pg" / "settings"
OUTPUT_SETTINGS_DIR = REPO_ROOT / "pg" / "settings_10weeks_7days_PROFILE_CLUSTERS"
# ============================================================

OUTPUT_SETTINGS_DIR.mkdir(parents=True, exist_ok=True)

def load_all_keys(folder):
    """Load every .yml in a folder, return {key: (filename, value)} for all top-level keys."""
    result = {}
    for f in sorted(folder.glob("*.yml")):
        with open(f) as fh:
            data = yaml.safe_load(fh)
        if data:
            for k, v in data.items():
                result[k] = (f.name, v)
    return result

def load_yml(path):
    with open(path, "r") as f:
        data = yaml_rt.load(f)
    return data if data is not None else {}

def save_yml(data, path):
    with open(path, "w") as f:
        yaml_rt.dump(data, f)
    print(f"  ✓ Wrote {path.name} ({len(data)} keys) → file://{path.resolve()}")
from datetime import datetime

HEADER_COMMENT = """\
File was generated by pg/SettingsBuilder.ipynb on {timestamp}.
NOTE: Comments in this file were copied from the reference settings and
may not fully reflect the current content, some values and flags have
been changed by the notebook. Treat inline comments as context from the
original reference, not as authoritative documentation of this file.
"""

def save_yml(data, path):
    timestamp = datetime.now().strftime("%Y/%m/%d %H:%M")
    header = HEADER_COMMENT.format(timestamp=timestamp)
    try:
        data.yaml_set_start_comment(header)
    except AttributeError:
        # Plain dict (e.g. assembled from translate_gui_dict) — wrap it
        # in a ruamel CommentedMap so we can attach the header.
        from ruamel.yaml.comments import CommentedMap
        cm = CommentedMap(data)
        cm.yaml_set_start_comment(header)
        data = cm
    with open(path, "w") as f:
        yaml_rt.dump(data, f)
    print(f"  ✓ Wrote {path.name} ({len(data)} keys) → file://{path.resolve()}")
def show_yml(path):
    with open(path, "r") as f:
        print(f.read())

def preview(data, max_lines=20):
    """Print a YAML preview of a dict, truncated."""
    text = yaml.dump(data, default_flow_style=False, sort_keys=False)
    lines = text.strip().split("\n")
    for line in lines[:max_lines]:
        print(f"  {line}")
    if len(lines) > max_lines:
        print(f"  ... ({len(lines) - max_lines} more lines)")

# ---------------------------------------------------------------
# GUI v0.8 → PG v0.7 translation map
# Keys to rename (gui_name → ref_name)
# ---------------------------------------------------------------
TRANSLATE_KEYS = {
    "resource_modifiers":        "atb_modifiers",
    "resource_data_year":        "atb_data_year",
    "resource_financial_case":   "atb_financial_case",
    "resource_cap_recovery_years": "atb_cap_recovery_years",
    "fuel_data_year":            "fuel_eia_aeo_year",
    # new_resources and fuel_scenarios exist in v0.7 but
    # atb_new_gen and aeo_fuel_scenarios are the primary names
    "new_resources":             "atb_new_gen",
    "fuel_scenarios":            "aeo_fuel_scenarios",
}

# Keys to drop entirely (v0.8-only, no v0.7 equivalent)
DROP_KEYS = {
    "cache_resource_clusters",
    "use_resource_clusters_cache",
}

# model_periods needs special handling (unpacked into model_year + model_first_planning_year)

def translate_gui_dict(gui_dict):
    """Translate a GUI settings dict to v0.7-compatible key names."""
    result = {}
    dropped = []
    renamed = []
    
    for k, v in gui_dict.items():
        if k in DROP_KEYS:
            dropped.append(k)
        elif k in TRANSLATE_KEYS:
            new_key = TRANSLATE_KEYS[k]
            renamed.append(f"{k} → {new_key}")
            result[new_key] = v
        elif k == "model_periods":
            # Unpack [[start1, end1], [start2, end2], ...]
            # model_year = list of end years
            # model_first_planning_year = list of start years
            periods = v
            result["model_first_planning_year"] = [p[0] for p in periods]
            result["model_year"] = [p[1] for p in periods]
            renamed.append(f"model_periods → model_year + model_first_planning_year")
        else:
            result[k] = v
    
    if dropped:
        print(f"  Dropped (v0.8-only): {dropped}")
    if renamed:
        print(f"  Renamed:")
        for r in renamed:
            print(f"    {r}")
    
    return result

# Quick test
print("Sanity check — files found:")
print(f"  GUI:  {sorted(f.name for f in GUI_SETTINGS_DIR.glob('*.yml'))}")
print(f"  REF:  {sorted(f.name for f in REF_SETTINGS_DIR.glob('*.yml'))}")
print(f"  OUT:  {OUTPUT_SETTINGS_DIR}")

Sanity check — files found:
  GUI:  ['distributed_gen.yml', 'fuels.yml', 'model_definition.yml', 'resource_tags.yml', 'resources.yml', 'startup_costs.yml', 'transmission.yml']
  REF:  ['demand.yml', 'distributed_gen.yml', 'env.yml', 'extra_inputs.yml', 'flexible_load.yml', 'fuels.yml', 'model_definition.yml', 'regional_resource_tags.yml', 'resource_tags.yml', 'resources.yml', 'scenario_management.yml', 'startup_costs.yml', 'switch.yml', 'time_clustering.yml', 'transmission.yml']
  OUT:  /Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS


---

## Building settings, file by file

For each settings file we either:

- **Build it from a merge** of the GUI and reference versions (most files), translating GUI v0.8 key names to v0.7 equivalents and applying file-specific transformations like p-region remapping
- **Copy from the reference** unchanged or with small overrides (files the GUI doesn't produce)
- **Build from scratch** (only `demand.yml`)

The notebook prints a summary after each file showing what was overridden, merged, or remapped, plus a clickable `file://` link to the saved version. Files are organized in roughly increasing order of "how much the notebook does":

| # | File | Source | What the notebook does |
|---|---|---|---|
| 1 | `env.yml` | Reference only | Copy + edit paths for your machine |
| 2 | `model_definition.yml` | Both | Merge + remap `regional_capacity_reserves` to model regions |
| 3 | `resources.yml` | Both | 4 passes: GUI overrides → cost zone remap → hydro remap → ATB tech map validation |
| 4 | `fuels.yml` | Both | 3 passes: overrides + merges → fuel region remap → `waste_biomass` injection |
| 5 | `transmission.yml` | Both | 3 passes: generic merge → spur capex aggregation → expansion limit override |
| 6 | `startup_costs.yml` | Both | Generic merge |
| 7 | `distributed_gen.yml` | Both | Generic merge |
| 8 | `resource_tags.yml` | Both | Merge + ensure `Imports` is tagged `MUST_RUN` |
| 9 | `demand.yml` | Built from scratch | Hardcoded ReEDS load curves config |
| 10 | `scenario_management.yml` | Reference only | Copy as-is |
| 11 | `switch.yml` | Reference only | Copy + override `discount_rate` to match `interest_rate` |
| 12 | `extra_inputs.yml` | Reference only | Copy + patch transmission costs CSV (add annuity columns, dedupe pairs) |
| 13 | `time_clustering.yml` | Reference only | Copy as-is |
| 14 | `flexible_load.yml` | Reference only | Copy as-is |
| 15 | `regional_resource_tags.yml` | Reference only | **Write minimal disabled version** (state ESR not used in Switch studies) |

---

## 1. `env.yml` — Database paths and data locations

This file tells PowerGenome where to find every external data source on your machine: the PUDL and PowerGenome SQLite databases, the renewable resource group folders, the distributed generation data, and the NREL Electrification Futures Study load data.

This is the only settings file in the entire notebook that's genuinely machine-specific. Paths will look different, and there's no automatic way for the GUI or the reference to guess them. We start from the reference's defaults (which assume the standard `pg_data/` layout from the Switch-USA-PG-ReEDS setup instructions) and edit individual paths if your local structure is different.

#### Parameters

| Key | What it points to |
|---|---|
| `PUDL_DB` | `pudl.sqlite` — EIA Form 860/923 data on every US generating unit (technology, capacity, vintage, location, ownership, fuel use, emissions). Used by PowerGenome to build the existing-generator dataset. |
| `PG_DB` | `pg_misc_tables_efs_*.sqlite` — NREL ATB technology costs, transmission limits, hourly demand profiles, and other PowerGenome-specific reference tables. |
| `RESOURCE_GROUPS` | Folder of parquet files describing renewable resource clusters (wind/solar sites grouped by capacity factor and cost). Each technology has a `<tech>_group.json` descriptor and a `<tech>_lcoe_*.parquet` data file. **The next section of this notebook overrides this path** to point at a custom `pg/my_resource_groups/` directory with GUI-computed LCOE files patched in. |
| `RESOURCE_GROUP_PROFILES` | Folder with hourly generation profile time series for each resource group. Used to build per-cluster capacity factor profiles for the model years. |
| `DISTRIBUTED_GEN_DATA` | Folder with NREL distributed PV data (referenced by `distributed_gen.yml`). |
| `EFS_DATA` | Folder with NREL Electrification Futures Study hourly load profiles. Currently unused in our setup since we pull loads from `load_curves_nrel_reeds` in `PG_DB` instead, but PowerGenome still expects the path to be valid. |

#### When to edit

You'll need to edit this file if:

- **You have a non-standard `pg_data/` location** — update all six paths to point at your actual data folder
- **You're using a newer PUDL or PG database vintage** — change `PUDL_DB` or `PG_DB` to point at the new files (filenames change between vintages, e.g., `pudl.2025_08.sqlite` vs `pudl.2025_03.sqlite`)

The code cell below has commented-out template lines for each path. Uncomment and edit only the ones that differ from the reference. Don't worry about `RESOURCE_GROUPS` — the next section of the notebook overwrites it regardless of what you set here.

#### Gotcha

`EFS_DATA` is required even though we don't actively use the EFS load profiles. PowerGenome refuses to load settings if any of the six paths are missing. If you really don't have the EFS data, point it at any valid (possibly empty) folder.

See [PG docs: env settings](https://powergenome.github.io/PowerGenome/v0.8-beta/how-to/configure-settings/?h=envir#environment-specific-settings).

In [3]:
# env.yml is loaded from reference. The reference paths assume the standard
# pg_data/ layout from the Switch-USA-PG-ReEDS setup instructions. If your
# pg_data lives somewhere else, uncomment and edit the relevant lines below.
ref_env = load_yml(REF_SETTINGS_DIR / "env.yml")

print("=== Reference env.yml ===")
preview(ref_env)

env_settings = copy.deepcopy(ref_env)

# Convert any absolute paths under REPO_ROOT into repo-relative POSIX paths so
# the saved env.yml stays portable across machines.
PATH_KEYS = [
    "PUDL_DB",
    "PG_DB",
    "RESOURCE_GROUPS",
    "RESOURCE_GROUP_PROFILES",
    "DISTRIBUTED_GEN_DATA",
    "EFS_DATA",
]

for key in PATH_KEYS:
    val = env_settings.get(key)
    if not val:
        continue
    p = Path(val)
    if p.is_absolute():
        try:
            p = p.relative_to(REPO_ROOT)
        except ValueError:
            # Absolute but lives outside REPO_ROOT; leave the original value alone.
            env_settings[key] = str(p)
            continue
    env_settings[key] = p.as_posix()

# Uncomment and edit if your paths differ from the reference defaults:
# env_settings["PUDL_DB"]                 = "pg_data/pudl.2025_08.sqlite"
# env_settings["PG_DB"]                   = "pg_data/pg_misc_tables_efs_2023_2.sqlite"
# env_settings["RESOURCE_GROUPS"]         = "pg_data/resource_groups"
# env_settings["RESOURCE_GROUP_PROFILES"] = "pg_data/profiles"
# env_settings["DISTRIBUTED_GEN_DATA"]    = "pg_data/NREL-dist-gen"
# env_settings["EFS_DATA"]                = "pg_data/efs_files_utc"

# Note: the next section of this notebook (Building a custom resource_groups
# directory) will overwrite RESOURCE_GROUPS to point at pg/my_resource_groups/
# regardless of what you set here.

save_yml(env_settings, OUTPUT_SETTINGS_DIR / "env.yml")

=== Reference env.yml ===
  !!python/object/apply:ruamel.yaml.comments.CommentedMap
  state:
    _yaml_line_col: !!python/object:ruamel.yaml.comments.LineCol
      line: 0
      col: 0
      data:
        PUDL_DB:
        - 0
        - 0
        - 0
        - 9
        PG_DB:
        - 1
        - 0
        - 1
        - 7
        EFS_DATA:
        - 2
        - 0
        - 2
  ... (33 more lines)
  ✓ Wrote env.yml (6 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/env.yml


### 1.1 Building a custom `resource_groups/` directory

PowerGenome's `resource_groups/` folder is what tells it where renewable sites are, how much capacity each site can host, what the LCOE looks like, and where to find the hourly generation profiles. The reference folder ships with files built for the 134 individual p-regions. We can't just point at it directly — two things are wrong for our use case, and one thing is wrong with how the GUI exports files.

#### Problem 1: Reference LCOE files are aggregated for the "wrong" regions

The reference `onshorewind_lcoe_ReEDS.csv` and `solar_lcoe_ReEDS.csv` give site-level cost and capacity factor data clustered for the reference's 134 p-regions. When you aggregate p-regions into your own model regions (e.g., `WA1 = [p1, p2, p3]`), those clusters no longer line up.

The GUI solves this by recomputing LCOE for *your* aggregation and writing the result to `powergenome_settings/resource_groups_resource_groups/` as `.parquet` files. We need to swap the GUI's parquet files in for the reference's CSVs.

#### Problem 2: GUI JSON files reference filenames that don't exist on disk

Each tech in `resource_groups/` has a `<tech>_group.json` that points PowerGenome at the actual data files (LCOE table, site map, hourly profiles). The GUI's JSONs sometimes reference filenames with a `_tidy` suffix (e.g., `wind_profiles_tidy.parquet`) that don't exist in `pg_data/profiles/` — only the un-suffixed versions are there. We patch the JSONs to point at the files that actually exist.

#### Problem 3: NaN capacity rows crash scipy's clustering

The existing-generator metadata CSVs (`existing_resource_groups-patched/*_metadata*.csv`) contain one row per (sub-region, technology) combination. When you aggregate regions, some sub-regions in the union have no installed capacity for a given technology and show up with `NaN` in the `mw` column. PowerGenome feeds this table into scipy's Ward hierarchical clustering, which crashes on NaN. We drop those rows.

#### What this section produces

A new directory `pg/my_resource_groups/` that is a copy of the reference folder with the GUI's LCOE files swapped in, JSON references patched, N

In [4]:
# Step 1: Copy the reference resource_groups, then swap in GUI LCOE files
# ----------------------------------------------------------------------
# We work on a copy so the reference folder stays clean. The copy lives at
# pg/my_resource_groups/ next to my_settings/.

import json
import os

ref_rg_dir = REPO_ROOT / "pg_data" / "resource_groups"
import warnings
name = OUTPUT_SETTINGS_DIR.name
if not name.startswith("settings_"):
    warnings.warn(f"OUTPUT_SETTINGS_DIR name {name!r} does not start with 'settings_'; using full name.")
suffix = name.removeprefix("settings_")
my_rg_dir = REPO_ROOT / "pg" / f"resource_groups_{suffix}"
gui_rg_dir = REPO_ROOT / "pg" / "powergenome_settings_46regions" / "resource_groups_resource_groups"
profiles_dir = REPO_ROOT / "pg_data" / "profiles"

# Fresh copy every run so this cell is idempotent
if my_rg_dir.exists():
    shutil.rmtree(my_rg_dir)
shutil.copytree(ref_rg_dir, my_rg_dir)
print(f"Copied {ref_rg_dir.name}/ → {my_rg_dir.name}/")

# Swap reference wind/solar files for GUI versions inside ReEDS-cpas-patched/.
# The reference uses .csv for LCOE; the GUI uses .parquet. We delete the old
# CSV and drop in the parquet, then replace the JSON descriptors that point
# at them.
target_dir = my_rg_dir / "ReEDS-cpas-patched"

gui_to_ref_map = {
    # GUI filename                                  → reference filename it replaces
    "onshorewind_lcoe_resource_groups.parquet":     "onshorewind_lcoe_ReEDS.csv",
    "solar_lcoe_resource_groups.parquet":           "solar_lcoe_ReEDS.csv",
    "onshorewind_group.json":                       "onshorewind_group.json",
    "solar_group.json":                             "solar_group.json",
}

for gui_file, ref_file in gui_to_ref_map.items():
    src = gui_rg_dir / gui_file
    if not src.exists():
        print(f"  ⚠ GUI file not found, skipping: {gui_file}")
        continue

    old = target_dir / ref_file
    if old.exists():
        old.unlink()
        print(f"  Removed reference file: {ref_file}")

    shutil.copy2(src, target_dir / gui_file)
    print(f"  Copied GUI file:        {gui_file}")

Copied resource_groups/ → resource_groups_10weeks_7days_PROFILE_CLUSTERS/
  Removed reference file: onshorewind_lcoe_ReEDS.csv
  Copied GUI file:        onshorewind_lcoe_resource_groups.parquet
  Removed reference file: solar_lcoe_ReEDS.csv
  Copied GUI file:        solar_lcoe_resource_groups.parquet
  Removed reference file: onshorewind_group.json
  Copied GUI file:        onshorewind_group.json
  Removed reference file: solar_group.json
  Copied GUI file:        solar_group.json


#### Patching the JSON descriptors

Each `<tech>_group.json` has three fields that point at data files: `profiles` (hourly generation), `site_map` (which sites belong to which cluster), and `metadata` (the LCOE/capacity table). When the GUI emits a JSON that references a `_tidy.parquet` file, the file isn't in `pg_data/profiles/` — only the version without `_tidy` is. The fix is mechanical: if the referenced file doesn't exist but the un-`_tidy` version does, rewrite the JSON to point at the version that exists.

In [5]:
# Step 2: Patch JSON profile references in the swapped-in files
# ----------------------------------------------------------------------
# Look at every <tech>_group.json in target_dir. For each of the three
# file-pointer fields, check that the referenced file exists somewhere
# (either alongside the JSON or in pg_data/profiles/). If it doesn't,
# try removing "_tidy" from the filename and see if that exists instead.

available_profiles = {f.name for f in profiles_dir.iterdir()}
POINTER_FIELDS = ["profiles", "site_map", "metadata"]



def file_exists_anywhere(fname):
    return (target_dir / fname).exists() or fname in available_profiles

for json_path in target_dir.glob("*.json"):
    with open(json_path) as f:
        group = json.load(f)

    modified = False
    for field in POINTER_FIELDS:
        fname = group.get(field)
        if not fname or file_exists_anywhere(fname):
            continue

        # Try the un-tidy version
        fixed = fname.replace("_tidy", "")
        if fixed != fname and file_exists_anywhere(fixed):
            print(f"  {json_path.name}: {field} '{fname}' → '{fixed}'")
            group[field] = fixed
            modified = True
        else:
            print(f"  ⚠ {json_path.name}: {field} '{fname}' not found anywhere")

    if modified:
        with open(json_path, "w") as f:
            json.dump(group, f, indent=2)

  onshorewind_group.json: profiles 'onshorewind_rev_profiles_20240801_tidy.parquet' → 'onshorewind_rev_profiles_20240801.parquet'
  solar_group.json: profiles 'solar_rev_profiles_20240801_tidy.parquet' → 'solar_rev_profiles_20240801.parquet'


#### Pointing `env.yml` at the new directory and verifying structure

`env.yml` was written earlier with `RESOURCE_GROUPS` set to the reference path. Now that `my_resource_groups/` exists and is patched, we update that key in place and print the directory tree so you can confirm everything landed where you expect.

In [10]:
# Step 3: Update env.yml to point at my_resource_groups/
# ----------------------------------------------------------------------
env = load_yml(OUTPUT_SETTINGS_DIR / "env.yml")

# Store as a repo-relative POSIX path so the saved env.yml stays portable.
my_rg_path = Path(my_rg_dir)
if my_rg_path.is_absolute():
    try:
        my_rg_rel = my_rg_path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        # Lives outside REPO_ROOT; keep the absolute path.
        my_rg_rel = str(my_rg_path)
else:
    my_rg_rel = my_rg_path.as_posix()

env["RESOURCE_GROUPS"] = my_rg_rel
save_yml(env, OUTPUT_SETTINGS_DIR / "env.yml")
print(f"env.yml RESOURCE_GROUPS → {my_rg_rel}")

# Print the directory tree so we can eyeball that the swap worked
print(f"\n=== my_resource_groups/ structure ===")
for root, dirs, files in os.walk(my_rg_dir):
    depth = root.replace(str(my_rg_dir), "").count(os.sep)
    print(f"{'  ' * depth}{Path(root).name}/")
    for f in sorted(files):
        print(f"{'  ' * (depth + 1)}{f}")

  ✓ Wrote env.yml (6 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/env.yml
env.yml RESOURCE_GROUPS → pg/resource_groups_10weeks_7days_PROFILE_CLUSTERS

=== my_resource_groups/ structure ===
resource_groups_10weeks_7days_PROFILE_CLUSTERS/
  .DS_Store
  existing_resource_groups-patched/
    NOTES.txt
    existing_hydro_reeds_ba.parquet
    existing_hydro_reeds_ba_metadata.csv
    existing_offshorewind_reeds_ba_metadata.csv
    existing_onshore_wind_reeds_ba_profiles_20250513.parquet
    existing_onshorewind_reeds_ba_metadata_20250513.csv
    existing_osw_profiles.csv
    existing_solar_reeds_ba_metadata_20250513.csv
    existing_solar_reeds_ba_profiles_20250513.parquet
    regional_existing_offshorewind_profiles.csv
    regional_existing_solarpv_profiles.csv
    regional_hydro_True.json
    regional_offshore_True.json
    regional_small_hydro_True.json
    regional_solar_photovoltaic_True.json
    regional_wind_True.json
  imp

#### Dropping NaN capacity rows from existing-generator metadata

This is the last fix and the most opaque, so it's worth spelling out. PowerGenome clusters existing generators with `scipy.cluster.hierarchy.linkage` using Ward's method. Ward minimizes within-cluster variance, which requires all input rows to have a defined value in every column it clusters on. If any row in `*_metadata*.csv` has `NaN` in the capacity column (`mw` or `capacity_mw`), scipy raises a cryptic `ValueError` and PowerGenome dies before you ever see model output.

This happens specifically when region aggregation produces an aggregated region whose constituent sub-regions have no installed capacity for some technology. The metadata file still has a row for that (sub-region, technology) pair, but with `NaN` for capacity. Dropping those rows is safe: they represent zero capacity, so removing them changes nothing about what the cluster contains.

In [11]:
# Step 4: Drop NaN-capacity rows from existing generator metadata
# ----------------------------------------------------------------------
import pandas as pd

existing_dir = my_rg_dir / "existing_resource_groups-patched"
patched_count = 0

for csv_path in existing_dir.glob("*_metadata*.csv"):
    df = pd.read_csv(csv_path)

    # The capacity column is named "mw" in some files and "capacity_mw" in others
    cap_col = next((c for c in ("mw", "capacity_mw") if c in df.columns), None)
    if cap_col is None:
        print(f"  {csv_path.name}: no capacity column, skipping")
        continue

    n_nan = df[cap_col].isna().sum()
    if n_nan == 0:
        print(f"  {csv_path.name}: clean")
        continue

    df_clean = df.dropna(subset=[cap_col])
    df_clean.to_csv(csv_path, index=False)
    print(f"  {csv_path.name}: dropped {n_nan} NaN rows ({len(df)} → {len(df_clean)})")
    patched_count += 1

print(f"\nPatched {patched_count} metadata file(s)")

  existing_onshorewind_reeds_ba_metadata_20250513.csv: dropped 54 NaN rows (134 → 80)
  existing_hydro_reeds_ba_metadata.csv: clean
  existing_offshorewind_reeds_ba_metadata.csv: dropped 34 NaN rows (36 → 2)
  existing_solar_reeds_ba_metadata_20250513.csv: dropped 16 NaN rows (134 → 118)

Patched 3 metadata file(s)


In [12]:
# Step 5: Validate that PowerGenome can actually load every expected group
# ------------------------------------------------------------------------
# File existence isn't enough. PG silently drops groups whose internal
# refs don't link (metadata.cpa_id ↔ site_map, site_map ↔ profile cols).
# We invoke PG's own loader here so any breakage shows up now, not 30
# minutes into a pipeline run as an empty find_groups result.

import os
from powergenome import params as pg_params
from powergenome.params import build_resource_clusters
from powergenome.nrelatb import load_resource_group_data

env = load_yml(OUTPUT_SETTINGS_DIR / "env.yml")
rg_path   = (REPO_ROOT / env["RESOURCE_GROUPS"]).resolve()
prof_path = (REPO_ROOT / env["RESOURCE_GROUP_PROFILES"]).resolve()

os.environ["RESOURCE_GROUPS"]         = str(rg_path)
os.environ["RESOURCE_GROUP_PROFILES"] = str(prof_path)
pg_params.SETTINGS["RESOURCE_GROUPS"]         = str(rg_path)
pg_params.SETTINGS["RESOURCE_GROUP_PROFILES"] = str(prof_path)

cb = build_resource_clusters(group_path=rg_path, profile_path=prof_path)
print(f"build_resource_clusters loaded {len(cb.groups)} group(s)\n")

# New-build techs we expect to resolve. Add/remove as your setup requires.
EXPECTED = [
    ("landbasedwind", False),
    ("utilitypv",     False),
    # ("offshorewind", False),   # uncomment if you have an offshore group
]

failures = []
for tech, existing in EXPECTED:
    hits = cb.find_groups(existing=existing, technology=tech)
    if not hits:
        failures.append(f"  ✗ {tech} (existing={existing}): 0 groups — "
                        f"PG dropped it during load")
        continue
    # Force the actual data read. This is the call that raises if the
    # metadata ↔ site_map ↔ profiles bridge is broken, which is what
    # yesterday's error came down to.
    try:
        renew_data, site_map = load_resource_group_data(hits[0], cache=False)
        print(f"  ✓ {tech:<15} (existing={existing}): {len(hits)} group(s), "
              f"renew_data {renew_data.shape}, site_map {site_map.shape}")
    except Exception as e:
        failures.append(f"  ✗ {tech} (existing={existing}): "
                        f"load_resource_group_data raised "
                        f"{type(e).__name__}: {e}")

if failures:
    print("\n" + "\n".join(failures))
    raise RuntimeError("Resource group validation failed — see above.")
print("\nAll expected resource groups resolve ✓")

build_resource_clusters loaded 12 group(s)

  ✓ landbasedwind   (existing=False): 1 group(s), renew_data (76739, 17), site_map (84704,)
  ✓ utilitypv       (existing=False): 1 group(s), renew_data (405479, 17), site_map (406109,)

All expected resource groups resolve ✓


In [13]:
print("\nAll 12 groups:")
for i, rg in enumerate(cb.groups):
    print(f"  [{i:2d}] tech={rg.group.get('technology')!r:<18} "
          f"existing={rg.group.get('existing', False)}")


All 12 groups:
  [ 0] tech='offshorewind'     existing=True
  [ 1] tech='hydro'            existing=True
  [ 2] tech='hydro'            existing=True
  [ 3] tech='utilitypv'        existing=True
  [ 4] tech='landbasedwind'    existing=True
  [ 5] tech='imports'          existing=False
  [ 6] tech='offshorewind'     existing=False
  [ 7] tech='offshorewind'     existing=False
  [ 8] tech='offshorewind'     existing=False
  [ 9] tech='offshorewind'     existing=False
  [10] tech='landbasedwind'    existing=False
  [11] tech='utilitypv'        existing=False


## 2. `time_clustering.yml` — Representative period selection

PowerGenome can run a model with full hourly resolution (8,760 hours per year) or a reduced-form clustered version that picks N representative days. Clustering dramatically speeds up solver time but trades off temporal fidelity. This file controls how aggressively to cluster.

#### Parameters

| Key | What it does |
|---|---|
| `reduce_time_domain` | Master switch. Set `true` to enable clustering (recommended for testing and most production runs). Set `false` to use full hourly profiles, which is usually only practical for very small region setups. |
| `time_domain_periods` | Number of representative clusters to pick. The reference uses `4` (very fast, low fidelity). Production runs typically use `10–20`. Higher values give better resolution of seasonal and diurnal variation but slow the solver roughly linearly. |
| `time_domain_days_per_period` | Days per cluster. The reference uses `1` (single representative day). Increasing to `2–7` captures multi-day weather events (cold snaps, calm wind periods) at the cost of more model size. |
| `include_peak_day` | When `true`, the system-wide peak load day is always added as an extra cluster regardless of what the clustering algorithm picks. Almost always leave this `true` — without it the model can underbuild peaking capacity. |
| `demand_weight_factor` | Weight applied to demand profiles when clustering. Demand is normalized to 0–1 then scaled by this factor. Increasing it makes the algorithm prioritize matching demand shape over matching renewable generation shape. Default of `1` is fine for most cases. |

#### What to change

- **First-run testing**: leave the reference values (4 periods × 1 day + peak day = 5 representative days). The model will solve in seconds.
- **Published results**: bump to at least `10` periods and consider `2` days per period. This catches more of the variability that drives capacity expansion decisions.
- **Large region counts**: clustering matters more, not less. Don't be tempted to skip it just because the model already feels big.

See [PowerGenome time-domain reduction docs](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/time-reduction/) for the underlying clustering algorithm.

In [14]:
# time_clustering.yml is copied from the reference. See the markdown above
# for what each parameter does and when to change it.
tc_data = load_yml(REF_SETTINGS_DIR / "time_clustering.yml")


# Optional overrides — uncomment any line to change a setting from the
# reference value. See the markdown above for valid values and tradeoffs.

# Note: changing time_domain_periods or time_domain_days_per_period will
# cause the auto-add cell in section 11 to create a new case in
# scenario_inputs.csv (named my_case_s<P>x<D>) so the three files stay
# consistent.

# tc_data["reduce_time_domain"]          = True   # False = full hourly (slow, only for tiny models)
tc_data["time_domain_periods"]         = 10      # number of representative clusters
tc_data["time_domain_days_per_period"] = 7      # days per cluster (1 = single day, 2-7 = multi-day)
# tc_data["include_peak_day"]            = True   # always include system-wide peak load day
# tc_data["demand_weight_factor"]        = 1      # how much demand shape matters in clustering

print("\n=== time_clustering.yml ===")
print(f"  reduce_time_domain:          {tc_data.get('reduce_time_domain')}")
print(f"  time_domain_periods:         {tc_data.get('time_domain_periods')}")
print(f"  time_domain_days_per_period: {tc_data.get('time_domain_days_per_period')}")
print(f"  include_peak_day:            {tc_data.get('include_peak_day')}")
print(f"  demand_weight_factor:        {tc_data.get('demand_weight_factor')}")

save_yml(tc_data, OUTPUT_SETTINGS_DIR / "time_clustering.yml")


=== time_clustering.yml ===
  reduce_time_domain:          True
  time_domain_periods:         10
  time_domain_days_per_period: 7
  include_peak_day:            True
  demand_weight_factor:        1
  ✓ Wrote time_clustering.yml (5 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/time_clustering.yml


---

## 3. `model_definition.yml` — Core model structure

This is the skeleton of the model: which regions exist, which planning years to solve, what timezone the profiles are in, and what columns appear in the generator output table. Almost every other settings file references something defined here, so this is where bugs in your model setup tend to surface first.

Several keys in this file are read directly by the Switch conversion code (`pg_to_switch.py`), not just by PowerGenome itself. Keys flagged below as **(also read by Switch glue)** affect Switch's `gen_info.csv`, `financials.csv`, `transmission_lines.csv`, or `planning_reserve_requirements.csv`, so getting them wrong has consequences beyond PowerGenome.

### Key parameters

| Key | What it does |
|---|---|
| `model_regions` | List of region names. Every other settings file references these. **(also read by Switch glue)** |
| `region_aggregations` | Maps your model regions to underlying IPM/ReEDS balancing areas. `null` in the reference means each BA is its own region. The GUI produces explicit mappings when you cluster BAs. |
| `model_year` | List of planning years to solve, e.g. `[2030, 2035, 2040]`. **(also read by Switch glue)** |
| `model_first_planning_year` | List of first years in each planning period, e.g. `[2024, 2031, 2036]`. Each entry is the first year capex from ATB will be averaged from for that period. Generators scheduled to retire before this year are dropped from that period. **(also read by Switch glue)** |
| `target_usd_year` | All costs are converted to this dollar year via CPI. Must be a historical year so CPI data is available. **(also read by Switch glue)** |
| `utc_offset` | Hours offset from UTC for demand/generation profile alignment. US East Coast is `-5`, Pacific is `-8`. All profile data is stored in UTC and shifted by this amount during processing. |
| `generator_columns` | Column names for the final generator output table. The reference has ~238 entries (including many policy tag columns); the GUI typically has ~84. The reference list is a superset of what the GUI produces. |
| `regional_capacity_reserves` | Planning reserve margin per region per program. Structure: `{CapRes_N: {region: fraction}}`. The reference uses a uniform 2% margin everywhere, on top of T&D losses, spinning reserves, and forced outages. Pass 3 of the build cell below remaps the inner p-region keys to your aggregated model regions. |
| `cap_res_network_derate_default` | Default transmission derate factor used for capacity reserve calculations. Reference uses `1.0` (no derate) because effective transfer capability is already captured by `firm_ttc_mw` in `transmission.yml`. **(also read by Switch glue: written as `trans_derating_factor` in Switch's transmission lines table)** |
| `interest_compound_method` | `discrete` or `continuous` compounding for annuity calculations. Reference uses `discrete` to suppress a PowerGenome 0.7.0 warning. |

### What the notebook does

The build cell below performs three passes:

1. **Override simple GUI keys** — `model_regions`, `region_aggregations`, `target_usd_year`, `utc_offset`. These represent your specific model design.
2. **Merge `generator_columns`** — Use the reference list as the base (it's a superset including policy tag columns) and add any GUI-only columns.
3. **Remap `regional_capacity_reserves`** — Walk the inner `{p_region: value}` dicts and group p-regions by their parent model region. If all p-regions in a group agree on the value, use that value. If they disagree, use the first value and warn. Replicates the behavior of the old generic remapper with `strategy="first"`.

### `model_year` and `model_first_planning_year` are deliberately not overridden

The build cell does **not** copy planning years from the GUI even though the GUI provides them. Planning years are tightly coupled to other parts of the pipeline:

- Year-specific overrides in `scenario_management.yml`
- Hard-coded years in several CSVs in `extra_inputs/` and in `flexible_load.yml`'s `flexible_demand_resources` block
- Switch input file structure (Switch expects `period_start` and `period_end` to match what's in the CSVs)

Until that cross-file synchronization is automated, the safest path is to keep the reference's planning years and let the GUI's planning years be ignored. The GUI's values are preserved as a comment in the build cell so you can see them and decide whether to update the reference manually.

This is the single biggest piece of "manual work" when starting a new study. If you change planning years, walk through every file in the list above and verify the years match.

See [PG docs: model definition](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/model-definition/?h=model+def).

In [15]:
ref_md = load_yml(REF_SETTINGS_DIR / "model_definition.yml")
gui_md_raw = load_yml(GUI_SETTINGS_DIR / "model_definition.yml")

print("Translating GUI keys...")
gui_md = translate_gui_dict(gui_md_raw)

print(f"\nRef keys:  {sorted(ref_md.keys())}")
print(f"GUI keys (translated): {sorted(gui_md.keys())}")

# Keys only in reference (need to keep)
only_ref = set(ref_md) - set(gui_md)
print(f"\nOnly in ref (will keep): {sorted(only_ref)}")

# Keys only in GUI (need to add)
only_gui = set(gui_md) - set(ref_md)
print(f"Only in GUI (will add):  {sorted(only_gui)}")

# Keys in both
both = set(ref_md) & set(gui_md)
print(f"In both (GUI overrides): {sorted(both)}")

Translating GUI keys...
  Renamed:
    model_periods → model_year + model_first_planning_year

Ref keys:  ['cap_res_network_derate_default', 'generator_columns', 'interest_compound_method', 'model_first_planning_year', 'model_regions', 'model_year', 'region_aggregations', 'regional_capacity_reserves', 'target_usd_year', 'utc_offset']
GUI keys (translated): ['generator_columns', 'model_first_planning_year', 'model_regions', 'model_year', 'region_aggregations', 'target_usd_year', 'utc_offset']

Only in ref (will keep): ['cap_res_network_derate_default', 'interest_compound_method', 'regional_capacity_reserves']
Only in GUI (will add):  []
In both (GUI overrides): ['generator_columns', 'model_first_planning_year', 'model_regions', 'model_year', 'region_aggregations', 'target_usd_year', 'utc_offset']


In [16]:
# Start from reference, override with GUI choices
md_settings = copy.deepcopy(ref_md)

# --- Use GUI values for region-related and period-related keys ---
gui_override_keys = [
    "model_regions",
    "region_aggregations",
    "model_year",
    "model_first_planning_year",
    "target_usd_year",
    "utc_offset",
]

for key in gui_override_keys:
    if key in gui_md:
        old_val = md_settings.get(key, "NOT SET")
        md_settings[key] = gui_md[key]
        if isinstance(old_val, list):
            print(f"  {key}: [{len(old_val)} items] → [{len(gui_md[key])} items]")
        else:
            print(f"  {key}: {str(old_val)[:40]} → {str(gui_md[key])[:40]}")

# --- generator_columns: use reference (superset) as base ---
# The reference has policy tag columns the GUI doesn't include.
# Add any GUI-only columns that aren't already in the reference list.
if "generator_columns" in gui_md and "generator_columns" in ref_md:
    ref_cols = set(ref_md["generator_columns"])
    gui_cols = set(gui_md["generator_columns"])
    gui_only_cols = gui_cols - ref_cols
    if gui_only_cols:
        print(f"\n  generator_columns: adding {len(gui_only_cols)} GUI-only columns: {gui_only_cols}")
        md_settings["generator_columns"] = ref_md["generator_columns"] + list(gui_only_cols)
    else:
        print(f"\n  generator_columns: reference is superset, keeping reference ({len(ref_md['generator_columns'])} cols)")

# --- regional_capacity_reserves: remap inner p-region dicts to model regions ---
# Structure: {CapRes_1: {p1: 0.02, p2: 0.02, ...}, CapRes_2: {...}}
# For each policy, group the inner dict by parent model region.
# If all p-regions in a group have the same value, use it. If they disagree,
# use the first value and warn. p-regions not in any aggregation are kept as-is.
region_aggregations = gui_md.get("region_aggregations", {}) or {}

if region_aggregations and "regional_capacity_reserves" in md_settings:
    p_to_model_region = {}
    for model_region, p_list in region_aggregations.items():
        if p_list:
            for p in p_list:
                p_to_model_region[p] = model_region

    print("\n  Remapping regional_capacity_reserves p-regions → model regions")
    for policy, inner in md_settings["regional_capacity_reserves"].items():
        if not isinstance(inner, dict):
            continue

        groups = {}
        unmapped = {}
        for p, val in inner.items():
            if p in p_to_model_region:
                groups.setdefault(p_to_model_region[p], []).append((p, val))
            else:
                unmapped[p] = val

        remapped = {}
        for model_region, entries in groups.items():
            values = [v for _, v in entries]
            if len(set(str(v) for v in values)) == 1:
                remapped[model_region] = values[0]
            else:
                remapped[model_region] = values[0]
                print(
                    f"    ⚠ {policy}.{model_region}: {len(values)} p-regions disagree, "
                    f"using first ({entries[0][0]}={entries[0][1]})"
                )

        remapped.update(unmapped)
        md_settings["regional_capacity_reserves"][policy] = remapped
        print(f"    {policy}: {len(inner)} → {len(remapped)} entries")
elif "regional_capacity_reserves" in md_settings:
    print("\n  regional_capacity_reserves: no region_aggregations, keeping reference")

# --- Check that every model_year exists in scenario_inputs.csv ---
# pg_to_switch.py builds its available_years list from the CSV, NOT from
# model_definition.yml. If a year in model_year isn't in the CSV, the run
# will fail with a "No scenarios matching" error. The next cell of this
# section can auto-add the missing rows (and a matching time_series block
# in scenario_management.yml).
ei_data = load_yml(REF_SETTINGS_DIR / "extra_inputs.yml")
scen_inputs_path = (
    REF_SETTINGS_DIR.parent / ei_data["input_folder"] / ei_data["scenario_definitions_fn"]
)

# Read time_clustering.yml so the next cell knows what (periods, days) the
# labmate wants. Prefer my_settings/ if it exists from a previous run (so
# any overrides from the time_clustering.yml cell are picked up), and fall
# back to the reference on first run.
tc_my = OUTPUT_SETTINGS_DIR / "time_clustering.yml"
if tc_my.exists():
    tc_data_for_check = load_yml(tc_my)
    tc_source = GUI_SETTINGS_DIR # "my_settings/"
else:
    tc_data_for_check = load_yml(REF_SETTINGS_DIR / "time_clustering.yml")
    tc_source = "reference (my_settings/ not yet built)"
tc_periods = tc_data_for_check.get("time_domain_periods")
tc_days    = tc_data_for_check.get("time_domain_days_per_period")


if not scen_inputs_path.exists():
    print(f"\n  ⚠ Could not find {scen_inputs_path}, skipping year sync check")
    missing_years = []
else:
    import pandas as pd
    scen_df = pd.read_csv(scen_inputs_path)

    # Find cases in the CSV whose time_series name matches what time_clustering.yml
    # currently asks for. Convention: my_case_s<P>x<D>. We look for any case_id
    # that matches this pattern OR has a time_series column matching it. This is
    # how we know whether the *right* clustering case exists, not just whether
    # the years exist for some unrelated case.
    expected_ts_name = f"my_case_s{tc_periods}x{tc_days}"
    matching_cases = scen_df[scen_df["time_series"] == expected_ts_name]
    matching_years = set(matching_cases["year"].unique().tolist())

    model_years = set(md_settings["model_year"])
    missing_years = sorted(model_years - matching_years)

    print(f"\n  Checking model_year against {scen_inputs_path.name}...")
    print(f"    model_year:                       {sorted(model_years)}")
    print(f"    time_clustering.yml:              {tc_periods} periods × {tc_days} days/period  (source: {tc_source})")
    print(f"    looking for time_series:          {expected_ts_name}")
    print(f"    years available for that series:  {sorted(matching_years) if matching_years else '(no matching case in CSV)'}")
    if missing_years:
        print(f"    ⚠ MISSING for {expected_ts_name}: {missing_years}")
        print(f"    → The next cell can auto-add rows for these years.")
    else:
        print(f"    ✓ All model years present for {expected_ts_name}")
        
print(f"\n=== Final model_definition.yml ({len(md_settings)} keys) ===")
preview(md_settings, max_lines=30)

save_yml(md_settings, OUTPUT_SETTINGS_DIR / "model_definition.yml")

  model_regions: [134 items] → [46 items]
  region_aggregations: None → {'LAX': ['p10'], 'NY': ['p127'], 'Bay_ar
  model_year: [4 items] → [2 items]
  model_first_planning_year: [4 items] → [2 items]
  target_usd_year: 2024 → 2024
  utc_offset: -5 → -5

  generator_columns: adding 4 GUI-only columns: {'MinCapTag_2', 'MinCapTag_1', 'ESR_2', 'ESR_1'}

  Remapping regional_capacity_reserves p-regions → model regions
    CapRes_1: 134 → 46 entries

  Checking model_year against scenario_inputs.csv...
    model_year:                       [2035, 2050]
    time_clustering.yml:              10 periods × 7 days/period  (source: /Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/powergenome_settings_46regions/settings)
    looking for time_series:          my_case_s10x7
    years available for that series:  [2035, 2050]
    ✓ All model years present for my_case_s10x7

=== Final model_definition.yml (10 keys) ===
  !!python/object/apply:ruamel.yaml.comments.CommentedMap
  state:
    _yaml_form

---

## 4. `resources.yml` — Generators: existing clusters and new-build technologies

The largest and most complex settings file in the notebook. It controls two things:

**Existing generators:** How the ~10,000+ EIA generators get clustered into a manageable number of representative units. PowerGenome groups generators by technology and region, then clusters within each group based on heat rate similarity.

**New-build technologies:** Which new generators the model is allowed to build, where their costs come from (NREL ATB), and how renewable sites are clustered by cost and capacity factor.

A handful of keys here are read directly by the Switch conversion code (`pg_to_switch.py`), not just by PowerGenome itself. They're flagged in the parameter table as **(also read by Switch glue)**.

### Translation: GUI v0.8 → PG v0.7

This file has the most parameter name conflicts of any settings file. The notebook's `translate_gui_dict` helper handles the translations automatically, but it's worth knowing what changed:

| GUI key (v0.8) | Must use (v0.7) | What it does |
|---|---|---|
| `new_resources` | `atb_new_gen` | List of `[Technology, TechDetail, CostCase, SizeMW]` tuples defining new-build options |
| `resource_modifiers` | `atb_modifiers` | Cost/performance overrides applied to ATB technologies |
| `resource_data_year` | `atb_data_year` | Which ATB vintage to pull costs from (e.g., 2024) |
| `resource_financial_case` | `atb_financial_case` | ATB financial assumptions ("Market" or "R&D") |
| `resource_cap_recovery_years` | `atb_cap_recovery_years` | Default capital recovery period for annuity calculations |
| `cache_resource_clusters` | (drop) | v0.8 caching flag, not recognized by v0.7 |
| `use_resource_clusters_cache` | (drop) | v0.8 caching flag, not recognized by v0.7 |

### Key parameters from the reference

These are the keys the GUI doesn't produce but the reference provides. The notebook keeps them as-is unless one of the four passes (next markdown cell) modifies them.

| Key | What it does |
|---|---|
| `eia_data_years` | List of years used to query the EIA dataset for existing-generator attributes (heat rates, capacity factors, etc.). **(also read by Switch glue)** |
| `eia_860m_fn` | Filename of the EIA 860m monthly update CSV used to refresh planned/under-construction generator data. |
| `capacity_col` | Which column to pull plant capacity from in PUDL: typically `capacity_mw`, but you can use `winter_capacity_mw` or `summer_capacity_mw` if you want season-specific ratings. **(also read by Switch glue)** |
| `cluster_with_retired_gens` | Whether to include retired generators when computing cluster averages. Reference uses `false`. |
| `num_clusters`, `alt_num_clusters`, `tech_groups`, `regional_no_grouping` | Control how PG clusters existing generators by technology and region. The GUI sets these based on your aggregation choices. |
| `group_technologies` | Whether to merge similar technology variants into a single group before clustering. |
| `derate_techs` | Technologies whose capacity gets derated based on historical performance. **Only takes effect if `derate_capacity: true` is also set** (the reference does not set it; add it if you want derating). **(also read by Switch glue)** |
| `capacity_factor_default_year_filter` | Range of historical years used to estimate capacity factors. |
| `small_hydro`, `small_hydro_mw`, `small_hydro_regions` | Whether to treat small hydro as a separate category, the threshold MW, and which regions have small hydro. Pass 3 of the build cell remaps `small_hydro_regions` from p-regions to your aggregated model regions. |
| `hydro_factor`, `regional_hydro_factor` | Global and per-region multipliers applied to hydro capacity factors. Pass 3 of the build cell remaps the per-region keys when needed. **(also read by Switch glue: passed to PG's hydro adjustment function)** |
| `energy_storage_duration` | Default storage durations (hours) for new battery technologies. |
| `atb_existing_year` | Which ATB vintage to use for existing-plant cost lookups (separate from `atb_data_year`, which is for new builds). |
| `atb_data_year` | Which ATB vintage to use for new-build technology costs. **(also read by Switch glue: written as `base_financial_year` in Switch's `financials.csv`)** |
| `atb_financial_case` | ATB financial assumption set (typically `"Market"`). |
| `atb_cap_recovery_years` | Default capital recovery period for annuity calculations. |
| `alt_atb_cap_recovery_years`, `alt_resource_cap_recovery_years` | Per-technology overrides for capital recovery years. |
| `atb_modifiers` | Cost/performance overrides applied to specific ATB technologies (e.g., custom battery settings). |
| `atb_battery_wacc` | Battery-specific WACC override. |
| `modified_atb_new_gen` | Custom new-build technologies not in ATB (e.g., the `Imports` virtual resource). |
| `additional_technologies_fn`, `additional_new_gen` | Path to and content of a CSV with additional custom technologies beyond ATB. |
| `new_gen_not_available` | List of new-build technologies to explicitly disable. |
| `cost_multiplier_fn` | File containing regional capital cost adjustment factors from AEO. |
| `cost_multiplier_region_map` | Maps cost-multiplier regions to model regions. Pass 2 of the build cell automatically reconstructs this from p-regions to your aggregated model regions, with tie-breaking when an aggregated region spans multiple cost zones. |
| `cost_multiplier_technology_map` | Maps technologies to cost-multiplier categories. |
| `eia_atb_tech_map` | Maps EIA technology names to NREL ATB technology names. Pass 4 of the build cell auto-fills missing entries when the GUI adds new-build techs to `startup_fuel_use`. |
| `proposed_status_included` | Which EIA 860m project statuses to include as "planned" capacity. |
| `proposed_gen_heat_rates`, `proposed_min_load` | Default attributes for planned generators when EIA doesn't report them. |
| `additional_retirements` | Manual retirement schedule additions beyond what EIA reports. |
| `resource_attr_modifiers` | Corrections to existing plant attributes (e.g., coal FOM adjustments). |
| `renewables_clusters` | How renewable sites are clustered into bins by capacity factor and cost. |
| `shorten_cluster_cache_paths` | Workaround for filesystem path length limits when caching cluster results. |

### Optional keys not in the reference

These are read by `pg_to_switch.py` if you set them, but the reference omits them:

| Key | What it does |
|---|---|
| `derate_capacity` | Master toggle for the `derate_techs` derating logic. Set to `true` to enable. |
| `retirement_ages` | Dict of `{technology: max_age_years}` for forced retirement based on plant age. Set to `~` (null) to disable forced retirement entirely (used in economic-retirement scenarios). |

### EIA technology names

The 28 canonical technology names from EIA Form 860 are what PowerGenome uses internally to look up existing plants. The `eia_atb_tech_map` setting maps these to NREL ATB technology names. If the GUI adds new-build technologies to `startup_fuel_use`, their name prefixes must also appear in `eia_atb_tech_map` or PowerGenome will crash — Pass 4 of the build cell handles this automatically. ATB technology names (~890 combinations in `technology_costs_nrelatb`) are too numerous to list; use the GUI's ATB picker or browse the [NREL ATB documentation](https://atb.nrel.gov/electricity/2024b/technologies) for available options.

See [PG docs: existing generators](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/existing-generators/) and [PG docs: new build resources](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/new-build/).

In [17]:
ref_res = load_yml(REF_SETTINGS_DIR / "resources.yml")
gui_res_raw = load_yml(GUI_SETTINGS_DIR / "resources.yml")

print("Translating GUI keys...")
gui_res = translate_gui_dict(gui_res_raw)

print(f"\nRef keys ({len(ref_res)}):  {sorted(ref_res.keys())}")
print(f"GUI keys ({len(gui_res)}):  {sorted(gui_res.keys())}")

only_ref = sorted(set(ref_res) - set(gui_res))
only_gui = sorted(set(gui_res) - set(ref_res))
both = sorted(set(ref_res) & set(gui_res))

print(f"\nOnly in ref ({len(only_ref)}): {only_ref}")
print(f"Only in GUI ({len(only_gui)}): {only_gui}")
print(f"In both ({len(both)}): {both}")

Translating GUI keys...
  Dropped (v0.8-only): ['cache_resource_clusters', 'use_resource_clusters_cache']
  Renamed:
    resource_data_year → atb_data_year
    resource_financial_case → atb_financial_case
    resource_cap_recovery_years → atb_cap_recovery_years
    new_resources → atb_new_gen
    resource_modifiers → atb_modifiers

Ref keys (43):  ['GEM_coal_tracker_fn', 'MaxCapReq', 'additional_new_gen', 'additional_retirements', 'additional_technologies_fn', 'alt_atb_cap_recovery_years', 'alt_num_clusters', 'alt_resource_cap_recovery_years', 'atb_battery_wacc', 'atb_cap_recovery_years', 'atb_data_year', 'atb_existing_year', 'atb_financial_case', 'atb_modifiers', 'atb_new_gen', 'capacity_col', 'capacity_factor_default_year_filter', 'cluster_with_retired_gens', 'cost_multiplier_fn', 'cost_multiplier_region_map', 'cost_multiplier_technology_map', 'derate_techs', 'eia_860m_fn', 'eia_atb_tech_map', 'eia_data_years', 'energy_storage_duration', 'group_technologies', 'hydro_factor', 'modifie

### Building `resources.yml` in four passes

The reference and the GUI both contribute to this file, but in different ways. We process it in four passes, each handling one logical concern:

1. **Override simple keys** where the GUI's choice is unambiguously yours: which ATB technologies to allow, how many clusters per region, interconnection costs, etc.
2. **Remap `cost_multiplier_region_map`** from p-regions to your aggregated model regions. This is the messy one because cost multipliers apply at a different geographic resolution than your model regions, so aggregation can produce ambiguous assignments that need a tie-breaking rule.
3. **Remap `small_hydro_regions`** using the same p-region → model-region mapping built in pass 2.
4. **Validate `startup_fuel_use`** against `eia_atb_tech_map`. If the GUI added a new-build tech (e.g., `NaturalGas_1-on-1`) without adding it to the EIA→ATB mapping, PowerGenome's `startup_fuel()` function will crash. We auto-fill the obvious ones and flag anything ambiguous.

Each pass is its own cell below.

In [18]:
# Pass 1: Override reference values with GUI choices for simple keys
# ----------------------------------------------------------------------
# These keys all represent direct model-design choices made in the GUI.
# We start from the reference (which has ~89 keys we need to keep) and
# only replace the keys listed below.

res_settings = copy.deepcopy(ref_res)

gui_override_keys = [
    # New-build technology selections (translated from v0.8 names)
    "atb_new_gen",             # was: new_resources
    "atb_modifiers",           # was: resource_modifiers
    "atb_data_year",           # was: resource_data_year
    "atb_financial_case",      # was: resource_financial_case
    "atb_cap_recovery_years",  # was: resource_cap_recovery_years

    # Existing-generator clustering, set up by the GUI for your regions
    "num_clusters",
    "alt_num_clusters",
    "tech_groups",
    "renewables_clusters",
    "energy_storage_duration",
    "regional_hydro_factor",

    # Interconnection cost
    "interconnect_capex_mw",
]

print("=== Pass 1: GUI overrides ===")
for key in gui_override_keys:
    if key not in gui_res:
        print(f"  {key}: NOT IN GUI (keeping reference)")
        continue
    old_size = len(str(res_settings.get(key, "")))
    new_size = len(str(gui_res[key]))
    res_settings[key] = gui_res[key]
    print(f"  {key}: ref({old_size} chars) → gui({new_size} chars)")

# # Prepend region: all defaults to renewables_clusters (applied before regional overrides)
# # #LAZARD UPPER QUARTILES FOR 2025, ADJUSTED TO 2017 DOLLARS USING GDP DEFLATOR
# # https://www.lazard.com/media/5tlbhyla/lazards-lcoeplus-june-2025-_vf.pdf#page=9
# lazard_override_wind = 86*0.76 # $86/MWh × 0.76 ($2025 --> $2017)
# lazard_override_solar = 78*0.76 # $78/MWh × 0.76 ($2025 --> $2017)

# # # LAZARD AVERAGE FOR 2025, ADJUSTED TO 2017 DOLLARS USING GDP DEFLATOR
# # https://www.lazard.com/media/5tlbhyla/lazards-lcoeplus-june-2025-_vf.pdf#page=14
# lazard_override_wind = 61*0.76 # $86/MWh × 0.76 ($2025 --> $2017)
# lazard_override_solar = 58*0.76 # $78/MWh × 0.76 ($2025 --> $2017)

# # # LAZARD LOWER QUARTILES FOR 2025, ADJUSTED TO 2017 DOLLARS USING GDP DEFLATOR
# # # https://www.lazard.com/media/5tlbhyla/lazards-lcoeplus-june-2025-_vf.pdf#page=9
# # lazard_override_wind = 37*0.76 # $37/MWh × 0.76 ($2025 --> $2017)
# # lazard_override_solar = 38*0.76 # $38/MWh × 0.76 ($2025 --> $2017)

# all_region_defaults = [
    
#     {"region": "all",
#     "technology": "landbasedwind",
#      "filter": [{"feature": "lcoe", "max": lazard_override_wind}],
#      "bin": [{"feature": "lcoe", "weights": "mw", "q": 3}]},

#     {"region": "all", 
#      "technology": "utilitypv",
#      "filter": [{"feature": "lcoe", "max": lazard_override_solar}],
#      "bin": [{"feature": "lcoe", "weights": "mw", "q": 6}]},

#     {"region": "all", 
#      "technology": "offshorewind", "turbine_type": "fixed", "pref_site": 1,
#      "filter": [{"feature": "lcoe", "max": 125}],
#      "bin": [{"feature": "lcoe", "weights": "mw", "q": 2}]},

#     {"region": "all", 
#     "technology": "offshorewind", "turbine_type": "fixed", "pref_site": 0,
#      "filter": [{"feature": "lcoe", "max": 125}],
#      "bin": [{"feature": "lcoe", "weights": "mw", "q": 2}]},

#     {"region": "all", 
#      "technology": "offshorewind", "turbine_type": "floating", "pref_site": 0,
#      "filter": [{"feature": "lcoe", "max": 125}],
#      "bin": [{"feature": "lcoe", "weights": "mw", "q": 2}]},

#     {"region": "all", 
#      "technology": "imports"},
# ]
# gui_rc = res_settings.get("renewables_clusters") or []

# def _lcoe_max(entry):
#     for f in entry.get("filter") or []:
#         if f.get("feature") == "lcoe" and "max" in f:
#             return f["max"]
#     return None

# def _key(entry):
#     return (entry.get("technology"), entry.get("turbine_type"), entry.get("pref_site"))

# # Default-key -> default lcoe_max threshold
# default_thresholds = {_key(d): _lcoe_max(d) for d in all_region_defaults}
# default_keys = set(default_thresholds)

# gui_rc_filtered = []
# dropped_all = 0
# dropped_below = []  # (region, tech, region_max, default_max)
# for e in gui_rc:
#     k = _key(e)
#     # (1) drop pre-existing region:all entries that collide with our defaults
#     if e.get("region") == "all" and k in default_keys:
#         dropped_all += 1
#         continue
#     # (2) drop regional entries whose lcoe_max is below the default's threshold -
#     #     the region:all default is less restrictive, so let it handle them
#     thr = default_thresholds.get(k)
#     reg_max = _lcoe_max(e)
#     if (thr is not None and reg_max is not None
#             and e.get("region") != "all" and reg_max < thr):
#         dropped_below.append((e.get("region"), e.get("technology"), reg_max, thr))
#         continue
#     gui_rc_filtered.append(e)

# res_settings["renewables_clusters"] = all_region_defaults + gui_rc_filtered
# print(f"  renewables_clusters: prepended {len(all_region_defaults)} region:all defaults "
#       f"(dropped {dropped_all} colliding all-entries, "
#       f"dropped {len(dropped_below)} regional below-threshold, "
#       f"kept {len(gui_rc_filtered)} regional)")
# if dropped_below:
#     print("  Regional entries replaced by region:all default (region_max < default_max):")
#     for region, tech, rm, thr in sorted(dropped_below, key=lambda x: (x[1], x[2])):
#         print(f"    {tech:15s} {region:25s} {rm:7.2f} < {thr}")

=== Pass 1: GUI overrides ===
  atb_new_gen: ref(419 chars) → gui(673 chars)
  atb_modifiers: ref(520 chars) → gui(1269 chars)
  atb_data_year: ref(4 chars) → gui(4 chars)
  atb_financial_case: ref(6 chars) → gui(6 chars)
  atb_cap_recovery_years: ref(2 chars) → gui(2 chars)
  num_clusters: ref(570 chars) → gui(227 chars)
  alt_num_clusters: NOT IN GUI (keeping reference)
  tech_groups: ref(185 chars) → gui(185 chars)
  renewables_clusters: ref(911 chars) → gui(22981 chars)
  energy_storage_duration: ref(116 chars) → gui(54 chars)
  regional_hydro_factor: ref(63 chars) → gui(16 chars)
  interconnect_capex_mw: ref(0 chars) → gui(6 chars)


### Renewable site clustering: filter, bin, and cluster decisions

The `renewables_clusters` setting controls how PowerGenome collapses the hundreds of thousands of candidate sites in `resource_groups/` down to a small number of representative clusters that the capacity expansion model will see. This cell overrides whatever per-region spec came from the GUI with a single, documented `region: all` spec that applies uniformly across all 46 model regions.

Three decisions shape the output, each controlled by a toggle at the top of the cell:

1. **Whether to filter sites by LCOE** before clustering (`APPLY_LCOE_FILTER`)
2. **Whether to partition sites into bins** on a physical feature before clustering (`APPLY_BIN`)
3. **How to cluster** the surviving sites into representative profiles (`APPLY_CLUSTER`)

#### Decision 1: No LCOE filter (`APPLY_LCOE_FILTER = False`)

PowerGenome's default pattern, inherited from the GUI, is to drop sites with LCOE above a threshold (e.g., Lazard's average) before clustering. This is fine for reference runs, but it contaminates any downstream analysis that internalizes externalities. Our optimizer is explicitly re-pricing generation through air-quality health damages; if we pre-filter sites on a *private* cost basis and then ask the model to solve on a *social* cost basis, we've baked in one cost structure while solving for another. Sites that look uneconomic on a pure-LCOE basis — for instance, high-CF sites in remote locations with expensive interconnection but low downwind population exposure — may be precisely the sites a health-aware optimizer wants to build.

Keeping all sites preserves the full supply curve and lets the cost-minimization step see the same site-level economic heterogeneity that the damage-cost step sees.

#### Decision 2: Bin by capacity factor, capacity-weighted quantiles

Without filtering, all 405,479 utility PV sites and 76,739 land-based wind sites flow into the clustering step. PowerGenome's profile-based clustering uses scikit-learn's `AgglomerativeClustering`, which is *O(n²)* in memory — it builds a pairwise distance matrix between every pair of sites in 8,760-dimensional profile space. For our largest solar region (e_WY_and_w_SD, ~29k sites) that matrix alone would be ~7 GB; without binning we observed the pipeline hanging for 24+ hours while swapping to disk. Wind site counts are roughly 5× smaller per region, so wind is less constrained computationally but benefits from the same treatment for consistency and methodological simplicity.

Binning is the standard workaround: partition sites into groups on a scalar feature and run the clustering independently within each group. The per-bin cost is *O((n/k)²)* per bin, *O(n²/k)* overall, for an order-*k* speedup in both time and memory.

**Why CF, not LCOE.** Both are supported by PowerGenome's `value_bin` function, but CF is a physical resource-quality feature — the same value regardless of cost scenario. LCOE is a derived economic quantity that depends on capex, discount rate, and interconnection cost assumptions; binning on it would reintroduce the economic filtering we excluded in Decision 1. CF binning is a pure dimensionality reduction that preserves every site's capacity.

**Why quantile (`q`), not equal-width (`bins`).** CF distributions are heavily skewed across candidate sites — a long tail of low-CF marginal sites and a concentration near the regional resource peak. Equal-width bins (`pd.cut`) put the majority of sites in a single middle bin, defeating the purpose. Quantile cuts guarantee roughly equal site count per bin, and capacity-weighted quantiles (`weights: mw`) guarantee roughly equal capacity per bin, which is the physically meaningful partition.

**Why three bins.** We profiled both input parquets empirically and verified that `q: 3, weights: mw` keeps the worst-case bin in any region under manageable limits: ~9,500 sites for solar (e_WY_and_w_SD) and ~2,330 sites for wind. Agglomerative clustering on ~10k sites runs in a minute or two on this hardware. Increasing to four or five bins would be faster still but produce too many final clusters when multiplied through the cluster step. Three bins is the coarsest partition that keeps the pipeline tractable while stratifying sites across the low / mid / high CF distribution by capacity.

#### Decision 3: Three profile clusters per bin (two for offshore)

Within each CF bin, we cluster on full hourly generation profiles with `method: agg`. Profile clustering groups sites by *when* they generate, not just how much — a Great Plains wind site peaking at night ends up in a different cluster from a coastal site peaking in the afternoon, even at the same annual CF. This temporal diversity is what drives storage sizing, transmission expansion, and residual-load-shape-sensitive dispatch decisions — which in turn drive the spatial pattern of fossil dispatch, and therefore damages.

**Final cluster counts.** For land-based wind and utility PV: 3 CF bins × 3 profile clusters = **9 clusters per region**, ~414 representative clusters across 46 regions per tech. Each cluster's profile is a capacity-weighted average of its member sites (`calc_cluster_values` in `renewables.py`), and the cluster's capacity is the sum of member capacities. No sites and no capacity are dropped at any stage.

**Offshore wind is unbinned.** Per-region offshore site counts are in the low thousands, where agglomerative profile clustering is computationally cheap without any pre-binning. We set `n_clusters: 2` for each of the three offshore specs (fixed-preferred, fixed-non-preferred, floating-non-preferred), since offshore resource is comparatively homogeneous within a region and we prefer simpler representations when we can afford them.

#### How the emitted YAML looks

After `_make_entry` processes each tech, the `renewables_clusters` block in the final `resources.yml` contains:

```yaml
renewables_clusters:
  - region: all
    technology: landbasedwind
    bin:
      - feature: cf
        weights: mw
        q: 3
    cluster:
      - feature: profile
        method: agg
        n_clusters: 3
  - region: all
    technology: utilitypv
    bin:
      - feature: cf
        weights: mw
        q: 3
    cluster:
      - feature: profile
        method: agg
        n_clusters: 3
  - region: all
    technology: offshorewind
    turbine_type: fixed
    pref_site: 1
    cluster:
      - feature: profile
        method: agg
        n_clusters: 2
  # ... two more offshore specs (fixed/pref_site:0, floating/pref_site:0), same structure
  - region: all
    technology: imports
```

PowerGenome expands each `region: all` entry internally to cover every region in `model_definition.yml`, so we never write per-region overrides.

#### Modifying these decisions

- **Restore LCOE filtering** (e.g., for a pure-cost reference run): set `APPLY_LCOE_FILTER = True`. Thresholds are in `TECH_PARAMS`; the default wind and solar thresholds are Lazard 2025 averages converted to 2017\$ via the GDP deflator.
- **Change cluster counts**: edit the `n_clusters` dict in `CLUSTER_SPECS`. Either an int (uniform across techs) or a dict keyed by technology.
- **Change the bin feature** (e.g., to `interconnect_capex_mw` for cost-stratified bins, or `cpa_mw` for site-size bins): edit `BIN_SPECS`. The comment block at the top of the cell lists every numeric column available on each schema.
- **Stack multiple bin or cluster features**: append additional dicts to `BIN_SPECS` or `CLUSTER_SPECS`. PowerGenome nests them in the order given — the second bin feature subdivides within bins of the first.
- **Swap from quantile to equal-width binning**: replace `"q"` with `"bins"` in the `BIN_SPECS` entry. Only use this if the feature has a roughly uniform distribution.

In [19]:
# ----------------------------------------------------------------------
# Config: region:all renewable clustering overrides
# ----------------------------------------------------------------------
OVERRIDE_ALL_REGIONS = True    # False → skip prepend entirely, keep GUI rc as-is
DROP_GUI_REGIONAL    = True   # True  → drop ALL regional GUI entries; keep only region:all
APPLY_LCOE_FILTER    = False    # filter:  [{feature: <filter>, max: ...}]
APPLY_BIN            = False    # bin:     [{feature: ..., weights: ..., q: N}, ...] can't bin on profile; disable binning entirely
APPLY_CLUSTER        = True   # cluster: [{feature: ..., method: ..., n_clusters: N}, ...]

# When APPLY_BIN and APPLY_CLUSTER are both True, PowerGenome bins first and
# then clusters WITHIN each bin — so you get (q × n_clusters) groups per tech.

# ----------------------------------------------------------------------
# Available options — confirmed from resource_groups file schemas
# Onshore (solar/wind) parquets: 16 shared cols.
# Offshore CSV: different schema, many offshore-specific features.
# ----------------------------------------------------------------------
# NUMERIC FEATURES — SHARED across solar / onshore wind / offshore wind:
#   "lcoe"                   — levelized cost of electricity ($/MWh)
#   "cf"                     — capacity factor (fraction 0-1)
#   "cpa_mw"                 — CPA capacity (MW)
#   "interconnect_capex_mw"  — interconnection capex ($/MW)
#   "total_interconnect_km"  — total interconnection distance (km)
#
# NUMERIC FEATURES — ONSHORE ONLY (solar + land-based wind):
#   "capacity_mw"            — site capacity (MW; offshore uses "cap_mw")
#   "path_len"               — interconnect path length
#   "m_popden"               — population density
#   "anyQual" / "exFacil" / "plFacil"  — CERF siting qualification flags
#
# NUMERIC FEATURES — OFFSHORE WIND ONLY:
#   "cap_mw"                 — site capacity (MW; equivalent to capacity_mw)
#   "area_km2"               — site area
#   "d_shore_m"              — distance to shore (m) — canonical offshore cost driver
#   "d_trans" / "d_sub" / "d_road" / "d_existing" / "d_load_750" / "d_EIA860fa"
#                            — various distance-to-infrastructure metrics
#   "m_seafloor"             — seafloor depth (drives fixed vs floating feasibility)
#   "BathySlope"             — bathymetric slope
#   "m_CF_MEAN" / "m_CF_wLoss"  — capacity factor metrics (wLoss includes wake/elec losses)
#   "anngen_MWh"             — annual generation
#   "interconnect_annuity"   — annualized interconnection cost
#   "resource_annuity" / "resource_fom"  — resource-side economics
#   "offshore_interconnect_km"  — offshore portion of interconnect distance
#   "ocean_impa" / "ocean_im_1" / "ocean_im_2" / "HAPC_Area_"  — ocean impact metrics
#   "major_port" / "incap" / "prefSite"  — port/siting flags
#
# SPECIAL:
#   "profile"   — hourly generation shape. CLUSTER ONLY. Added at runtime from
#                 profile_path (renewables.py L506-509). Method MUST be agg.
#   "mw"        — shorthand for site capacity; valid in `weights` only. Also
#                 works: "capacity_mw" (onshore), "cap_mw" (offshore).
#
# CATEGORICAL FEATURES (usable for `group` only, not bin/cluster):
#   Onshore:  "region", "msa_id", "msa_name", "path"
#   Offshore: "ipm_region", "nearest_region", "metro_id", "metro_region",
#             "turbine_ty" (=turbine_type), "tech", "path"
#
# CLUSTER METHODS:
#   "agg" / "agglomerative" / "hierarchical"  — AgglomerativeClustering
#                                               REQUIRED when feature="profile"
#   "kmeans"                                  — KMeans (numeric features only)
#
# BIN OPTIONS (pick ONE per spec):
#   "q": int or [edges]         — quantile bins (weighted if weights given)
#   "bins": int or [edges]      — equal-width or custom-edge value bins
#   "mw_per_q" / "mw_per_bin"   — auto-derive q/bins from total capacity
#
# NOTE: feature names are passed through snake_case_str() in renewables.py
#       (L484, L514, etc.), so CamelCase/mixed-case column names are normalized.
#       "BathySlope" → "bathy_slope" when referenced; match whatever PowerGenome
#       loads as the DataFrame column (check with a quick print if unsure).
# ----------------------------------------------------------------------
# Lazard 2025 average LCOE → 2017$ (GDP deflator 0.76)
lazard_override_wind  = 61 * 0.76
lazard_override_solar = 58 * 0.76
LCOE_MAX_OFFSHORE     = 125

# Per-technology filter thresholds (only used when APPLY_LCOE_FILTER is True)
TECH_PARAMS = {
    "landbasedwind": {"lcoe_max": lazard_override_wind},
    "utilitypv":     {"lcoe_max": lazard_override_solar},
    "offshorewind":  {"lcoe_max": LCOE_MAX_OFFSHORE},
}

# Feature used for the filter clause (e.g. swap to "cf" with a min threshold)
FILTER_FEATURE = "lcoe"

# ----------------------------------------------------------------------
# Bin / cluster specs — each entry becomes one clause in the output YAML.
# q / bins / n_clusters accept either:
#   - int  → applied to every tech
#   - dict → keyed by technology; techs not in dict are skipped
# Add more dicts to stack features (PowerGenome nests them in order).
# ----------------------------------------------------------------------
BIN_SPECS = [
    {
        "feature": "lcoe",
        "weights": "mw",
        "q": {"landbasedwind": 3, "utilitypv": 6, "offshorewind": 2},
        # Alternative: "bins": 4  (int) or [0, 25, 50, 100] (edges)
    },
    # Example second bin — uncomment to nest by interconnect cost within LCOE bins:
    # {"feature": "interconnect_annuity", "weights": "mw", "q": 2},
]
BIN_SPECS = [

]
# ----------------------------------------------------------------------
# Cluster spec — profile shape only, no financial features
# ----------------------------------------------------------------------
CLUSTER_SPECS = [
    {"feature": "profile", "method": "agg",
     "n_clusters": {"landbasedwind": 9, "utilitypv": 9, "offshorewind": 2}},
]

# BIN_SPECS is unused when APPLY_BIN=False — leave as-is or empty.
# TECH_PARAMS is unused when APPLY_LCOE_FILTER=False — same.
def _resolve_count(val, tech):
    """Accept int (uniform across techs) or dict (per-tech)."""
    if isinstance(val, dict):
        return val.get(tech)
    return val

def _make_entry(**fields):
    """Build a region:all entry honoring toggles, TECH_PARAMS, and *_SPECS."""
    tech = fields["technology"]
    params = TECH_PARAMS.get(tech, {})
    entry = {"region": "all", **fields}

    if APPLY_LCOE_FILTER and "lcoe_max" in params:
        entry["filter"] = [{"feature": FILTER_FEATURE, "max": params["lcoe_max"]}]

    if APPLY_BIN:
        bins_out = []
        for spec in BIN_SPECS:
            q    = _resolve_count(spec.get("q"),    tech)
            bins = _resolve_count(spec.get("bins"), tech)
            if q is None and bins is None:
                continue
            b = {"feature": spec["feature"]}
            if "weights" in spec:
                b["weights"] = spec["weights"]
            if q is not None:
                b["q"] = q
            if bins is not None:
                b["bins"] = bins
            bins_out.append(b)
        if bins_out:
            entry["bin"] = bins_out

    if APPLY_CLUSTER:
        cluster_out = []
        for spec in CLUSTER_SPECS:
            n = _resolve_count(spec.get("n_clusters"), tech)
            if n is None:
                continue
            c = {"feature": spec["feature"], "n_clusters": n}
            if "method" in spec:
                c["method"] = spec["method"]
            cluster_out.append(c)
        if cluster_out:
            entry["cluster"] = cluster_out

    return entry

all_region_defaults = [
    _make_entry(technology="landbasedwind"),
    _make_entry(technology="utilitypv"),
    _make_entry(technology="offshorewind", turbine_type="fixed",    pref_site=1),
    _make_entry(technology="offshorewind", turbine_type="fixed",    pref_site=0),
    _make_entry(technology="offshorewind", turbine_type="floating", pref_site=0),
    {"region": "all", "technology": "imports"},
]

# ----------------------------------------------------------------------
# Merge with GUI entries
# ----------------------------------------------------------------------
if not OVERRIDE_ALL_REGIONS:
    print(f"  renewables_clusters: OVERRIDE_ALL_REGIONS=False → GUI kept unchanged "
          f"({len(res_settings.get('renewables_clusters') or [])} entries)")
elif DROP_GUI_REGIONAL:
    gui_rc = res_settings.get("renewables_clusters") or []
    res_settings["renewables_clusters"] = list(all_region_defaults)
    enabled = ", ".join(n for n, on in [("filter", APPLY_LCOE_FILTER),
                                        ("bin", APPLY_BIN),
                                        ("cluster", APPLY_CLUSTER)] if on) or "none"
    print(f"  renewables_clusters: DROP_GUI_REGIONAL=True → dropped all {len(gui_rc)} "
          f"GUI entries, kept only {len(all_region_defaults)} region:all defaults "
          f"(enabled: {enabled})")
else:
    gui_rc = res_settings.get("renewables_clusters") or []
    def _lcoe_max(entry):
        for f in entry.get("filter") or []:
            if f.get("feature") == "lcoe" and "max" in f:
                return f["max"]
        return None

    def _key(entry):
        return (entry.get("technology"), entry.get("turbine_type"), entry.get("pref_site"))

    # Build thresholds from TECH_PARAMS directly so collision logic works even
    # when APPLY_LCOE_FILTER is False (defaults have no filter, but we still
    # know what threshold *would* apply).
    default_thresholds = {}
    for d in all_region_defaults:
        tech = d.get("technology")
        if tech in TECH_PARAMS:
            default_thresholds[_key(d)] = TECH_PARAMS[tech].get("lcoe_max")
    default_keys = set(_key(d) for d in all_region_defaults)

    gui_rc_filtered = []
    dropped_all = 0
    dropped_below = []
    for e in gui_rc:
        k = _key(e)
        # (1) drop pre-existing region:all entries that collide with our defaults
        if e.get("region") == "all" and k in default_keys:
            dropped_all += 1
            continue
        # (2) drop regional entries whose lcoe_max is below the default's —
        #     the region:all default is less restrictive, so let it handle them.
        #     Only meaningful when we're actually emitting a filter.
        if APPLY_LCOE_FILTER:
            thr = default_thresholds.get(k)
            reg_max = _lcoe_max(e)
            if (thr is not None and reg_max is not None
                    and e.get("region") != "all" and reg_max < thr):
                dropped_below.append((e.get("region"), e.get("technology"), reg_max, thr))
                continue
        gui_rc_filtered.append(e)

    res_settings["renewables_clusters"] = all_region_defaults + gui_rc_filtered

    enabled = ", ".join(n for n, on in [("filter", APPLY_LCOE_FILTER),
                                        ("bin", APPLY_BIN),
                                        ("cluster", APPLY_CLUSTER)] if on) or "none"
    print(f"  renewables_clusters: prepended {len(all_region_defaults)} region:all defaults "
          f"(enabled: {enabled})")
    print(f"    dropped {dropped_all} colliding all-entries, "
          f"{len(dropped_below)} regional below-threshold, "
          f"kept {len(gui_rc_filtered)} regional")
    if dropped_below:
        print("  Regional entries replaced by region:all default (region_max < default_max):")
        for region, tech, rm, thr in sorted(dropped_below, key=lambda x: (x[1], x[2])):
            print(f"    {tech:15s} {region:25s} {rm:7.2f} < {thr}")

  renewables_clusters: DROP_GUI_REGIONAL=True → dropped all 91 GUI entries, kept only 6 region:all defaults (enabled: cluster)


### Pass 2: Remapping `cost_multiplier_region_map`

PowerGenome adjusts capital costs by region using a `cost_multiplier_region_map` that lists which p-regions belong to each cost multiplier zone (defined by AEO). The reference file is keyed by raw p-regions:

```yaml
cost_multiplier_region_map:
  ECCA:  [p1, p2, p3, p4]
  WECC:  [p33, p34, p35]
  ...
```

When you aggregate p-regions in the GUI (e.g., `WA1 = [p1, p2]`, `WA2 = [p3, p4]`), this map needs to point at your aggregated names instead. Most of the remapping is mechanical: replace each p-region with its parent model region and dedupe. But two edge cases come up:

**Edge case 1: The same model region ends up in multiple cost zones.** If `WA1 = [p1, p2]` and the reference puts `p1` in `ECCA` but `p2` in `WECC`, then `WA1` is now in both. PowerGenome expects each model region to live in exactly one cost zone, so we need a tie-breaking rule.

**Edge case 2: A p-region in the reference map isn't in any GUI aggregation.** This happens if the GUI's region setup excludes some p-regions entirely. We just record those as missing and skip them.

#### Tie-breaking rule

When a model region is assigned to multiple cost zones, we keep it in the cost zone where the most of its constituent p-regions originally lived. So if `WA1 = [p1, p2, p3]` with two of those in `ECCA` and one in `WECC`, `WA1` stays in `ECCA`. If there's still a tie (e.g., `WA1 = [p1, p2]` split 1-1), we keep it in whichever cost zone has the smaller assigned region list, which keeps the cost zone groupings as balanced as possible.

This is a heuristic. If your aggregation produces a lot of ties, you may want to inspect the warnings and override the result manually.

In [20]:
# Pass 2: Remap cost_multiplier_region_map from p-regions to model regions
# ----------------------------------------------------------------------
# This also builds p_to_model_region, which Pass 3 reuses for small_hydro.

print("\n=== Pass 2: cost_multiplier_region_map ===")

region_aggregations = gui_md.get("region_aggregations", {})

if not (region_aggregations and "cost_multiplier_region_map" in ref_res):
    print("  region_aggregations not found; keeping reference value")
else:
    # Build reverse map: p_region -> aggregated model region.
    # Warn if any p-region appears in more than one aggregation
    # (shouldn't happen with a well-formed GUI export, but worth checking).
    p_to_model_region = {}
    duplicate_p_assignments = {}
    for model_region, p_list in region_aggregations.items():
        for p in p_list:
            if p in p_to_model_region:
                duplicate_p_assignments.setdefault(
                    p, [p_to_model_region[p]]
                ).append(model_region)
            else:
                p_to_model_region[p] = model_region

    if duplicate_p_assignments:
        print("  ⚠ p-regions appearing in multiple aggregations:")
        for p, regions in sorted(duplicate_p_assignments.items()):
            print(f"    {p}: {regions}")

    # Translate each cost zone's p-region list into model regions.
    # raw_cost_region_map keeps duplicates (used later for tie-breaking);
    # updated_cost_multiplier_region_map is the deduped version we save.
    raw_cost_region_map = {}
    updated_cost_multiplier_region_map = {}
    missing_p_regions = {}

    for cost_region, p_list in ref_res["cost_multiplier_region_map"].items():
        mapped = []
        missing = []
        for p in p_list:
            target = p_to_model_region.get(p)
            if target is None:
                missing.append(p)
            else:
                mapped.append(target)
        raw_cost_region_map[cost_region] = mapped
        updated_cost_multiplier_region_map[cost_region] = list(dict.fromkeys(mapped))
        if missing:
            missing_p_regions[cost_region] = missing

    # Find model regions that ended up in more than one cost zone
    model_region_to_cost_regions = {}
    for cost_region, model_regions in updated_cost_multiplier_region_map.items():
        for mr in model_regions:
            model_region_to_cost_regions.setdefault(mr, []).append(cost_region)

    duplicated = {
        mr: crs for mr, crs in model_region_to_cost_regions.items() if len(crs) > 1
    }

    if not duplicated:
        print("  ✓ No model regions assigned to multiple cost zones")
    else:
        print(f"  ⚠ {len(duplicated)} model regions in multiple cost zones; resolving:")
        for mr, candidate_zones in sorted(duplicated.items()):
            # Tie-breaker 1: cost zone where this model region appears most often
            counts = {cz: raw_cost_region_map[cz].count(mr) for cz in candidate_zones}
            max_count = max(counts.values())
            top_zones = [cz for cz, c in counts.items() if c == max_count]

            # Tie-breaker 2: smallest assigned list (keeps zones balanced)
            if len(top_zones) == 1:
                keep_zone = top_zones[0]
                reason = f"highest count ({max_count})"
            else:
                keep_zone = min(
                    top_zones,
                    key=lambda cz: len(updated_cost_multiplier_region_map[cz]),
                )
                reason = f"tied at {max_count}, kept smallest list"

            dropped = [cz for cz in candidate_zones if cz != keep_zone]
            print(f"    {mr}: kept in {keep_zone} ({reason}); dropped from {dropped}")

            for cz in dropped:
                updated_cost_multiplier_region_map[cz] = [
                    r for r in updated_cost_multiplier_region_map[cz] if r != mr
                ]

    if missing_p_regions:
        print("  ⚠ p-regions in reference map not found in GUI aggregations:")
        for cz, missing in missing_p_regions.items():
            print(f"    {cz}: {missing}")

    res_settings["cost_multiplier_region_map"] = updated_cost_multiplier_region_map
    print(f"  ✓ remapped {len(updated_cost_multiplier_region_map)} cost zones")


=== Pass 2: cost_multiplier_region_map ===
  ⚠ 12 model regions in multiple cost zones; resolving:
    AL_and_e_MS_and_FL_pnh: kept in SRSE (highest count (3)); dropped from ['SRCE']
    Chicago_and_sw_MI: kept in PJMC (tied at 1, kept smallest list); dropped from ['PJMW']
    IA: kept in MISW (highest count (2)); dropped from ['SPPN']
    ID_and_w_MT_and_w_WY: kept in NWPP (highest count (5)); dropped from ['BASN', 'RMRG']
    IN_and_W_KY: kept in MISC (highest count (3)); dropped from ['SRCE']
    MD_and_VA_Island: kept in PJME (highest count (2)); dropped from ['PJMW']
    MN_and_e_ND: kept in MISW (highest count (4)); dropped from ['SPPN']
    NV_and_UT_and_n_CA: kept in BASN (highest count (4)); dropped from ['NWPP']
    PA: kept in PJME (highest count (3)); dropped from ['PJMW']
    VA: kept in PJMW (highest count (2)); dropped from ['PJMD']
    WI_and_w_MI: kept in MISW (highest count (6)); dropped from ['SPPN']
    n_KY: kept in SRCE (tied at 1, kept smallest list); dropped fr

### Pass 3: Remapping `small_hydro_regions` and `regional_hydro_factor`

Both of these fields can be keyed by p-regions in the reference, and both need to be remapped to model regions using `p_to_model_region` from Pass 2. `small_hydro_regions` is a flat list so we just map each entry and dedupe. `regional_hydro_factor` is a dict of `{region: multiplier}`, so we group by parent model region and warn if values disagree within a group.

`regional_hydro_factor` is also in `gui_override_keys` in Pass 1, so if the GUI provides it, it already has model-region keys and the remap is a no-op. We detect that case and skip cleanly.

In [21]:
# Pass 3: Remap small_hydro_regions using p_to_model_region from Pass 2
# ----------------------------------------------------------------------
print("\n=== Pass 3: small_hydro_regions ===")

if not (region_aggregations and "small_hydro_regions" in ref_res):
    print("  region_aggregations not found; keeping reference value")
else:
    mapped = []
    missing = []
    for p in ref_res["small_hydro_regions"]:
        target = p_to_model_region.get(p)
        if target is None:
            missing.append(p)
        else:
            mapped.append(target)

    mapped = list(dict.fromkeys(mapped))
    res_settings["small_hydro_regions"] = mapped
    print(f"  mapped {len(ref_res['small_hydro_regions'])} p-regions → {len(mapped)} model regions")

    if missing:
        print(f"  ⚠ no mapping found for: {missing}")
# --- regional_hydro_factor: remap p-region keys to model regions if needed ---
# If the GUI provided this key in Pass 1, it's already keyed by model regions
# and has_p_keys below is false, making this block a clean no-op.
rhf = res_settings.get("regional_hydro_factor")
if isinstance(rhf, dict) and region_aggregations:
    has_p_keys = any(k in p_to_model_region for k in rhf)
    if has_p_keys:
        groups = {}
        unmapped_rhf = {}
        for p, val in rhf.items():
            if p in p_to_model_region:
                groups.setdefault(p_to_model_region[p], []).append((p, val))
            else:
                unmapped_rhf[p] = val

        remapped_rhf = {}
        for model_region, entries in groups.items():
            values = [v for _, v in entries]
            if len(set(str(v) for v in values)) == 1:
                remapped_rhf[model_region] = values[0]
            else:
                remapped_rhf[model_region] = values[0]
                print(
                    f"  ⚠ regional_hydro_factor.{model_region}: disagreeing values, "
                    f"using first ({entries[0][0]}={entries[0][1]})"
                )

        remapped_rhf.update(unmapped_rhf)
        res_settings["regional_hydro_factor"] = remapped_rhf
        print(f"  regional_hydro_factor: remapped {len(rhf)} → {len(remapped_rhf)} entries")
    else:
        print("  regional_hydro_factor: already keyed by model regions, no remap needed")


=== Pass 3: small_hydro_regions ===
  mapped 134 p-regions → 46 model regions
  regional_hydro_factor: already keyed by model regions, no remap needed


### Pass 4: Validating `startup_fuel_use` against `eia_atb_tech_map`

`startup_fuel_use` (defined in `startup_costs.yml` from the GUI) lists the fuel each technology burns during startup. PowerGenome's internal `startup_fuel()` function looks up every key from this dict in `eia_atb_tech_map`, which maps EIA technology names to ATB technology names. If a key is missing from the map, the lookup returns `None` and PowerGenome crashes downstream with a confusing error.

This breaks because the GUI sometimes adds new-build tech variants (like `NaturalGas_1-on-1` or `NaturalGas_Combustion`) to `startup_fuel_use` without also adding them to `eia_atb_tech_map`. We catch the missing keys here and try to auto-fill them in two ways:

1. **Exact alias rules**: hard-coded mappings for variants we've seen before.
2. **Pattern fallbacks**: if the key starts with `NaturalGas_`, `Coal_`, or `Nuclear_`, we look up the canonical EIA name (`Natural Gas Fired Combined Cycle`, `Conventional Steam Coal`, etc.) and use whatever ATB name the reference map already has for it.

Anything we can't confidently match gets printed as a list at the end so you can resolve it manually before running the model. **If you see unresolved techs, do not skip past them — the model will crash.**

In [22]:
# Pass 4: Validate startup_fuel_use against eia_atb_tech_map
# ----------------------------------------------------------------------
print("\n=== Pass 4: startup_fuel_use validation ===")

gui_sc = load_yml(GUI_SETTINGS_DIR / "startup_costs.yml")
startup_fuel = gui_sc.get("startup_fuel_use", {}) or {}
eia_map = res_settings.get("eia_atb_tech_map", {}) or {}

# Hard-coded aliases for variants seen in past GUI exports
EXACT_STARTUP_FUEL_MAP = {
    "NaturalGas_1-on-1":     "NaturalGas_1-on-1 Combined Cycle (H-Frame)",
    "NaturalGas_Combustion": "NaturalGas_Combustion Turbine (F-Frame)",
}

def guess_atb_name(tech):
    """Try to find an ATB name for an unmapped startup_fuel_use tech.
    Returns (atb_name, reason) or (None, None) if no confident guess."""
    if tech in EXACT_STARTUP_FUEL_MAP:
        return EXACT_STARTUP_FUEL_MAP[tech], "exact alias"

    if tech.startswith("NaturalGas_"):
        t = tech.lower()
        if any(kw in t for kw in ("cc", "combined", "1-on-1", "2-on-1")):
            return eia_map.get("Natural Gas Fired Combined Cycle"), "looks like CC variant"
        if any(kw in t for kw in ("combustion", "ct", "peaker", "turbine")):
            return eia_map.get("Natural Gas Fired Combustion Turbine"), "looks like CT variant"

    if tech.startswith("Coal_"):
        return eia_map.get("Conventional Steam Coal"), "looks like coal variant"

    if tech.startswith("Nuclear_"):
        return eia_map.get("Nuclear"), "looks like nuclear variant"

    return None, None

missing = [t for t in startup_fuel if t not in eia_map]

if not missing:
    print("  ✓ All startup_fuel_use keys found in eia_atb_tech_map")
else:
    print(f"  ⚠ {len(missing)} startup_fuel_use keys missing from eia_atb_tech_map:")
    unresolved = []
    for tech in missing:
        guess, reason = guess_atb_name(tech)
        if guess:
            eia_map[tech] = guess
            print(f"    {tech}: AUTO-ADDED → '{guess}' ({reason})")
        else:
            unresolved.append(tech)
            print(f"    {tech}: ❌ no confident guess available")

    res_settings["eia_atb_tech_map"] = eia_map

    if unresolved:
        print("\n  Techs needing manual review (model WILL crash if left unfixed):")
        for tech in unresolved:
            print(f"    - {tech}")
        print("\n  Helper list for copy-paste:")
        print("  missing_startup_fuel_techs = [")
        for tech in unresolved:
            print(f'      "{tech}",')
        print("  ]")
# ----------------------------------------------------------------------
# Extend eia_atb_tech_map: broaden Conventional Steam Coal to cover IGCC variants
# ----------------------------------------------------------------------
# The EIA bucket "Conventional Steam Coal" currently maps to a single ATB tech.
# Widen it to a list so IGCC and IGCC-CCS variants inherit the same EIA mapping
# (CSV has no dedicated IGCC bucket; conventional coal is the closest available).
eia_map = res_settings.get("eia_atb_tech_map", {}) or {}

eia_extensions = {
    "Conventional Steam Coal": [
        "Coal_newAvgCF",
        "Coal_IGCCAvgCF",
        "Coal_IGCC",
        "Coal_IGCC-90%-CCS",
    ],
}

print("\n=== Extend eia_atb_tech_map ===")
for category, new_techs in eia_extensions.items():
    existing = eia_map.get(category, []) or []
    if not isinstance(existing, list):
        existing = [existing]
    for tech in new_techs:
        if tech not in existing:
            existing.append(tech)
            print(f"  Added {tech!r} under {category!r}")
        else:
            print(f"  {tech!r} already under {category!r}")
    eia_map[category] = existing

res_settings["eia_atb_tech_map"] = eia_map

# Extend cost_multiplier_technology_map with new ATB techs that have legitimate
# matching categories in AEO_2020_regional_cost_corrections.csv:
#   - LandbasedWind_Class4 → Wind (same category as Class3, different resource class)
#   - NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS → CC with 90% CCS
#     (closest available CCS-CC bucket; 95 vs 90 is below regional-multiplier resolution)
#   - Coal_IGCC variants and Coal_newHighCF → Ultra-supercritical coal (USC)
#     (CSV has no IGCC column; USC is the closest available coal bucket)
cm_map = res_settings.get("cost_multiplier_technology_map", {}) or {}

extensions = {
    "Wind": ["LandbasedWind_Class4"],
    "CC with 90% CCS": ["NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS"],
    "Ultra-supercritical coal (USC)": [
        "Coal_newHighCF",
        "Coal_IGCCAvgCF",
        "Coal_IGCCHighCF",
        "Coal_IGCC_Moderate",
        "Coal_IGCC-90%-CCS",
    ],
}

print("=== Extend cost_multiplier_technology_map ===")
for category, new_techs in extensions.items():
    existing = cm_map.get(category, []) or []
    if not isinstance(existing, list):
        existing = [existing]
    for tech in new_techs:
        if tech not in existing:
            existing.append(tech)
            print(f"  Added {tech!r} under {category!r}")
        else:
            print(f"  {tech!r} already under {category!r}")
    cm_map[category] = existing

res_settings["cost_multiplier_technology_map"] = cm_map


=== Pass 4: startup_fuel_use validation ===
  ⚠ 2 startup_fuel_use keys missing from eia_atb_tech_map:
    NaturalGas_1-on-1: AUTO-ADDED → 'NaturalGas_1-on-1 Combined Cycle (H-Frame)' (exact alias)
    NaturalGas_Combustion: AUTO-ADDED → 'NaturalGas_Combustion Turbine (F-Frame)' (exact alias)

=== Extend eia_atb_tech_map ===
  'Coal_newAvgCF' already under 'Conventional Steam Coal'
  Added 'Coal_IGCCAvgCF' under 'Conventional Steam Coal'
  Added 'Coal_IGCC' under 'Conventional Steam Coal'
  Added 'Coal_IGCC-90%-CCS' under 'Conventional Steam Coal'
=== Extend cost_multiplier_technology_map ===
  Added 'LandbasedWind_Class4' under 'Wind'
  Added 'NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS' under 'CC with 90% CCS'
  'Coal_newHighCF' already under 'Ultra-supercritical coal (USC)'
  Added 'Coal_IGCCAvgCF' under 'Ultra-supercritical coal (USC)'
  Added 'Coal_IGCCHighCF' under 'Ultra-supercritical coal (USC)'
  Added 'Coal_IGCC_Moderate' under 'Ultra-supercritical coal (USC)'
  Added

In [23]:
# Pass 5: Set retirement_ages
# ----------------------------------------------------------------------
# Asset lifetimes by technology, used to retire existing units and to cap
# the economic life of new builds. Sources: NREL ATB 2022 (pdf p.42),
# with TD/MBA updates noted inline.
print("\n=== Pass 5: retirement_ages ===")

retirement_ages = {
    # Existing-fleet (EIA tech names)
    "Conventional Hydroelectric": 300,    # MBA update
    "Small Hydroelectric": 300,           # MBA update
    "Conventional Steam Coal": 65,        # TD update
    "Natural Gas Fired Combined Cycle": 60,
    "Natural Gas Fired Combustion Turbine": 50,
    "Petroleum Liquids": 40,
    "Natural Gas Internal Combustion Engine": 60,
    "Nuclear": 80,
    "Onshore Wind Turbine": 30,
    "Hydroelectric Pumped Storage": 300,  # MBA update
    "Natural Gas Steam Turbine": 50,
    "Solar Photovoltaic": 30,
    "Solar Thermal without Energy Storage": 30,  # CSP in SWITCH
    "Geothermal": 30,
    "Municipal Solid Waste": 40,
    "Landfill Gas": 40,
    "Batteries": 15,
    "Wood/Wood Waste Biomass": 50,
    "Petroleum Coke": 40,
    "All Other": 40,
    "Natural Gas with Compressed Air Storage": 60,
    "Coal Integrated Gasification Combined Cycle": 65,
    "Other Waste Biomass": 50,
    "Other Gases": 50,
    "Other Natural Gas": 50,
    "Solar Thermal with Energy Storage": 30,
    "Flywheels": 40,
    "Offshore Wind Turbine": 30,
    "Biomass": 50,

    # New-build (ATB names)
    "NaturalGas_CCCCSAvgCF_Conservative": 30,
    "NaturalGas_CCAvgCF_Moderate": 30,
    "NaturalGas_CTAvgCF_Moderate": 30,
    "Battery_*_Moderate": 15,
    "NaturalGas_CCS100_Moderate": 30,
    "UtilityPV_Class1_Moderate_100": 30,
    "UtilityPV_Class1_Moderate": 30,
    "UtilityPV_Class1_Moderate_50": 30,
    "LandbasedWind_Class1_Moderate": 30,
    "LandbasedWind_Class1_Moderate_100": 30,
    "LandbasedWind_Class3_Moderate": 30,
    "OffShoreWind_Class1_Moderate": 30,
    "OffShoreWind_Class3_Moderate": 30,
    "OffShoreWind_Class3_Moderate_fixed_1": 30,
    "OffShoreWind_Class3_Moderate_fixed_0": 30,
    "OffShoreWind_Class12_Moderate": 30,
    "OffShoreWind_Class12_Moderate_floating_1": 30,
    "OffShoreWind_Class12_Moderate_floating_0": 30,
    "Coal_CCS90AvgCF_Moderate": 40,
    "Coal_New_Moderate": 40,
    "Coal_95%-CCS_Moderate": 40,
    "Coal_99%-CCS_Moderate": 40,
    "Nuclear_Nuclear_Moderate": 40,
    "Hydrogen_CTAvgCF_Moderate": 30,
    "Hydrogen_CCAvgCF_Moderate": 30,
    "NaturalGas_H-Frame CC_Moderate": 30,
    "NaturalGas_H-Frame CC 95% CCS_Moderate": 30,
    "NaturalGas_H-Frame CC 97% CCS_Moderate": 30,
    "Hydrogen_H-Frame CC_Moderate": 30,
    "NaturalGas_F-Frame CT": 30,
    "NaturalGas_F-Frame CT_Moderate": 30,
    "Hydrogen_F-Frame CT": 30,
    "Hydrogen_F-Frame CT_Moderate": 30,
    "distributed_generation": 100,
    "Biopower_Dedicated": 40,
    "Biopower_Dedicated_Moderate": 40,
    "Coal_IGCC_Moderate": 65,                                              # matches Coal IGCC existing
    "Coal_IGCC-90%-CCS_Moderate": 40,                                      # matches other Coal CCS new-builds
    "Geothermal_HydroBinary_Moderate": 30,                                 # matches Geothermal
    "LandbasedWind_Class4_Moderate": 30,                                   # matches other LandbasedWind classes
    "NaturalGas_2-on-1 Combined Cycle (H-Frame)_Moderate": 30,             # matches NaturalGas_H-Frame CC_Moderate
    "NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS_Moderate": 30,     # matches NaturalGas_H-Frame CC 95% CCS_Moderate
    "NaturalGas_Combustion Turbine (F-Frame)_Moderate": 30,                # matches NaturalGas_F-Frame CT_Moderate
    "Nuclear_Nuclear - Large_Moderate": 40,                                # matches Nuclear_Nuclear_Moderate
    "Utility-Scale Battery Storage_Lithium Ion_Moderate": 15,              # matches Battery_*_Moderate
}

prev = res_settings.get("retirement_ages") or {}
res_settings["retirement_ages"] = retirement_ages
print(f"  set {len(retirement_ages)} entries (was {len(prev)})")
# Flag any new-build techs in atb_new_gen that lack a retirement age
atb_new = res_settings.get("atb_new_gen") or []
atb_names = {
    f"{row[0]}_{row[1]}_{row[2]}"
    for row in atb_new
    if isinstance(row, list) and len(row) >= 3
}
unmapped = sorted(n for n in atb_names if n not in retirement_ages)
if unmapped:
    print(f"  ⚠ {len(unmapped)} atb_new_gen techs without retirement_age:")
    for n in unmapped:
        print(f"    - {n}")
save_yml(res_settings, OUTPUT_SETTINGS_DIR / "resources.yml")
print(f"  re-saved resources.yml ({len(res_settings)} keys)")


=== Pass 5: retirement_ages ===
  set 74 entries (was 0)
  ✓ Wrote resources.yml (45 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/resources.yml
  re-saved resources.yml (45 keys)


In [24]:
# Pass 6: num_clusters overrides
# ----------------------------------------------------------------------
# Hardcode cluster counts per existing-tech category. Use:
#   int  → cluster into N groups
#   None → emits `~` in YAML, telling PowerGenome NOT to cluster (keep units separate)
print("\n=== Pass 6: num_clusters ===")

num_clusters_overrides = {
    # Existing entries (edit freely) None,  # don't cluster (→ ~)
    "Conventional Steam Coal": None,
    "Natural Gas Fired Combined Cycle": None,
    "Natural Gas Fired Combustion Turbine": None,

    "Nuclear": 1,
    "Conventional Hydroelectric": 1,
    "Solar Photovoltaic": 1,
    "Onshore Wind Turbine": 1,
    "Batteries": 1,

    ## Add missing techs here, e.g.:
    # "Natural Gas Steam Turbine": 1,
    "Natural Gas Internal Combustion Engine": None,
    "Petroleum Liquids": None,
    "Hydroelectric Pumped Storage": 1,
    "Small Hydroelectric": 1,
    "Geothermal": 1,
    "Biomass": None,
    # "Wood/Wood Waste Biomass": 1, # Aggregated into Biomass
    # "Landfill Gas": 1, # Aggregated into Biomass
    # "Municipal Solid Waste": 1, # Aggregated into Biomass
    # "Other Waste Biomass": 1, # Aggregated into Biomass
    "Offshore Wind Turbine": 1,
    # "Solar Thermal without Energy Storage": 1
}

prev = res_settings.get("num_clusters") or {}
res_settings["num_clusters"] = num_clusters_overrides

added   = sorted(set(num_clusters_overrides) - set(prev))
removed = sorted(set(prev) - set(num_clusters_overrides))
changed = sorted(k for k in num_clusters_overrides
                 if k in prev and prev[k] != num_clusters_overrides[k])

def _fmt(v): return "~" if v is None else v
print(f"  set {len(num_clusters_overrides)} entries (was {len(prev)})")
for k in added:   print(f"    + {k}: {_fmt(num_clusters_overrides[k])}")
for k in changed: print(f"    ~ {k}: {_fmt(prev[k])} → {_fmt(num_clusters_overrides[k])}")
for k in removed: print(f"    - {k} (was {_fmt(prev[k])})")

save_yml(res_settings, OUTPUT_SETTINGS_DIR / "resources.yml")
print(f"  re-saved resources.yml ({len(res_settings)} keys)")


=== Pass 6: num_clusters ===
  set 15 entries (was 8)
    + Biomass: ~
    + Geothermal: 1
    + Hydroelectric Pumped Storage: 1
    + Natural Gas Internal Combustion Engine: ~
    + Offshore Wind Turbine: 1
    + Petroleum Liquids: ~
    + Small Hydroelectric: 1
    ~ Conventional Steam Coal: 1 → ~
    ~ Natural Gas Fired Combined Cycle: 1 → ~
    ~ Natural Gas Fired Combustion Turbine: 1 → ~
  ✓ Wrote resources.yml (45 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/resources.yml
  re-saved resources.yml (45 keys)


In [65]:
# ============================================================
# Sanity check: any GUI keys we forgot to handle?
# ============================================================
untranslated_gui = set(gui_res) - set(gui_override_keys) - set(ref_res)
if untranslated_gui:
    print(f"\n  ⚠ GUI keys not handled by any pass: {sorted(untranslated_gui)}")
    for k in sorted(untranslated_gui):
        print(f"    {k}: {str(gui_res[k])[:80]}")

print(f"\n=== resources.yml: {len(res_settings)} keys total ===")
save_yml(res_settings, OUTPUT_SETTINGS_DIR / "resources.yml")


=== resources.yml: 45 keys total ===
  ✓ Wrote resources.yml (45 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days/resources.yml


---

## 5. `fuels.yml` — Fuel prices, emissions, and CCS

This file controls how PowerGenome constructs fuel price projections for each region and planning year, plus the emission factors and CCS-specific fuel definitions that downstream calculations need. Prices come from EIA's Annual Energy Outlook (AEO), pre-downloaded into `pg_data/`.

Two keys here are read directly by the Switch conversion code: `aeo_fuel_region_map` (used by `conversion_functions.py` to build Switch's `fuel_cost.csv`) and `fuel_emission_factors` (used to set `co2_intensity` in Switch's `fuels.csv`). The other keys are read by PowerGenome itself, which produces the intermediate `fuel_prices` DataFrame that the glue code then consumes — so they affect Switch outputs indirectly.

### How fuel prices work end-to-end

1. `fuel_eia_aeo_year` picks which AEO edition (e.g., `2025`)
2. `aeo_fuel_scenarios` picks a scenario per fuel (e.g., `reference` for gas, `no_111d` for coal)
3. `aeo_fuel_region_map` maps each AEO census division to the model regions inside it
4. `eia_series_*_names` settings translate between user-friendly names and EIA Open Data API codes

PowerGenome assembles a price trajectory for every fuel × region × year combination, and the Switch conversion code reshapes it into Switch's `fuel_cost.csv` format.

### Translation: GUI v0.8 → PG v0.7

| GUI key (v0.8) | Must use (v0.7) | What it does |
|---|---|---|
| `fuel_scenarios` | `aeo_fuel_scenarios` | Per-fuel scenario selection |
| `fuel_data_year` | `fuel_eia_aeo_year` | Which AEO edition to pull from |

The notebook's `translate_gui_dict` helper handles both translations automatically.

### Parameters

| Key | What it does |
|---|---|
| `fuel_eia_aeo_year` | AEO edition year (e.g., `2025`). Note that scenario names differ between AEO editions, so changing this may require updating `eia_series_scenario_names` too. |
| `aeo_fuel_usd_year` | Dollar year of the AEO fuel price data. Used by PG to convert prices to `target_usd_year` from `model_definition.yml`. |
| `eia_series_scenario_names` | Maps friendly names to EIA AEO scenario codes (`reference: REF2025`, `no_111d: nocaa111`, `low_price: LOWPRICE`, etc.). This is the lookup table for `aeo_fuel_scenarios`. |
| `eia_series_fuel_names` | Maps friendly fuel names to EIA fuel codes (`distillate: DFO`, `uranium: U`, `naturalgas: NG`, `coal: STC`). Pass 1 of the build cell ensures all four are present. |
| `aeo_fuel_scenarios` | Per-fuel scenario selection, e.g. `{naturalgas: reference, coal: no_111d}`. Some fuels are commonly omitted here and added via `scenario_management.yml` instead. |
| `aeo_fuel_region_map` | Dict of `{aeo_fuel_region: [list of regions]}`. Tells PG which AEO census division supplies prices for each region. The reference uses raw p-regions; Pass 2 of the build cell remaps these to your aggregated model regions. **(also read by Switch glue: passed to `switch_fuel_cost_table()` in `conversion_functions.py`)** |
| `eia_series_region_names` | Maps user fuel region names to EIA region codes (e.g., `mountain: MTN`). Used when querying the EIA Open Data API for regional prices. |
| `tech_fuel_map` | Maps technology names to fuel names. PG uses this to figure out which fuel each generator burns. The notebook merges GUI and reference versions in Pass 1. |
| `ccs_fuel_map` | Maps CCS technology names (e.g., `Coal_CCS90`) to derived CCS fuel names (`coal_ccs90`). The derived fuel name format is `<base_fuel>_<ccs_level>`. |
| `ccs_capture_rate` | Capture rate per CCS fuel, e.g. `naturalgas_ccs100: 1.0`, `coal_ccs90: 0.9`. Used to compute net emissions. |
| `ccs_disposal_cost` | Pipeline + sequestration cost per tonne of captured CO2, in dollars. Reference uses `10`. |
| `carbon_tax` | Uniform carbon tax across all regions and fuels, in $/tonne. Reference uses `0` because carbon pricing is handled via `carbon_cost_dollar_per_tco2` in `switch.yml` instead. |
| `fuel_emission_factors` | Dict of `{fuel: tonnes_CO2_per_MMBtu}`. Reference values: `naturalgas: 0.05306`, `coal: 0.09552`, `distillate: 0.07315`. **(also read by Switch glue: written as `co2_intensity` in Switch's `fuels.csv`)** |

### Optional keys not in the reference

| Key | What it does |
|---|---|
| `regional_fuel_adjustments` | Per-region multiplicative adjustments to fuel prices, e.g. `{p1: {naturalgas: [mul, 1.267]}}`. Used to apply region-specific price corrections (the reference file has a large commented-out block showing how this can be used to apply NYMEX-derived gas prices). When set, PG uses this to override AEO regional prices. The current setup uses `scenario_management.yml`'s `fuel_price_forecast` mechanism instead. |

### What the notebook does

The build cell below performs three passes:

1. **Overrides, merges, and required entries** — Replace simple keys with GUI values, merge `tech_fuel_map` (GUI has entries the reference lacks), and ensure `eia_series_fuel_names` contains the four entries PG requires (`distillate`, `uranium`, `naturalgas`, `coal`).
2. **Remap `aeo_fuel_region_map`** — Walk each fuel region's list of p-regions and replace them with your aggregated model region names. Unlike `cost_multiplier_region_map` in `resources.yml`, no tie-breaking is needed here because a model region can legitimately appear in multiple fuel regions.
3. **Add `waste_biomass` entries** — Hardcoded additions extending the fuel set with a zero-emission biomass option. Review before running on a new model (see the Pass 3 markdown).

See [PG docs: fuels](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/fuels/).

In [25]:
ref_fuel = load_yml(REF_SETTINGS_DIR / "fuels.yml")
gui_fuel_raw = load_yml(GUI_SETTINGS_DIR / "fuels.yml")

print("Translating GUI keys...")
gui_fuel = translate_gui_dict(gui_fuel_raw)

only_ref = sorted(set(ref_fuel) - set(gui_fuel))
only_gui = sorted(set(gui_fuel) - set(ref_fuel))
both     = sorted(set(ref_fuel) & set(gui_fuel))

print(f"\nOnly in ref ({len(only_ref)}): {only_ref}")
print(f"Only in GUI ({len(only_gui)}): {only_gui}")
print(f"In both ({len(both)}): {both}")

fuel_settings = copy.deepcopy(ref_fuel)

Translating GUI keys...
  Renamed:
    fuel_data_year → fuel_eia_aeo_year
    fuel_scenarios → aeo_fuel_scenarios

Only in ref (9): ['aeo_fuel_region_map', 'aeo_fuel_usd_year', 'carbon_tax', 'ccs_capture_rate', 'ccs_disposal_cost', 'ccs_fuel_map', 'eia_series_fuel_names', 'eia_series_region_names', 'eia_series_scenario_names']
Only in GUI (0): []
In both (4): ['aeo_fuel_scenarios', 'fuel_eia_aeo_year', 'fuel_emission_factors', 'tech_fuel_map']


### Building `fuels.yml` in three passes

Like `resources.yml`, the fuels file needs several distinct transformations. We do them in three passes:

1. **Overrides, merges, and required entries.** Replace simple keys with GUI values, merge `tech_fuel_map` (GUI has entries the reference lacks), and ensure `eia_series_fuel_names` contains the four entries PowerGenome requires (distillate, uranium, naturalgas, coal).
2. **Remap `aeo_fuel_region_map`** from p-regions to your aggregated model regions. This file tells PowerGenome which AEO census division to pull fuel prices from for each of your regions, so the keys need to match your GUI region names.
3. **Add `waste_biomass` entries.** Four hardcoded values that extend the fuel set with a zero-emission biomass option. See the notes above that cell before running it on a new model, since these values may be specific to past runs.

Each pass is its own cell below.

In [26]:
# Pass 1: Overrides, merges, and required entries
# ----------------------------------------------------------------------
print("=== Pass 1: overrides and merges ===")

# Simple overrides: GUI value wins outright
SIMPLE_OVERRIDES = [
    "aeo_fuel_scenarios",    # translated from fuel_scenarios
    "fuel_eia_aeo_year",     # translated from fuel_data_year
    "fuel_emission_factors", # same name in both
    "eia_series_fuel_names", # patched below to guarantee required entries
]
for key in SIMPLE_OVERRIDES:
    if key in gui_fuel:
        fuel_settings[key] = gui_fuel[key]
        print(f"  overrode {key}")

# tech_fuel_map: merge instead of override. The GUI has entries the
# reference lacks (e.g., NaturalGas, Natural Gas Internal Combustion Engine),
# and the reference has entries the GUI lacks. We want the union.
if "tech_fuel_map" in gui_fuel and "tech_fuel_map" in ref_fuel:
    merged_tfm = {**ref_fuel["tech_fuel_map"], **gui_fuel["tech_fuel_map"]}
    fuel_settings["tech_fuel_map"] = merged_tfm
    print(
        f"  merged tech_fuel_map: "
        f"{len(ref_fuel['tech_fuel_map'])} ref + "
        f"{len(gui_fuel['tech_fuel_map'])} gui → {len(merged_tfm)} total"
    )

# Extend tech_fuel_map with techs that the merged map doesn't cover.
# Coal IGCC (and IGCC-CCS handled below) burns coal; Biopower burns biomass.
# Without these, PowerGenome can't resolve a fuel for these ATB techs.
TECH_FUEL_EXTENSIONS = {
    "Coal IGCC": "coal",
    "Biopower":  "waste_biomass",
}

tfm = fuel_settings.get("tech_fuel_map", {}) or {}
for tech, fuel in TECH_FUEL_EXTENSIONS.items():
    if tech in tfm:
        print(f"  tech_fuel_map[{tech!r}] already present ({tfm[tech]!r})")
    else:
        tfm[tech] = fuel
        print(f"  added tech_fuel_map[{tech!r}] = {fuel!r}")
fuel_settings["tech_fuel_map"] = tfm

# Extend ccs_fuel_map with CCS variants the reference lacks.
# Coal_IGCC-90%-CCS uses the same CCS-coal fuel bucket as the other 90% CCS coal techs.
CCS_FUEL_EXTENSIONS = {
    # "Coal_IGCC-90%-CCS": "coal_ccs90",
}

ccs_map = fuel_settings.get("ccs_fuel_map", {}) or {}
for tech, fuel in CCS_FUEL_EXTENSIONS.items():
    if tech in ccs_map:
        print(f"  ccs_fuel_map[{tech!r}] already present ({ccs_map[tech]!r})")
    else:
        ccs_map[tech] = fuel
        print(f"  added ccs_fuel_map[{tech!r}] = {fuel!r}")
fuel_settings["ccs_fuel_map"] = ccs_map

# Ensure required eia_series_fuel_names entries exist. These map
# user-friendly fuel names to EIA Open Data API series codes.
# PowerGenome expects all four; if any are missing, fuel price lookups fail.
REQUIRED_EIA_SERIES = {
    "distillate": "DFO",
    "uranium":    "U",
    "naturalgas": "NG",
    "coal":       "STC",
}

series = fuel_settings.get("eia_series_fuel_names") or {}
if not isinstance(series, dict):
    print("  ⚠ eia_series_fuel_names was not a dict; replacing with required entries")
    series = {}

for fuel_name, code in REQUIRED_EIA_SERIES.items():
    if fuel_name in series:
        print(f"  eia_series_fuel_names[{fuel_name!r}] already present ({series[fuel_name]!r})")
    else:
        series[fuel_name] = code
        print(f"  added eia_series_fuel_names[{fuel_name!r}] = {code!r}")

fuel_settings["eia_series_fuel_names"] = series

=== Pass 1: overrides and merges ===
  overrode aeo_fuel_scenarios
  overrode fuel_eia_aeo_year
  overrode fuel_emission_factors
  merged tech_fuel_map: 7 ref + 9 gui → 9 total
  added tech_fuel_map['Coal IGCC'] = 'coal'
  added tech_fuel_map['Biopower'] = 'waste_biomass'
  eia_series_fuel_names['distillate'] already present ('DFO')
  eia_series_fuel_names['uranium'] already present ('U')
  added eia_series_fuel_names['naturalgas'] = 'NG'
  added eia_series_fuel_names['coal'] = 'STC'


### Pass 2: Remapping `aeo_fuel_region_map`

`aeo_fuel_region_map` tells PowerGenome which AEO census division supplies fuel prices for each model region. The reference file is inverted compared to most region maps: the keys are fuel regions (e.g., census divisions like `mountain`, `pacific`), and the values are lists of p-regions that belong to each one:

```yaml
aeo_fuel_region_map:
  mountain: [p29, p30, p31, p32]
  pacific:  [p9, p10, p11]
  ...
```

When you aggregate p-regions, we need to replace each p-region in these lists with the model region it belongs to, then dedupe. Unlike `cost_multiplier_region_map`, there's no tie-breaking problem here because a model region can legitimately appear in multiple fuel regions (fuel prices don't need to be exclusive). We just flag any p-regions that don't appear in any aggregation so you know something is off.

This pass builds its own `p_to_model_region`, separate from the one in Pass 2 of `resources.yml`. Consolidating the three copies across the notebook is a reasonable future refactor but not done here.

In [27]:
# Pass 2: Remap aeo_fuel_region_map from p-regions to model regions
# ----------------------------------------------------------------------
print("\n=== Pass 2: aeo_fuel_region_map ===")

region_aggregations = gui_md.get("region_aggregations", {}) or {}

# Build reverse map: p_region -> model region.
# (Duplicate detection: warn if any p-region appears under multiple
# aggregations, which shouldn't happen in a well-formed GUI export.)
p_to_model_region = {}
duplicate_assignments = {}

for model_region, p_regions in region_aggregations.items():
    if p_regions is None:
        continue
    if not isinstance(p_regions, list):
        p_regions = [p_regions]
    for p in p_regions:
        existing = p_to_model_region.get(p)
        if existing is not None and existing != model_region:
            duplicate_assignments.setdefault(p, {existing}).add(model_region)
        else:
            p_to_model_region[p] = model_region

if duplicate_assignments:
    print("  ⚠ p-regions assigned to multiple aggregations:")
    for p, regions in sorted(duplicate_assignments.items()):
        print(f"    {p}: {sorted(regions)}")

print(f"  built p_to_model_region: {len(p_to_model_region)} p-regions mapped")

# Walk every fuel_region -> p_list entry and rewrite the p_list in place.
aeo_map = fuel_settings.get("aeo_fuel_region_map") or {}

if not isinstance(aeo_map, dict):
    print("  ⚠ aeo_fuel_region_map is not a dict; skipping remap")
else:
    # Pass 2a: rewrite p-regions to model regions, counting how many p-regions
    # from each AEO region map to each model region. Counts are used in Pass 2b
    # to resolve model regions that end up under multiple AEO regions (which
    # breaks PowerGenome's fuel-labeling — see add_fuel_labels + split_fuel).
    # ------------------------------------------------------------------
    # model_region -> {aeo_region: count_of_p_regions_contributing}
    model_to_aeo_counts = {}
    unresolved = {}
    original_list_lens = {}  # for "changed" reporting

    for fuel_region, region_list in aeo_map.items():
        if region_list is None:
            original_list_lens[fuel_region] = 0
            continue
        if not isinstance(region_list, list):
            region_list = [region_list]
        original_list_lens[fuel_region] = len(region_list)

        missing = []
        for region in region_list:
            if isinstance(region, str) and region in p_to_model_region:
                mr = p_to_model_region[region]
                model_to_aeo_counts.setdefault(mr, {})
                model_to_aeo_counts[mr][fuel_region] = (
                    model_to_aeo_counts[mr].get(fuel_region, 0) + 1
                )
            else:
                # Check if this looks like a p-region code (lowercase p + digits).
                is_pregion_code = (
                    isinstance(region, str)
                    and len(region) > 1
                    and region[0] == "p"
                    and region[1:].isdigit()
                )
                if is_pregion_code:
                    # Unknown p-region — not in region_aggregations. Drop it
                    # from the final map (it's outside model scope) and record
                    # it for reporting.
                    missing.append(region)
                elif isinstance(region, str):
                    # Not a p-region code. Treat as a direct model-region or
                    # custom-label assignment (this is what lets capital-P
                    # names like "PA" pass through correctly).
                    model_to_aeo_counts.setdefault(region, {})
                    model_to_aeo_counts[region][fuel_region] = (
                        model_to_aeo_counts[region].get(fuel_region, 0) + 1
                    )

        if missing:
            unresolved[fuel_region] = sorted(set(missing))
    # Pass 2b: resolve cross-AEO duplicates. For each model region appearing
    # under multiple AEO regions, assign it to the AEO region holding the most
    # of its underlying p-regions. Alphabetical tiebreak for determinism.
    # ------------------------------------------------------------------
    resolved_assignments = {}    # model_region -> winning aeo_region
    conflicts_resolved = []      # for logging

    for mr, aeo_counts in model_to_aeo_counts.items():
        if len(aeo_counts) == 1:
            resolved_assignments[mr] = next(iter(aeo_counts))
        else:
            # sort by (-count, aeo_name) so highest count wins, alpha breaks ties
            ranked = sorted(aeo_counts.items(), key=lambda kv: (-kv[1], kv[0]))
            winner = ranked[0][0]
            resolved_assignments[mr] = winner
            conflicts_resolved.append((mr, dict(aeo_counts), winner))

    # Pass 2c: build final remapped dict with each model region appearing
    # exactly once. Preserve original AEO-region ordering.
    # ------------------------------------------------------------------
    remapped = {}
    changed = []
    for fuel_region, orig_list in aeo_map.items():
        if orig_list is None:
            remapped[fuel_region] = None
            continue

        members = sorted(
            mr for mr, winning_aeo in resolved_assignments.items()
            if winning_aeo == fuel_region
        )
        remapped[fuel_region] = members

        old_n = original_list_lens.get(fuel_region, 0)
        new_n = len(members)
        if new_n != old_n:
            changed.append((fuel_region, old_n, new_n))

    fuel_settings["aeo_fuel_region_map"] = remapped

# Reporting
    # ------------------------------------------------------------------
    # Categorize unresolved p-regions:
    #   - "dropped": silently removed from the final map because they don't
    #     map to any model region. Harmless if they're outside the model scope
    #     (e.g., Canadian/Mexican p-regions in a CONUS-only run).
    #   - "retained as literals": still present in the final map as raw strings
    #     (shouldn't happen with current logic, but check defensively).
    dropped_pregs = {}
    retained_pregs = {}
    for fr, missing in unresolved.items():
        final_members = set(remapped.get(fr) or [])
        dropped = sorted(p for p in missing if p not in final_members)
        retained = sorted(p for p in missing if p in final_members)
        if dropped:
            dropped_pregs[fr] = dropped
        if retained:
            retained_pregs[fr] = retained

    # Summary line: what actually happened to the map
    total_model_regions = sum(
        len(v) for v in remapped.values() if isinstance(v, list)
    )
    print(
        f"  final map: {len(remapped)} AEO regions, "
        f"{total_model_regions} model-region assignments"
    )

    if changed:
        print(f"  updated {len(changed)} AEO region entries "
              f"(p-regions → model regions):")
        for fr, old_n, new_n in changed[:10]:
            print(f"    {fr}: {old_n} p-regions → {new_n} model regions")
        if len(changed) > 10:
            print(f"    ... {len(changed) - 10} more")

    if conflicts_resolved:
        print(f"  resolved {len(conflicts_resolved)} cross-AEO duplicates "
              f"(model regions spanning multiple AEO regions):")
        for mr, counts, winner in conflicts_resolved:
            losers = sorted(a for a in counts if a != winner)
            print(f"    {mr}: {counts} → assigned to {winner} "
                  f"(removed from {losers})")

    if dropped_pregs:
        total_dropped = sum(len(v) for v in dropped_pregs.values())
        print(f"  dropped {total_dropped} p-regions not covered by "
              f"region_aggregations (likely outside model scope):")
        for fr, pregs in dropped_pregs.items():
            preview = pregs if len(pregs) <= 6 else pregs[:5] + [f"... +{len(pregs)-5} more"]
            print(f"    {fr}: {preview}")

    if retained_pregs:
        print("  ⚠ p-regions retained as literals in final map "
              "(no model-region aggregation found — THIS IS A BUG):")
        for fr, pregs in retained_pregs.items():
            print(f"    {fr}: {pregs}")
        print("\n  Helper dict for copy-paste (add these to region_aggregations):")
        print("  missing_aeo_fuel_region_map_regions = {")
        for fr, pregs in retained_pregs.items():
            print(f'      "{fr}": {pregs},')
        print("  }")

    if not changed and not conflicts_resolved and not dropped_pregs and not retained_pregs:
        print("  no changes needed, no conflicts, no unresolved regions")


=== Pass 2: aeo_fuel_region_map ===
  built p_to_model_region: 134 p-regions mapped
  final map: 9 AEO regions, 46 model-region assignments
  updated 9 AEO region entries (p-regions → model regions):
    east_north_central: 20 p-regions → 6 model regions
    east_south_central: 9 p-regions → 4 model regions
    middle_atlantic: 11 p-regions → 4 model regions
    mountain: 26 p-regions → 6 model regions
    new_england: 15 p-regions → 4 model regions
    pacific: 16 p-regions → 4 model regions
    south_atlantic: 17 p-regions → 8 model regions
    west_north_central: 24 p-regions → 5 model regions
    west_south_central: 18 p-regions → 5 model regions
  resolved 5 cross-AEO duplicates (model regions spanning multiple AEO regions):
    IN_and_W_KY: {'east_north_central': 3, 'east_south_central': 1} → assigned to east_north_central (removed from ['east_south_central'])
    e_AR_and_w_MS: {'east_south_central': 1, 'west_south_central': 1} → assigned to east_south_central (removed from ['w

### Pass 3: Adding `waste_biomass` entries

This pass adds four hardcoded entries to `fuels.yml` that extend the fuel set with a zero-emission `waste_biomass` fuel. These were added during earlier runs to get biomass technologies working end-to-end. Before re-running this notebook for a new model, confirm that:

1. Your resource set still includes a `Biomass` technology
2. The placeholder price (`user_fuel_price['waste_biomass'] = 1`) is acceptable, or replace it with a real value
3. The `user_fuel_usd_year` key correctly uses `biomass` (lowercase) to match how PowerGenome references it elsewhere

If any of these are no longer true, edit the cell below or skip it entirely.

In [28]:
# Pass 3: Add waste_biomass fuel entries
# ----------------------------------------------------------------------
# Hardcoded additions. See the markdown above for when to review these.
print("\n=== Pass 3: waste_biomass entries ===")

fuel_settings.setdefault("tech_fuel_map", {})["Biomass"] = "waste_biomass"
fuel_settings.setdefault("fuel_emission_factors", {})["waste_biomass"] = 0
fuel_settings.setdefault("user_fuel_price", {})["waste_biomass"] = 1
fuel_settings.setdefault("user_fuel_usd_year", {})["biomass"] = 2020

print("  tech_fuel_map['Biomass'] = 'waste_biomass'")
print("  fuel_emission_factors['waste_biomass'] = 0")
print("  user_fuel_price['waste_biomass'] = 1")
print("  user_fuel_usd_year['biomass'] = 2020")

save_yml(fuel_settings, OUTPUT_SETTINGS_DIR / "fuels.yml")


=== Pass 3: waste_biomass entries ===
  tech_fuel_map['Biomass'] = 'waste_biomass'
  fuel_emission_factors['waste_biomass'] = 0
  user_fuel_price['waste_biomass'] = 1
  user_fuel_usd_year['biomass'] = 2020
  ✓ Wrote fuels.yml (15 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/fuels.yml


---

### Region name remapping: building `p_to_gui` for the transmission cell

The reference settings use 134 individual p-regions (`p1, p2, ..., p134`). The GUI clusters these into your named model regions via `region_aggregations`. Most files that need remapping handle it inline in their own cell above (resources Passes 2 and 3, fuels Pass 2, model_definition's `regional_capacity_reserves` block). The one exception is `transmission.yml`, which reuses a shared `p_to_gui` dictionary built in the cell below. That cell also sanity-checks that every reference p-region is covered by the GUI aggregations.

In [30]:
# ============================================================
# Build p-region → GUI-region mapping from region_aggregations
# ============================================================

gui_md = load_yml(GUI_SETTINGS_DIR / "model_definition.yml")
region_aggs = gui_md.get("region_aggregations", {})

if not region_aggs:
    print("No region_aggregations found. Reference uses p-regions directly, no remapping needed.")
else:
    # Build reverse map: p_region -> gui_region
    p_to_gui = {}
    for gui_name, p_list in region_aggs.items():
        if p_list:  # some might be None if single-BA region
            for p in p_list:
                p_to_gui[p] = gui_name
        else:
            # GUI region with no sub-regions listed = same as the p-region?
            # This shouldn't happen with proper aggregations
            print(f"  ⚠ {gui_name} has no p-regions listed")
    
    print(f"Built mapping: {len(p_to_gui)} p-regions → {len(region_aggs)} GUI regions")
    
    # Quick sanity: how many p-regions from reference are covered?
    ref_md = load_yml(REF_SETTINGS_DIR / "model_definition.yml")
    ref_regions = set(ref_md.get("model_regions", []))
    mapped_regions = set(p_to_gui.keys())
    unmapped = ref_regions - mapped_regions
    if unmapped:
        print(f"  ⚠ {len(unmapped)} ref p-regions not in GUI aggregations: {sorted(unmapped)[:10]}...")
    else:
        print(f"  ✓ All {len(ref_regions)} reference p-regions are mapped")


Built mapping: 134 p-regions → 46 GUI regions
  ✓ All 134 reference p-regions are mapped


## 6. `transmission.yml` — Inter-regional and spur transmission

This file controls inter-regional transmission: how much capacity exists between regions today, how much can be built, what spur lines (generator-to-grid connections) cost, and how losses are computed.

The reference file is small (4 top-level keys) but the spur cost structure is nested two levels deep and keyed by p-region, so the build cell does meaningful work to remap it. One key (`transmission_investment_cost`) is read by both `pg_to_switch.py` and `conversion_functions.py`, though they consume different sub-paths of it.

#### Two things need special handling

1. **Spur capex is keyed by p-region in the reference.** The reference stores `transmission_investment_cost.spur.capex_mw_mile` as a `{p_region: $/MW-mile}` dict. We aggregate to model regions by averaging the per-MW-mile costs across all p-regions in each model region. Averaging is appropriate because the costs are already normalized per MW per mile, so summing would be wrong.

2. **Reference expansion limits are restrictive.** The reference values for `tx_expansion_per_period` and `tx_expansion_mw_per_period` constrain how much new transmission the model can build. We override them to large values that effectively allow unlimited buildout, since our research questions focus on what an unconstrained system would prefer. Override these back if your study needs realistic build limits.

#### Parameters

| Key | What it does |
|---|---|
| `transmission_investment_cost` | Nested dict with two relevant sub-paths: `spur.capex_mw_mile` (per-MW-mile cost of building spur lines from new generators to the grid) and `tx.capex_mw_mile` (per-MW-mile cost of building inter-regional transmission). The reference only sets `spur`. **(also read by Switch glue: `conversion_functions.py` reads `spur.capex_mw_mile` when computing per-generator interconnect costs; `pg_to_switch.py` reads `tx.capex_mw_mile` only when `user_transmission_costs` is unset — see below)** |
| `transmission_investment_cost.spur.wacc` | Weighted-average cost of capital for spur line investments. Reference uses `0.044` (4.4% real). |
| `transmission_investment_cost.spur.investment_years` | Capital recovery period in years. Reference uses `60`. The implied capital recovery factor (~0.0476) is what `extra_inputs.yml`'s transmission CSV patcher uses to compute annuity columns. |
| `transmission_investment_cost.use_total` | If `true`, uses pre-computed `interconnect_annuity` values from PG's network data when available, falling back to the spur capex calculation otherwise. |
| `tx_expansion_per_period` | Maximum allowed transmission expansion as a fraction of existing capacity per planning period. Reference is `0` (no expansion); the build cell overrides to `100.0` (effectively unlimited). |
| `tx_expansion_mw_per_period` | Absolute MW cap on new transmission per planning period. Reference is `0`; build cell overrides to `100000`. The minimum of the two limits applies. |
| `tx_value_col` | Which column to read for existing transmission capacity: `firm_ttc_mw` (firm contracts only, more conservative) or `nonfirm_ttc_mw` (includes non-firm). Reference uses `firm_ttc_mw` for conservative deployability. |

#### A note on `transmission_investment_cost.tx`

The reference file does **not** define a `tx` sub-block under `transmission_investment_cost` — only `spur`. `pg_to_switch.py` reads `transmission_investment_cost["tx"]["capex_mw_mile"]` only inside an `if not settings.get("user_transmission_costs")` branch. Since this notebook always sets `user_transmission_costs` via `extra_inputs.yml` (pointing at `network_costs_ReEDS_patched_for_REFS2009.csv`), that branch is never taken and the missing `tx` block isn't a problem.

If you ever switch to letting `pg_to_switch.py` compute transmission costs from PG's own network database instead of from a CSV, you'll need to add a `tx` sub-block here with `capex_mw_mile` (either a scalar or a `{region: cost}` dict), `wacc`, and `investment_years`.

#### What the notebook does

The build cell below performs three passes:

1. **Generic merge of GUI into reference** — Same pattern as `startup_costs.yml`, `distributed_gen.yml`, and `resource_tags.yml`.
2. **Aggregate spur capex from p-regions to model regions** — Walk `transmission_investment_cost.spur.capex_mw_mile`, group p-regions by their parent model region, and average. Uses `p_to_gui` from the cell above. The print output shows the resulting per-region range so you can spot anything that looks off.
3. **Override transmission expansion limits** — Hardcoded change to `tx_expansion_per_period` and `tx_expansion_mw_per_period`. Edit if your study needs realistic build limits.

See [PG docs: transmission](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/transmission/).

In [31]:
# Pass 1: Generic merge of GUI values into the reference
# ----------------------------------------------------------------------
# Same merge pattern used by startup_costs, distributed_gen, and
# resource_tags below: walk the GUI dict, override matching keys,
# add new ones, track what changed.
ref_data = load_yml(REF_SETTINGS_DIR / "transmission.yml")
gui_data = load_yml(GUI_SETTINGS_DIR / "transmission.yml")

trans_settings = copy.deepcopy(ref_data)
overridden, added = [], []
for k, v in gui_data.items():
    if k in trans_settings:
        if trans_settings[k] != v:
            trans_settings[k] = v
            overridden.append(k)
    else:
        trans_settings[k] = v
        added.append(k)

print("=== Pass 1: merge ===")
print(f"  Ref: {len(ref_data)} keys, GUI: {len(gui_data)} keys → Merged: {len(trans_settings)} keys")
if overridden:
    print(f"  GUI overrode: {overridden}")
if added:
    print(f"  GUI added: {added}")

=== Pass 1: merge ===
  Ref: 4 keys, GUI: 2 keys → Merged: 4 keys
  GUI overrode: ['tx_expansion_per_period', 'tx_expansion_mw_per_period']


### Pass 2: Aggregating spur capex from p-regions to model regions

`transmission_investment_cost.spur.capex_mw_mile` is the per-MW per-mile cost of building a spur line from a new generator to the grid. The reference stores this as a dict keyed by p-region, with one cost per p-region. When we aggregate p-regions into model regions, we need a single cost per model region.

We average the per-MW-mile costs of all p-regions inside each model region. Averaging (rather than summing or taking the max) is right because the values are already normalized per unit of capacity and per unit of distance, so they're comparable across p-regions of any size. The print output below shows the resulting range and per-region breakdown so you can spot anything that looks off.

This pass is the only one in `transmission.yml` that uses `p_to_gui` from the cell above.

In [32]:
# Pass 2: Aggregate spur capex_mw_mile from p-regions to model regions
# ----------------------------------------------------------------------
from collections import defaultdict
from statistics import mean

print("\n=== Pass 2: spur capex aggregation ===")

spur_p = trans_settings["transmission_investment_cost"]["spur"]["capex_mw_mile"]

grouped = defaultdict(list)
unmapped = []
for p, val in spur_p.items():
    model = p_to_gui.get(p)
    if model is None:
        unmapped.append(p)
        continue
    grouped[model].append(val)

spur_agg = {m: round(mean(vals)) for m, vals in grouped.items()}
trans_settings["transmission_investment_cost"]["spur"]["capex_mw_mile"] = spur_agg

print(f"  Spur capex: {len(spur_p)} p-regions → {len(spur_agg)} model regions")
print(f"    Range: ${min(spur_agg.values())}–${max(spur_agg.values())}/MW-mile")
for model, val in sorted(spur_agg.items()):
    n = len(grouped[model])
    print(f"    {model}: ${val}/MW-mile (from {n} p-regions)")

if unmapped:
    print(f"  ⚠ {len(unmapped)} p-regions not in p_to_gui: {unmapped}")


=== Pass 2: spur capex aggregation ===
  Spur capex: 134 p-regions → 46 model regions
    Range: $1668–$2998/MW-mile
    AL_and_e_MS_and_FL_pnh: $1943/MW-mile (from 4 p-regions)
    AZ: $1753/MW-mile (from 4 p-regions)
    Bay_area: $2153/MW-mile (from 1 p-regions)
    CO: $1705/MW-mile (from 2 p-regions)
    CT_and_RT: $2823/MW-mile (from 2 p-regions)
    Central_FL: $2127/MW-mile (from 1 p-regions)
    Chicago_and_sw_MI: $1809/MW-mile (from 2 p-regions)
    DE_and_NJ: $2786/MW-mile (from 2 p-regions)
    GA: $2158/MW-mile (from 1 p-regions)
    Houston_ERCOT: $1849/MW-mile (from 1 p-regions)
    IA: $1710/MW-mile (from 3 p-regions)
    ID_and_w_MT_and_w_WY: $1723/MW-mile (from 9 p-regions)
    IL_no_Chicago: $1769/MW-mile (from 3 p-regions)
    IN_and_W_KY: $1878/MW-mile (from 4 p-regions)
    KS_and_w_MO: $1700/MW-mile (from 4 p-regions)
    LAX: $2033/MW-mile (from 1 p-regions)
    LA_and_e_TX: $1754/MW-mile (from 3 p-regions)
    MA: $2880/MW-mile (from 1 p-regions)
    MD_and_VA

### Pass 3: Override transmission expansion limits

The reference sets `tx_expansion_per_period` and `tx_expansion_mw_per_period` to values that constrain how much new transmission the model can build in each planning period. For our research, we want to see what the system would prefer with no transmission constraints, so we override both to large values.

**If your study needs realistic build limits, edit this cell or skip it entirely.** The values below are model-design choices, not bug fixes. If running a different study, you should not assume these are correct for them.

In [33]:
# Pass 3: Override transmission expansion limits
# ----------------------------------------------------------------------
# These are deliberate model-design choices, not bug fixes. Override
# them in your own copy of the notebook if your study needs realistic
# transmission build constraints.
print("\n=== Pass 3: expansion limit overrides ===")

trans_settings["tx_expansion_per_period"]    = 100.0
trans_settings["tx_expansion_mw_per_period"] = 100000

print("  tx_expansion_per_period    = 100.0   (unlimited fractional buildout per period)")
print("  tx_expansion_mw_per_period = 100000  (effectively unconstrained)")

save_yml(trans_settings, OUTPUT_SETTINGS_DIR / "transmission.yml")


=== Pass 3: expansion limit overrides ===
  tx_expansion_per_period    = 100.0   (unlimited fractional buildout per period)
  tx_expansion_mw_per_period = 100000  (effectively unconstrained)
  ✓ Wrote transmission.yml (4 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/transmission.yml


## 7. `startup_costs.yml` — Generator startup fuel and VOM costs


Controls how much fuel and variable O&M each generator burns on a cold start. Used by Switch's unit-commitment logic to decide whether cycling a plant is worth it versus keeping it online at minimum load.

All default values are from the NREL Western wind/solar integration study (Lew et al. 2013 *Finding Flexibility* and NREL 2012 cycling costs). Only change them if you have better data for your specific generator fleet.

#### The four pieces

- **`startup_fuel_use`** — mmbtu/MW burned per cold start, keyed by EIA technology name. Pass 4 of `resources.yml` validates that every key here exists in `eia_atb_tech_map`; if it doesn't, the model will crash at runtime.
- **`startup_vom_costs_mw`** — VOM cost per MW of startup capacity, keyed by a short tech label (`coal_large_sub`, `gas_cc`, etc.). Paired with `startup_vom_costs_usd_year` for inflation.
- **`startup_costs_per_cold_start_mw`** — One-time fixed cost per cold start, per MW, keyed by the same short tech labels. Paired with `startup_costs_per_cold_start_usd_year`.
- **`existing_startup_costs_tech_map`** and **`new_build_startup_costs`** — Translation layers mapping EIA technology names (and new-build ATB technology names) to the short tech labels used by the cost dicts above.

#### Gotchas

- The short tech labels (`coal_large_sub`, `gas_cc`, etc.) are only meaningful inside this file. They aren't used anywhere else in PowerGenome.
- If the GUI adds a new-build technology to `startup_fuel_use` (e.g. `NaturalGas_1-on-1`), Pass 4 of the resources cell will auto-add an entry to `eia_atb_tech_map` for it. You shouldn't need to edit this file directly.

See [Switch-US-PG-ReEDS docs: startup costs](https://github.com/switch-model/Switch-USA-PG-ReEDS/blob/main/pg/settings/startup_costs.yml).

In [105]:
ref_data = load_yml(REF_SETTINGS_DIR / "startup_costs.yml")
gui_data = load_yml(GUI_SETTINGS_DIR / "startup_costs.yml")

sc_settings = copy.deepcopy(ref_data)
overridden, added = [], []
for k, v in gui_data.items():
    if k in sc_settings:
        if sc_settings[k] != v:
            sc_settings[k] = v
            overridden.append(k)
    else:
        sc_settings[k] = v
        added.append(k)

print("\n=== startup_costs.yml ===")
print(f"  Ref: {len(ref_data)} keys, GUI: {len(gui_data)} keys → Merged: {len(sc_settings)} keys")
if overridden: print(f"  GUI overrode: {overridden}")
if added:      print(f"  GUI added: {added}")
save_yml(sc_settings, OUTPUT_SETTINGS_DIR / "startup_costs.yml")



=== startup_costs.yml ===
  Ref: 8 keys, GUI: 8 keys → Merged: 8 keys
  GUI overrode: ['startup_fuel_use', 'new_build_startup_costs']
  ✓ Wrote startup_costs.yml (8 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days/startup_costs.yml


## 8. `distributed_gen.yml` — Rooftop solar from NREL ReEDS

Two concerns live in this file: (1) how distributed (rooftop) solar is represented in the model, and (2) the T&D loss rate that Switch applies to loads. Both come out of the same NREL ReEDS distributed PV dataset in `pg_data/NREL-dist-gen/`.

For (1), PowerGenome can either subtract distributed PV from load before the model sees it, or represent it as a zero-cost resource the model dispatches. The reference uses the latter so that distributed PV curtailment is visible in the solution.

For (2), `avg_distribution_loss` becomes Switch's `local_td_loss_rate` in `load_zones.csv`. This is always applied, regardless of the (1) choice.

#### Parameters

| Key | What it does |
|---|---|
| `distributed_gen_fn` | Filename of the parquet file in `DISTRIBUTED_GEN_DATA` (env.yml). Current: `nrel_reeds_distr_pv_2025.2.0.parquet`. If you update the underlying data, change this. |
| `distributed_gen_scenario` | Which ReEDS scenario to use (`mid_case`, `high_case`, `low_case`, etc.). Controls the total deployed rooftop capacity trajectory. |
| `dg_as_resource` | If `true`, distributed PV becomes a zero-cost, must-run resource. If `false`, it's subtracted from load before dispatch. Leave `true` unless you have a specific reason. |
| `avg_distribution_loss` | Loss factor between the bulk transmission system and end-use loads. In Switch this becomes `load_zones.local_td_loss_rate` and is applied to all load regardless of how distributed generation is represented. Reference value `0.0453` (~4.5%) is close to ReEDS's ~5.3% and EIA's reported ~5% total T&D losses. |

See [Switch-US-PG-ReEDS docs: distributed generation](https://github.com/switch-model/Switch-USA-PG-ReEDS/blob/main/pg/settings/distributed_gen.yml).

In [34]:
ref_data = load_yml(REF_SETTINGS_DIR / "distributed_gen.yml")
gui_data = load_yml(GUI_SETTINGS_DIR / "distributed_gen.yml")

dg_settings = copy.deepcopy(ref_data)
overridden, added = [], []
for k, v in gui_data.items():
    if k in dg_settings:
        if dg_settings[k] != v:
            dg_settings[k] = v
            overridden.append(k)
    else:
        dg_settings[k] = v
        added.append(k)

print("\n=== distributed_gen.yml ===")
print(f"  Ref: {len(ref_data)} keys, GUI: {len(gui_data)} keys → Merged: {len(dg_settings)} keys")
if overridden: print(f"  GUI overrode: {overridden}")
if added:      print(f"  GUI added: {added}")
save_yml(dg_settings, OUTPUT_SETTINGS_DIR / "distributed_gen.yml")



=== distributed_gen.yml ===
  Ref: 4 keys, GUI: 2 keys → Merged: 4 keys
  ✓ Wrote distributed_gen.yml (4 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/distributed_gen.yml


## 9. `resource_tags.yml` — How generators are classified for dispatch and policy



Tags are how PowerGenome and Switch figure out what each generator can and cannot do. They're integer or decimal flags written as columns in `gen_info.csv`, and they control three things:

1. **Dispatch behavior** — Is this a thermal unit (`THERM`), a variable renewable (`VRE`), a battery (`STOR`), a must-run (`MUST_RUN`), a unit-commitment-modeled plant (`Commit`)?
2. **Capacity contributions** — What fraction of nameplate capacity counts toward each region's planning reserve margin (`CapRes_1`, `CapRes_2`)?
3. **Policy eligibility** — Which technologies count toward state RPS/CES standards (`ESR_*`), which face national build limits (`MaxCapTag_*`), which face minimums (`MinCapTag_*`)?

The full file is hundreds of lines long because it uses dozens of policy tags. Most of those (the `ESR_*` and `MinCapTag_*` ones) are not configured here — they're machine-generated by `make_emission_policies.py` from ReEDS state-policy data and end up in `regional_resource_tags.yml`. This file mostly handles the dispatch and capacity-contribution tags.

#### How technology matching works

PowerGenome matches technology names using **case-insensitive substring matching** — and matches are *cumulative*, with shorter (more general) patterns applied first and longer (more specific) patterns overriding them. So:

```yaml
model_tag_values:
  VRE:
    Wind: 1            # Applied first — matches LandbasedWind, OffShoreWind, every wind variant
    OffShoreWind: 0    # Applied second — overrides only the offshore wind techs
```

Underscores and whitespace in technology names are ignored during matching, so `NaturalGas`, `Natural Gas`, and `Natural_Gas` are all equivalent. This means a pattern like `NaturalGas` will match `Natural Gas Fired Combined Cycle`, `NaturalGas_CC`, and any other variant. Use the longest unambiguous prefix to avoid accidental matches.

All values default to whatever `default_model_tag` says (`0` in our setup). You only need to set entries that should be non-zero.

#### Top-level structure

| Key | What it does |
|---|---|
| `model_tag_names` | The complete list of every tag column that will appear in `gen_info.csv`. Every tag PowerGenome or Switch needs to read must be listed here, even if it's blank everywhere. The notebook merges this list from both the reference and the GUI versions to make sure neither side drops a tag the other expects. |
| `default_model_tag` | Default value for any (technology, tag) combination not explicitly set in `model_tag_values`. We use `0`. Almost never changed. |
| `model_tag_values` | The main payload. Nested dict of `{tag_name: {technology_pattern: value}}`. See the per-tag breakdown below. |
| `regional_tag_values` | Optional override layer for region-specific tag values. Not set in this file directly — it lives in `regional_resource_tags.yml`, which is machine-generated by `make_emission_policies.py`. |

#### Tags used in this study, grouped by purpose

**Dispatch behavior (binary flags):**

| Tag | What it means |
|---|---|
| `THERM` | Thermal generator (coal, gas, nuclear, biomass, oil, peakers). Subject to unit-commitment constraints if `Commit` is also set. |
| `VRE` | Variable renewable (wind, solar PV/CSP, distributed generation). Output is determined by exogenous capacity factor profiles. |
| `Num_VRE_Bins` | How many capacity-factor bins to use for the VRE clustering. Set to `1` for each VRE technology in this study. |
| `STOR` | Storage (batteries, pumped hydro). |
| `FLEX` | Flexible demand / virtual generator (used for `load_growth` and `us_exports` from `flexible_load.yml`). |
| `HYDRO` | Conventional hydroelectric. Special-cased by Switch's hydro modules. |
| `MUST_RUN` | Must-run resources (small hydro, geothermal, biomass, imports). Forced to run at their profile level. |
| `Commit` | Subject to unit-commitment constraints (startup costs, min uptime/downtime, etc.). Only meaningful if Switch is using its commitment module. |

**Build and retirement controls:**

| Tag | What it means |
|---|---|
| `New_Build` | Three-state flag: `1` = new build allowed, `0` = default (allowed but not preferred), `-1` = new build forbidden. The reference disables new builds for biomass, conventional hydro, geothermal, small hydro, nuclear (existing), distributed gen, and the demand-side virtual generators (imports/exports in this case). New nuclear is allowed via the more specific `Nuclear_Nuclear: 1` pattern. |
| `Can_Retire` | Whether economic retirement is allowed. Set to `1` for fossil techs (coal, gas, biomass, peakers, petroleum) and `0` for nuclear (which is policy-protected from economic retirement in this study). |

**Capacity contribution to planning reserves:**

| Tag | What it means |
|---|---|
| `CapRes_1`, `CapRes_2` | Decimal value (0–1) representing the fraction of nameplate capacity that counts toward each capacity reserve requirement. The two are identical in our setup but Switch supports defining two independent reserve programs. Reference values: thermal = 0.9, wind/solar/hydro = 0.8, battery/pumped storage = 0.95. These are blunt approximations of effective load carrying capability (ELCC). |

**Operational parameters that happen to be expressed as tags:**

| Tag | What it means |
|---|---|
| `gen_forced_outage_rate` | Decimal forced outage rate by technology. Sourced from NERC GADS statistical brochures (2020–24) for conventional plants and NERC GADS Wind for wind. Values like `Coal: 0.1019`, `Combined Cycle: 0.0491`, `Nuclear: 0.0224`. |
| `gen_max_annual_availability` | Annual capacity factor cap by technology. Defaults to `1.0` for all techs (`""` matches everything), but overridden to `0.0` for the `load_growth` and `us_exports` virtual generators so they don't add capacity to the system. |
| `gen_can_provide_spinning_reserves` | Whether the generator can supply spinning reserves. Default `1` for everything; `0` for the demand-response virtual generators since they have no annual availability. |

**Capacity build limits (national policy levers):**

These are toggled by `make_emission_policies.py` to enforce study-level constraints on tech buildout. The actual MW limits live in `emission_policies_fn`. We don't actually use these in this study.

| Tag | What it means |
|---|---|
| `MaxCapTag_WindGrowth` | Marks all wind technologies as subject to a national wind buildout limit. |
| `MaxCapTag_SolarGrowth` | Marks all solar PV (existing and new) as subject to a national solar buildout limit. |
| `MaxCapTag_NuclearGrowth` | Marks all nuclear as subject to a national nuclear buildout limit. |
| `MaxCapTag_GasTurbineSupply` | Marks all gas turbines (CC, CT, existing/planned/new) as subject to a national gas turbine supply limit. |
| `MaxCapTag_Ban` | Default `0` for all techs. Used as a "no build" hammer in some scenarios. |

#### What the notebook does

Three things happen in the code cell below:

1. **Generic merge of GUI into reference.** The GUI may have added tag names or tag values that the reference doesn't know about; we override matching keys with GUI values and add new ones. This is the same pattern as `transmission.yml`, `startup_costs.yml`, and `distributed_gen.yml`.
2. **Union of `model_tag_names`.** Tag names are merged across both files (preserving order, dropping duplicates). This matters because each side might list tags the other doesn't, and dropping any of them breaks downstream `gen_info.csv` writing.
3. **Forced `Imports` under `MUST_RUN`.** The `Imports` virtual technology represents fixed power flows between this study's modeled regions and the rest of the US/Canada/Mexico. PowerGenome doesn't tag it as must-run by default, but Switch needs it to be must-run so the imports show up at their scheduled levels. The notebook injects this entry if it's not already there.

#### Gotchas

- **Don't use short patterns casually.** A line like `Wind: 1` under `ESR_CA_rps` would mark every wind technology in the country as eligible for California's RPS. Use longer prefixes (`OffshoreWind`, `LandbasedWind_Class3`) when a tag should apply to a subset.
- **Substring precedence is order-independent.** PowerGenome doesn't use document order to decide which pattern wins — it uses pattern length. The longest matching substring wins regardless of where it appears in the file.
- **Tags not in `model_tag_names` get a warning, not an error.** PowerGenome will flag them but won't drop them. Conversely, missing tags raise `ResourceTagError` at runtime.
- **`regional_tag_values` lives in `regional_resource_tags.yml`, not here.** That file is auto-generated by `make_emission_policies.py` from ReEDS state-policy data. If you need to override a tag in a specific region manually, you'd add it there, not in `resource_tags.yml`.

#### A note on the GUI's ESR data

The PowerGenome GUI also generates Energy Share Requirement data, but in a different format than the lab's reference uses. The GUI defines `ESR_1` through `ESR_N` (generic numbered tags) in `model_tag_names`, embeds `regional_tag_values` directly inside `resource_tags.yml`, uses `rectable.csv` and `allowed_techs.csv` to determine which technologies satisfy each ESR program, and drops an `emission_policies.csv` into `extra_inputs/`.

The notebook's generic merge loop pulls all of this — including the GUI's `regional_tag_values` block — into `my_settings/resource_tags.yml`. For Switch studies in this lab, where state ESR policies aren't typically engaged, these GUI-generated entries are harmless padding. The lab's separate state-named scheme (`ESR_CA_rps`, `ESR_NY_ces`, etc.) lives in `regional_resource_tags.yml` and is disabled by default in the cell that handles that file (see section 15).

See [PG docs: resource tags](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/resource-tags/).

In [35]:
# resource_tags.yml is merged from reference and GUI. Six things happen:
#   1. Generic merge (override matching keys, add new ones from GUI)
#   2. Union of model_tag_names from both sides (drop duplicates, preserve order)
#   3. Force-add Imports to MUST_RUN so import flows show up at scheduled levels
#   4. Force-add Biopower/Coal_IGCC new-build techs to THERM (missing from GUI export)
#   5. Force-add CO2_Capture_Rate tag with capture rates for CCS techs
#   6. Force-add every atb_new_gen tech to New_Build (and THERM if thermal)
# See markdown above for what each tag means and how technology matching works.
ref_data = load_yml(REF_SETTINGS_DIR / "resource_tags.yml")
gui_data = load_yml(GUI_SETTINGS_DIR / "resource_tags.yml")

rt_settings = copy.deepcopy(ref_data)
overridden, added = [], []
for k, v in gui_data.items():
    if k in rt_settings:
        if rt_settings[k] != v:
            rt_settings[k] = v
            overridden.append(k)
    else:
        rt_settings[k] = v
        added.append(k)

# Step 2: union model_tag_names from both files (preserve order, drop duplicates).
# Either side might define tags the other lacks; dropping any breaks gen_info.csv.
ref_tag_names = ref_data.get("model_tag_names", []) or []
gui_tag_names = gui_data.get("model_tag_names", []) or []
merged_tag_names = list(dict.fromkeys(ref_tag_names + gui_tag_names))
if "CO2_Capture_Rate" not in merged_tag_names:
    merged_tag_names.append("CO2_Capture_Rate")
    print("  Added CO2_Capture_Rate to model_tag_names")
rt_settings["model_tag_names"] = merged_tag_names

# Step 3: ensure the Imports virtual technology is tagged MUST_RUN.
# Imports represent fixed power flows from neighboring regions and need to
# appear in dispatch at their scheduled levels.
model_tags = rt_settings.get("model_tag_values", {})
must_run_tags = model_tags.get("MUST_RUN", {})
if not isinstance(must_run_tags, dict):
    must_run_tags = {}
if "Imports" not in must_run_tags:
    must_run_tags["Imports"] = 1
    print("  Added Imports: 1 under MUST_RUN")
else:
    print("  Imports already exists under MUST_RUN")
model_tags["MUST_RUN"] = must_run_tags

# Step 4: ensure new-build Biopower and Coal_IGCC techs are tagged THERM.
# These are listed in atb_new_gen but the GUI export omits them from
# model_tag_values, which causes check_resource_tags to crash with
# "resource ... does not have any assigned resource tags".
therm_tags = model_tags.get("THERM", {})
if not isinstance(therm_tags, dict):
    therm_tags = {}
required_therm = [
    "Biopower_Dedicated_Moderate",
    "Coal_IGCC_Moderate",
    "Coal_IGCC-90%-CCS_Moderate",
]
for tech in required_therm:
    if tech not in therm_tags:
        therm_tags[tech] = 1
        print(f"  Added {tech}: 1 under THERM")
    else:
        print(f"  {tech} already exists under THERM")

# Step 5: ensure CCS techs have CO2_Capture_Rate tagged with their capture fractions.
# Required so emissions accounting reflects the post-capture CO2 stream rather
# than treating CCS units identically to their unabated counterparts.
ccs_capture_rates = {
    "Coal_IGCC-90%-CCS": 0.9,
    "NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS": 0.95,
}
capture_tags = model_tags.get("CO2_Capture_Rate", {})
if not isinstance(capture_tags, dict):
    capture_tags = {}
for tech, rate in ccs_capture_rates.items():
    if tech not in capture_tags:
        capture_tags[tech] = rate
        print(f"  Added {tech}: {rate} under CO2_Capture_Rate")
    else:
        print(f"  {tech} already exists under CO2_Capture_Rate")
model_tags["CO2_Capture_Rate"] = capture_tags

# Step 6: ensure every atb_new_gen entry is tagged New_Build, and tagged THERM
# if it's a thermal technology. atb_new_gen is the canonical list of buildable
# new-generation techs; pulling from it here keeps New_Build in sync with the
# actual candidate set instead of relying on the GUI export to stay current.
# Entries are [technology, techdetail, cost_case, size_mw]; the resource name
# convention used elsewhere in the pipeline is "{technology}_{techdetail}_{cost_case}".
resources_data = load_yml(OUTPUT_SETTINGS_DIR / "resources.yml")
atb_new_gen = resources_data.get("atb_new_gen", []) or []

# Prefix match: any tech whose `technology` field starts with one of these
# is treated as thermal and gets a THERM tag in addition to New_Build.
THERMAL_PREFIXES = ("NaturalGas", "Coal", "Nuclear", "Biopower", "Oil", "Petroleum")

new_build_tags = model_tags.get("New_Build", {})
if not isinstance(new_build_tags, dict):
    new_build_tags = {}

nb_added, therm_added_step6 = [], []
for entry in atb_new_gen:
    if not isinstance(entry, list) or len(entry) < 3:
        continue
    technology, techdetail, cost_case = entry[0], entry[1], entry[2]
    tech_name = f"{technology}_{techdetail}_{cost_case}"

    if tech_name not in new_build_tags:
        new_build_tags[tech_name] = 1
        nb_added.append(tech_name)

    if str(technology).startswith(THERMAL_PREFIXES) and tech_name not in therm_tags:
        therm_tags[tech_name] = 1
        therm_added_step6.append(tech_name)

if nb_added:
    print(f"  Added {len(nb_added)} techs under New_Build: {nb_added}")
else:
    print("  All atb_new_gen techs already present under New_Build")
if therm_added_step6:
    print(f"  Added {len(therm_added_step6)} techs under THERM (from atb_new_gen): {therm_added_step6}")

model_tags["New_Build"] = new_build_tags
model_tags["THERM"] = therm_tags

rt_settings["model_tag_values"] = model_tags

print("\n=== resource_tags.yml ===")
print(f"  Ref: {len(ref_data)} keys, GUI: {len(gui_data)} keys → Merged: {len(rt_settings)} keys")
if overridden: print(f"  GUI overrode: {overridden}")
if added:      print(f"  GUI added: {added}")
print(f"  Final model_tag_names ({len(merged_tag_names)} tags)")

save_yml(rt_settings, OUTPUT_SETTINGS_DIR / "resource_tags.yml")

  Added CO2_Capture_Rate to model_tag_names
  Added Imports: 1 under MUST_RUN
  Added Biopower_Dedicated_Moderate: 1 under THERM
  Added Coal_IGCC_Moderate: 1 under THERM
  Added Coal_IGCC-90%-CCS_Moderate: 1 under THERM
  Added Coal_IGCC-90%-CCS: 0.9 under CO2_Capture_Rate
  Added NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS: 0.95 under CO2_Capture_Rate
  Added 13 techs under New_Build: ['UtilityPV_Class1_Moderate', 'Nuclear_Nuclear - Large_Moderate', 'Utility-Scale Battery Storage_Lithium Ion_Moderate', 'Biopower_Dedicated_Moderate', 'Coal_IGCC_Moderate', 'Coal_IGCC-90%-CCS_Moderate', 'Geothermal_HydroBinary_Moderate', 'LandbasedWind_Class4_Moderate', 'NaturalGas_2-on-1 Combined Cycle (H-Frame)_Moderate', 'NaturalGas_2-on-1 Combined Cycle (H-Frame) 95% CCS_Moderate', 'NaturalGas_Combustion Turbine (F-Frame)_Moderate', 'OffShoreWind_Class3_Moderate', 'OffShoreWind_Class12_Moderate']
  Added 4 techs under THERM (from atb_new_gen): ['Nuclear_Nuclear - Large_Moderate', 'NaturalGas_

## 10. `demand.yml` — Load profiles and demand growth

This file controls where PowerGenome gets hourly electricity demand profiles from, how it handles load growth into future years, and what electrification scenario applies. This notebook builds `demand.yml` from scratch (it's not produced by the GUI and we don't copy it from the reference) because the reference version is ~250 lines of mostly commented-out IPM-region mappings we don't use.

#### How load profiles work in this setup

PowerGenome has two ways to get hourly load profiles into a model:

1. **Database-sourced profiles** — Pull hourly loads from a named table in `pg_db`. This is what we use. Profiles come from the `load_curves_nrel_reeds` table, which contains 2007–2013 weather years shifted to 2020–2050 under NREL's mid-case electrification assumptions.
2. **User CSV profiles** — Point at a `regional_load_fn` CSV with pre-built profiles. We don't use this here. If you want to override the ReEDS profiles with your own, add `regional_load_fn: your_file.csv.zip` and change `regional_load_source` to `USER` for those regions.

Load growth happens in two layers. The base layer applies AEO growth rates between the historical profile year and your model years (controlled by `growth_scenario`, `load_eia_aeo_year`, and `regular_load_growth_start_year`). The second layer is handled separately in `flexible_load.yml`, where we add ICF 2025–35 growth as flexible demand so the model can optionally interrupt it.

#### Parameters

| Key | What it does |
|---|---|
| `load_source_table_name` | Dict mapping a source label to a pg_db table name. We use `{reeds: load_curves_nrel_reeds}` to pull from the ReEDS load curves table. You can define multiple sources and mix them across regions via `regional_load_source`. |
| `regional_load_source` | Either a single source label (used for all regions) or a dict of `{source: [region1, region2]}`. We use the single-label form `reeds` because all our regions come from the same ReEDS dataset. Set a region's source to `USER` if you want it to come from `regional_load_fn` instead. |
| `regular_load_growth_start_year` | The historical year after which PowerGenome starts applying AEO growth rates. Profiles from earlier years are first brought forward to this year. We use `2020` because the ReEDS profiles are already shifted to a 2020 baseline. |
| `growth_scenario` | Which AEO scenario to pull growth rates from (e.g. `REF2021` for the 2021 reference case, `HIGHMACRO` for high growth, `LOWMACRO` for low growth). Only matters if your model year is outside the range already present in the load source table. |
| `load_eia_aeo_year` | Which AEO edition the growth rates come from. We use `2021` because that's what the ReEDS profiles were built against. |
| `electrification` | Name of the electrification scenario. Matches a value in the header of `regional_load_fn` if one is provided. We use `base`, which is a no-op label since the ReEDS profiles already bake in mid-case electrification. |

#### When to change

- **Different ReEDS vintage**: update `load_source_table_name` to point at the newer table.
- **Custom load profiles**: add `regional_load_fn` and switch `regional_load_source` to `USER` for the affected regions.
- **Sensitivity on growth assumptions**: change `growth_scenario`. This only matters if your model year is beyond what's already in the ReEDS profile dataset.

See [PG docs: demand settings](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/demand/?h=demand#demand-data).

In [36]:
# demand.yml is built from scratch rather than copied from reference.
# See markdown above for what each key does and when to change it.
demand_data = {
    "load_source_table_name": {"reeds": "load_curves_nrel_reeds"},
    "regional_load_source":            "reeds",
    "regular_load_growth_start_year":  2020,
    "growth_scenario":                 "REF2021",
    "load_eia_aeo_year":               2021,
    "electrification":                 "base",
}

print("\n=== demand.yml (built from scratch) ===")
save_yml(demand_data, OUTPUT_SETTINGS_DIR / "demand.yml")


=== demand.yml (built from scratch) ===
  ✓ Wrote demand.yml (6 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/demand.yml


## 11. `scenario_management.yml` — The scenario configuration system



This is the most important file in the notebook from a researcher's perspective. It's where you actually define **what cases get built and what makes each case different from the others**. Every other settings file in this notebook defines a single baseline; `scenario_management.yml` is what lets you fan that baseline out into many model runs that vary fuel prices, time resolution, retirement rules, carbon policy, and so on without manually maintaining N copies of every settings file.

It works in concert with `extra_inputs/scenario_inputs.csv` (referenced from `extra_inputs.yml` as `scenario_definitions_fn`). The CSV defines the *cases* — which combinations of settings get built — and the YAML defines the *menu* of named values each setting can take.

#### The two-file system, by example

Here's a row from `scenario_inputs.csv`:

```csv
case_id,year,time_series,load_growth_flexibility,load_growth,allow_retirement,policies,resource_limits,fuel_price_forecast
s4x1,2030,s4x1,firm,icf,yes,current,none,hist5
```

Each column after `case_id` and `year` is a **dimension** — a knob you can turn. Each cell value is the *name of a choice* for that dimension. Reading this row: "Build a case called `s4x1` for the year 2030, using the `s4x1` time-series clustering, `firm` load-growth flexibility, the `icf` load growth source, retirement allowed, the `current` policy regime, no resource limits, and the `hist5` historical fuel price forecast."

Now look in `scenario_management.yml`. Under `settings_management.all_years`, you'll find one block per dimension:

```yaml
settings_management:
  all_years:
    time_series:
      s4x1:
        time_domain_periods: 4
        time_domain_days_per_period: 1
      s10x5:
        time_domain_periods: 10
        time_domain_days_per_period: 5
      ...
    fuel_price_forecast:
      hist5:
        # use historical coal and gas prices as forecast
        ...
      aeo2025:
        eia_series_fuel_names:
          coal: STC
          naturalgas: NG
        aeo_fuel_scenarios:
          coal: reference
          naturalgas: reference
```

When `pg_to_switch.py` builds the `s4x1` case for 2030, it walks through every column in the CSV row, looks up that name in the corresponding YAML dimension, and applies the listed settings overrides on top of the baseline you built in this notebook. So `time_series: s4x1` swaps in 4 single-day clusters; `fuel_price_forecast: hist5` swaps in historical fuel prices instead of AEO; and so on.

#### Where the dimensions come from

The dimensions in this file are not magic — they're just the column names of `scenario_inputs.csv` minus `case_id` and `year`. The current setup defines:

| Dimension | What it controls | Example values |
|---|---|---|
| `time_series` | Time clustering: how many representative periods × days per period | `p1` (1 period × 1 day for fastest testing), `s4x1`, `s10x5`, `s20x1` |
| `load_growth` | Whether to apply ICF 2025–35 load growth on top of frozen ReEDS profiles | `none`, `icf` |
| `load_growth_flexibility` | How interruptible load growth is treated by the optimizer | `none`, `firm`, `flex_001` |
| `allow_retirement` | Whether thermal plants can retire economically | `yes`, `no` |
| `resource_limits` | National caps on new buildout for specific technologies | `none`, `high_gas`, `low_gas`, `increased_ambition` |
| `policies` | Carbon and clean-energy policy regime | `current`, `decarb` |
| `fuel_price_forecast` | Where coal and gas prices come from | `hist5`, `aeo2025` |

Each value here corresponds to a sub-block under that dimension in `scenario_management.yml` listing the actual settings to override.

#### Year-specific blocks

In addition to `all_years`, the file has blocks for specific years (`2024:`, `2025:`, …, `2050:`) that apply to *every* case running in that year. These exist mainly to gate which ATB technologies are available — for example, the 2024 and 2025 blocks set `atb_new_gen: []` (disabling new builds entirely, for historical-period cases), while 2026–2029 allow gas turbines and onshore wind but not offshore wind or new nuclear, and 2030+ allow everything. The year blocks also set up offshore wind minimum requirements that ramp up over time.

You normally don't need to touch the year blocks unless you're changing the assumed buildout timeline for a major technology.

#### How to add a new case

The quickest way to extend the study is to add a row to `scenario_inputs.csv` using values that already exist as menu entries in `scenario_management.yml`. For example, to add a "no retirement, decarb policy" version of `s4x1` in 2030, append this line:

```csv
s4x1_decarb_no_retire,2030,s4x1,firm,icf,no,decarb,none,hist5
```

That's it. No notebook re-run needed (since `scenario_management.yml` and `scenario_inputs.csv` aren't modified by this notebook), and `pg_to_switch.py --case-id s4x1_decarb_no_retire` will pick it up next time it runs.

#### How to add a new value to an existing dimension

Say you want a new fuel price forecast called `aeo2030_high_oil`. You'd:

1. Add a sub-block under `settings_management.all_years.fuel_price_forecast` in `scenario_management.yml`:
```yaml
   aeo2030_high_oil:
     fuel_eia_aeo_year: 2030
     aeo_fuel_scenarios:
       distillate: high_price
       naturalgas: reference
       coal: reference
```
2. Reference it in a new row in `scenario_inputs.csv` by putting `aeo2030_high_oil` in the `fuel_price_forecast` column.

#### How to add a new dimension

This is the heaviest change of the three. To add a brand-new knob like `transmission_buildout`:

1. Add a `transmission_buildout` column to `scenario_inputs.csv` and fill it in for every existing row.
2. Add a `transmission_buildout` block under `settings_management.all_years` in `scenario_management.yml` listing each named value and the settings it overrides.

`pg_to_switch.py` reads the dimensions dynamically from the CSV columns, so no code change is needed.

#### Important: why this notebook copies the file unchanged

This notebook doesn't modify `scenario_management.yml` because doing so would couple your scenario design to the GUI export step. The file is meant to live longer than any single GUI run — you'll iterate on scenarios many times without re-running the GUI. Edit it directly in your favorite editor, version-control it alongside the rest of the lab's research code, and re-run `pg_to_switch.py` to see the effects.

The same applies to `scenario_inputs.csv`. It lives in `extra_inputs/` and is referenced (not modified) by everything in this notebook.

#### Gotchas

- **Case names must be unique within `scenario_inputs.csv`**, but the same `case_id` can appear in multiple year rows. That's how you build multi-year cases (see how `s4x1` has rows for 2024, 2029, 2030, and 2035).
- **Every column value must exist as a sub-block in the corresponding dimension.** If you put `fuel_price_forecast: nymex2030` in the CSV but there's no `nymex2030` block under `fuel_price_forecast` in the YAML, `pg_to_switch.py` will fail when building that case.
- **Settings overrides are merged, not replaced wholesale.** A `fuel_price_forecast` override that only sets `aeo_fuel_scenarios.coal` will inherit `aeo_fuel_scenarios.naturalgas` from the baseline `fuels.yml`. This is usually what you want, but it can produce surprising behavior if you forget that other keys are still active.
- **`all_years` overrides are applied first, then year-specific blocks.** If both apply to the same setting, the year-specific value wins.
- **The `policies: decarb` value sets `carbon_cost_dollar_per_tco2: 200`** in addition to swapping `emission_policies_fn`. If you're sensitive to the exact carbon price, make sure you know which policy regime your case is using.

See [PG docs: scenario management](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/scenario-management/) and [PG docs: how-to run multi-scenario studies](https://powergenome.github.io/PowerGenome/v0.8-beta/how-to/run-scenarios/).

In [37]:
# scenario_management.yml is copied from the reference unchanged. The auto-add
# cell below can append a new time_series block to my_settings/scenario_management.yml
# (and a matching row to scenario_inputs.csv) if your model_year includes years
# that aren't already in the CSV.
sm_data = load_yml(REF_SETTINGS_DIR / "scenario_management.yml")

# --- Ensure allow_retirement is registered in settings_management ---
# scenario_inputs.csv has an allow_retirement column, but PG warns if the
# parameter isn't declared under settings_management. Adding an empty (~)
# entry under all_years silences the warning without changing behavior.
sm_block = sm_data.setdefault("settings_management", {})
all_years_block = sm_block.setdefault("all_years", {})
if "allow_retirement" not in all_years_block:
    all_years_block["allow_retirement"] = None  # serializes as `~`
    print("  + added allow_retirement: ~ under settings_management.all_years")
else:
    print("  · allow_retirement already present under settings_management.all_years")
    
print("\n=== scenario_management.yml (from reference) ===")
save_yml(sm_data, OUTPUT_SETTINGS_DIR / "scenario_management.yml")

  · allow_retirement already present under settings_management.all_years

=== scenario_management.yml (from reference) ===
  ✓ Wrote scenario_management.yml (1 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/scenario_management.yml


### Optional: auto-add a new case to `scenario_inputs.csv` and `scenario_management.yml`

If the model_definition cell flagged any missing years, this cell can append a new case definition for them. It does two things:

1. **Adds rows to `pg/extra_inputs/scenario_inputs.csv`** — one row per missing year, all sharing the same case_id and the same default scenario configuration
2. **Adds a matching `time_series` block to `pg/my_settings/scenario_management.yml`** — using the actual `time_domain_periods` and `time_domain_days_per_period` from your `time_clustering.yml`, so the three files (CSV, scenario_management, time_clustering) stay consistent

The new case_id is `my_case_s<P>x<D>` by default (where `P` and `D` come from `time_clustering.yml`). If that name is already taken in the CSV or YAML, the cell appends `_v2`, `_v3`, etc.

**This cell modifies files outside `my_settings/`.** It edits both `scenario_inputs.csv` (in `pg/extra_inputs/`) and `scenario_management.yml` (in `my_settings/`, which the previous cell wrote). The cell is gated by `APPLY_CHANGES` — set it to `True` only when you're ready to commit. The dry-run prints exactly what would be added so you can review first.

Edit `DEFAULT_ROW` in the cell if you want different defaults for your auto-added case (e.g., a different fuel price forecast or policy regime).

In [38]:
# ============================================================
# Set APPLY_CHANGES = True to actually write to the files
# Leave False to dry-run and see what would be added
# ============================================================
APPLY_CHANGES = True
# ============================================================

# Default scenario configuration for the auto-added case.
# - load_growth: 'none' is correct here because the ReEDS load source
#   table is shifted to 2020-2050 and contains projected demand for every
#   year out to 2050; PG uses those profiles as-is for any year that
#   already exists in the table, with no growth multiplier needed.
# - time_series gets filled in below from time_clustering.yml; don't set
#   it here.
DEFAULT_ROW = {
    "load_growth_flexibility":  "firm",
    "load_growth":              "none",
    "allow_retirement":         "yes",
    "policies":                 "current",
    "resource_limits":          "none",
    "fuel_price_forecast":      "hist5",
}

import pandas as pd
from ruamel.yaml import YAML

if not missing_years:
    print("✓ No missing years; nothing to do")
elif tc_periods is None or tc_days is None:
    print("⚠ time_clustering.yml does not set time_domain_periods or "
          "time_domain_days_per_period.")
    print("  This cell only handles the standard clustered case. If you're "
          "using reduce_time_domain: false, add the case manually.")
else:
    sm_path = OUTPUT_SETTINGS_DIR / "scenario_management.yml"
    if not sm_path.exists():
        print(f"⚠ {sm_path} not found.")
        print("  Run the scenario_management.yml cell (section 10) first, then re-run this cell.")
    else:
        scen_df = pd.read_csv(scen_inputs_path)

        # Use ruamel.yaml round-trip mode to preserve comments and ordering
        yaml_rt = YAML(typ="rt")
        yaml_rt.preserve_quotes = True
        yaml_rt.indent(mapping=2, sequence=4, offset=2)
        with open(sm_path) as f:
            sm_data = yaml_rt.load(f)

        # Pick a case_id (used as both case_id in CSV and time_series name in YAML).
        # Convention: my_case_s<P>x<D>, then _v2, _v3, ... if taken in either file.
        ts_block = sm_data["settings_management"]["all_years"]["time_series"]
        existing_cases = set(scen_df["case_id"].unique())
        existing_ts    = set(ts_block.keys())

        base_name = f"my_case_s{tc_periods}x{tc_days}"
        new_case = base_name
        suffix = 2
        while new_case in existing_cases or new_case in existing_ts:
            new_case = f"{base_name}_v{suffix}"
            suffix += 1
        if new_case != base_name:
            print(f"⚠ '{base_name}' already taken; using '{new_case}' instead")

        # Build the new CSV rows
        new_rows = [
            {
                "case_id":     new_case,
                "year":        year,
                "time_series": new_case,
                **DEFAULT_ROW,
            }
            for year in missing_years
        ]
        new_rows_df = pd.DataFrame(new_rows, columns=scen_df.columns)

        print(f"=== New case: '{new_case}' ===")
        print(f"\nRows to append to {scen_inputs_path.name}:")
        print(new_rows_df.to_string(index=False))

        # Build the new time_series block for scenario_management.yml
        new_ts_block = {
            "time_domain_periods":         tc_periods,
            "time_domain_days_per_period": tc_days,
        }

        print(f"\nBlock to add to {sm_path.name} under settings_management.all_years.time_series:")
        print(f"  {new_case}:")
        for k, v in new_ts_block.items():
            print(f"    {k}: {v}")

        if APPLY_CHANGES:
            # Write the CSV
            updated_csv = pd.concat([scen_df, new_rows_df], ignore_index=True)
            updated_csv.to_csv(scen_inputs_path, index=False)

            # Insert the new time_series block. ruamel preserves the rest
            # of the file (comments, ordering, year-specific blocks below).
            ts_block[new_case] = new_ts_block
            with open(sm_path, "w") as f:
                yaml_rt.dump(sm_data, f)

            print(f"\n✓ Wrote {len(new_rows)} rows to {scen_inputs_path.name}")
            print(f"  → file://{scen_inputs_path.resolve()}")
            print(f"✓ Added time_series block '{new_case}' to {sm_path.name}")
            print(f"  → file://{sm_path.resolve()}")
            print()
            print("Run with:")
            year_flags = " ".join(f"--year {y}" for y in missing_years)
            print(f"  python pg_to_switch.py pg/my_settings <out> --case-id {new_case} {year_flags}")
        else:
            print(f"\n⚠ DRY RUN — no changes written.")
            print(f"  Set APPLY_CHANGES = True at the top of this cell and re-run.")

✓ No missing years; nothing to do


## 12. `switch.yml` — Switch-specific settings and PG→Switch glue



This is the densest overlap between PowerGenome settings and the Switch conversion code. Unlike most settings files, `switch.yml` is read by **both** PowerGenome (during data loading) and `pg_to_switch.py` (during conversion to Switch input files). Several keys exist only for the Switch conversion step and have no effect inside PowerGenome itself.

Understanding which code path reads which key matters here — if you add a key that `pg_to_switch.py` uses but PowerGenome doesn't, don't expect to see any effect until the Switch conversion step runs.

#### Parameters present in the reference file

| Key | What it does |
|---|---|
| `gen_info_extra_columns` | Dict mapping Switch column names (keys) to PowerGenome column names (values). When `pg_to_switch.py` writes `gen_info.csv`, it renames these PG columns to their Switch names and passes them through. Reference sets `gen_max_annual_availability` and `gen_can_provide_spinning_reserves` (both self-mapping). Add entries here if you have custom PG columns you want to surface in Switch. |
| `discount_rate` | Real (inflation-adjusted) discount rate written to Switch's `financials.csv`. Default in `pg_to_switch.py` is `0.03` if omitted. The reference file's comment discusses three candidate values: ~6.4% (Lazard 2025 assumptions with 60/40 debt/equity), ~5% (NREL ATB average across technologies), and 3% (the value in the reference file itself). |
| `interest_rate` | Real interest rate written to `financials.csv`. Default `0.05` if omitted. The reference file now sets this explicitly to `0.05` (standard assumption for our Switch setup, close to the NREL ATB cross-technology average). |
| `prr_enforcement_timescale` | How strictly Switch enforces planning reserve requirements. `peak_load` only enforces them at the timeseries with highest coincident load (what the reference uses). `all_timepoints` enforces them at every hour (more conservative, slower). Default `all_timepoints` if omitted. Written to `planning_reserve_requirements.csv`. |
| `model_adjustment_scripts` | Dict of post-processing scripts that `pg_to_switch.py` runs after building each case directory. Each entry has a description (the key), a `script` path (relative to the settings folder), optional `arguments`, and an optional `order` (default 5). Scripts are sorted by `order` and run in sequence. The reference uses three: `Add reserve info` (order 1), `Add extreme day` (order 10), and `Define scenarios` (order 11). |
| `carbon_cost_dollar_per_tco2` | Single scalar carbon price used as the default escape price for all CO2 cap programs when writing `carbon_policies_regional.csv`. Reference uses `33.43`, the average of California ($35.21) and Washington ($31.64) 2024 cap-and-trade clearing prices. |

#### Optional keys that `pg_to_switch.py` will also read

These aren't in the reference file but are valid additions if you need the functionality. `pg_to_switch.py` checks for them and either skips the feature (if not set) or uses them:

| Key | What it does |
|---|---|
| `operation_model` | If truthy, `pg_to_switch.py` drops all new-build generators and keeps only existing ones. Used for pure operational dispatch runs without capacity expansion. Rarely used — normally operation models are built from a solved capacity-planning model instead. |
| `switch_module_list` | Path to a Switch module list file. If set, `pg_to_switch.py` adds `--module-list <path>` to each scenario's command line. Lets you swap in a custom subset of Switch modules per scenario. |
| `trans_fixed_om_fraction` | Fraction of transmission capital cost charged annually as fixed O&M. Default `0.0` (no fixed O&M). Written to Switch's transmission parameters table. |
| `alternative_carbon_slack` | List of alternative carbon prices. For each price in the list, `pg_to_switch.py` writes an additional `carbon_policies_regional.{price}.csv` and creates a matching scenario command line. Lets you run carbon-price sensitivities without manually duplicating scenarios. |

#### What the notebook currently overrides

The notebook overrides `discount_rate` from the reference's `0.03` to `0.05`, keeping it aligned with `interest_rate`. Change it in the code cell below if your study uses different financial assumptions (e.g., Lazard's 6.4% real cost of capital, or the reference's original 3%).

#### Gotchas

- **`gen_info_passthrough` appears in a stale docstring in `conversion_functions.py` but is never actually read from settings.** The working mechanism for passing custom columns to Switch is `gen_info_extra_columns`.
- **`carbon_cost_dollar_per_tco2` is applied uniformly across all regions**, even though Switch itself supports per-region prices. If you need region-specific carbon prices, you'll need to modify `pg_to_switch.py`'s carbon policy CSV building logic directly.
- **`model_adjustment_scripts` order matters, and PowerGenome's default sorting is wrong.** PowerGenome sorts entries by key length, which is almost never what you want. `pg_to_switch.py` re-sorts by the explicit `order` field (default 5) before running them, so always set `order` on every entry unless you specifically want the default-5 group.

See [Switch-USA-PG-ReEDS docs: switch settings](https://github.com/switch-model/Switch-USA-PG-ReEDS/blob/main/pg/settings/switch.yml).

In [44]:
# switch.yml is loaded from reference. We override discount_rate, disable the
# three reference adjust scripts, and add the clean_inputs adjust step.
switch_data = load_yml(REF_SETTINGS_DIR / "switch.yml")

switch_data["discount_rate"] = 0.05

# Add the new clean_inputs script (this one we DO want as a real entry).
reeds_inputs_dir_path = "/Users/melek/Documents/GitHub/ReEDS-2.0/inputs"

def to_repo_relative(path) -> str:
    """Return a repo-relative POSIX path, or the original absolute string if outside REPO_ROOT."""
    p = Path(path)
    if p.is_absolute():
        try:
            return p.relative_to(REPO_ROOT).as_posix()
        except ValueError:
            return str(p)
    return p.as_posix()

if not Path(reeds_inputs_dir_path).is_dir():
    raise FileNotFoundError(f"ReEDS inputs directory not found: {reeds_inputs_dir_path}")

switch_data["model_adjustment_scripts"]["Clean inputs"] = {
    "script": "../../adjust/clean_inputs.py",
    "args": (
        # f'--reeds-inputs "{reeds_inputs_dir_path}" '
        f'--settings "{to_repo_relative(OUTPUT_SETTINGS_DIR)}"'
    ),
    "order": 20,
}

# (… your optional-keys comment block stays here unchanged …)

out_path = OUTPUT_SETTINGS_DIR / "switch.yml"
print("\n=== switch.yml (from reference, discount_rate overridden) ===")
save_yml(switch_data, out_path)

# ---- Post-process: disable the three reference adjust scripts in-place ----
# Comment out their `script:` / `order:` lines and put ~ on the description.
# String-based so we don't disturb any of the surrounding ruamel-preserved
# comments inside model_adjustment_scripts.
import re

DISABLE = ("Add reserve info", "Add extreme day", "Define scenarios")

text = out_path.read_text()
for name in DISABLE:
    # 1. description line:  "  Add reserve info:" -> "  Add reserve info: ~"
    text = re.sub(
        rf"^(\s*){re.escape(name)}:\s*$",
        rf"\1{name}: ~",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    # 2. its script: line -> commented
    text = re.sub(
        rf"^(\s*)(script:\s*\.\./\.\./adjust/\S+\.py)\s*$",
        rf"\1# \2",
        text,
        count=1,
        flags=re.MULTILINE,
    )
# 3. order: lines under those entries -> commented (only the small ones, 1/10/11)
for order_val in ("1", "10", "11"):
    text = re.sub(
        rf"^(\s*)(order:\s*{order_val})\s*$",
        rf"\1# \2",
        text,
        count=1,
        flags=re.MULTILINE,
    )

out_path.write_text(text)
print(f"  ✓ Disabled {len(DISABLE)} reference adjust scripts ({', '.join(DISABLE)}) (script/order commented out)")


=== switch.yml (from reference, discount_rate overridden) ===
  ✓ Wrote switch.yml (6 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/switch.yml
  ✓ Disabled 3 reference adjust scripts (Add reserve info, Add extreme day, Define scenarios) (script/order commented out)


## 13. `extra_inputs.yml` — Paths to supplementary CSV files



PowerGenome pulls some data from CSVs in `pg/extra_inputs/` rather than from its databases. This file just lists filenames. Every file referenced here must exist or PowerGenome will crash at load time.

| Key | What it points to |
|---|---|
| `input_folder` | Subfolder name relative to the settings file. Always `extra_inputs` in this setup. |
| `scenario_definitions_fn` | CSV defining which parameter combinations each `case_id` uses. Paired with `scenario_management.yml`. |
| `emission_policies_fn` | Per-region per-year emission caps and carbon prices. Generated by `make_emission_policies.py`, which pulls RPS/CES data from ReEDS and adds RGGI/CA/NY/WA caps. |
| `capacity_limit_spur_fn` | Max buildable capacity and spur line distance for each renewable resource. |
| `demand_segments_fn` | Value-of-lost-load (VOLL) by demand segment. Used for scarcity pricing. |
| `misc_gen_inputs_fn` | Misc. generator attribute overrides that don't fit elsewhere. |
| `user_transmission_costs` | CSV of inter-regional transmission expansion costs. Patched by the cell below to add annuity columns PG v0.7 requires. |
| `user_transmission_constraints_fn` | Existing inter-regional transfer capacity. The reference uses REFS2009 (more capacity) over NARIS (less) because NARIS can't serve p119 in some timepoints. |
| `co2_pipeline_cost_fn` | CO2 transport cost percentiles for CCS pathways. |
| `plant_region_map_fn` | Maps EIA plant IDs to ReEDS regions. Used to assign existing generators to model regions. |
| `user_region_geodata_fn` | Shapefile-like name for region geometry, used for map outputs. |

#### When to edit

Only touch `user_transmission_costs` if you're swapping in different network data, or `emission_policies_fn` if you're running with or without state RPS/CES. The rest are set-and-forget.

See [PG docs: extra inputs](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/data-tables/?h=capacity_limit_spur_fn#policy-and-cost-tables).

#### Viewing cluster assignments with `extra_outputs_path`

Separately from the file references above, PowerGenome supports a setting called `extra_outputs_path` that writes debugging CSVs showing which real EIA plant ended up in which cluster. When set (or when left unset, which defaults to an `extra_outputs/` subfolder inside each case directory), PowerGenome emits one CSV per (region, technology) combination listing the plant IDs, capacities, and cluster assignments.

We use this for two things:

1. **Debugging unexpected groupings.** If a cluster's heat rate or capacity factor looks wrong, you can trace which specific plants went into it and see whether PowerGenome's clustering logic did something surprising.
2. **Looking up generator coordinates.** Clusters in `gen_info.csv` are abstract (a cluster is an aggregation, not a real physical location), but `extra_outputs` lets you walk back from a cluster to its member EIA plant IDs, then join to PUDL's plant metadata to get latitude/longitude for every real generator in the cluster. This is how we'll tie fossil clusters to actual plant locations later in the pipeline.

To enable it explicitly (rather than relying on the default), add this to `extra_inputs.yml` or to any scenario's `settings_management`:

```yaml
extra_outputs: extra_outputs
```

Note that PowerGenome will not create this directory itself — the path must exist before the run starts, or PG will crash when it tries to write the first cluster CSV. The setup script handles this by calling `mkdir(parents=True, exist_ok=True)` on the resolved path before writing `extra_inputs.yml`.

See [PG docs: viewing cluster assignments](https://powergenome.github.io/PowerGenome/v0.8-beta/explanation/clustering/?h=extra#viewing-cluster-assignments).

In [55]:
print("\n=== extra_inputs.yml (from reference) ===")
print("  Points to CSVs in extra_inputs/; must exist")
ei_data = load_yml(REF_SETTINGS_DIR / "extra_inputs.yml")
transmission_file_name = "network_costs_46r_eastern-ercot-western_st.csv"
transmission_path = GUI_SETTINGS_DIR.parent / "data" / transmission_file_name

if not transmission_path.is_file():
    raise FileNotFoundError(f"Transmission costs file not found: {transmission_path}")
ei_data['user_transmission_costs'] = str(to_repo_relative(transmission_path))
extra_outputs_path = OUTPUT_SETTINGS_DIR.parent / "extra_outputs"
extra_outputs_path.mkdir(parents=True, exist_ok=True)
ei_data["extra_outputs"] = str(to_repo_relative(extra_outputs_path))

save_yml(ei_data, OUTPUT_SETTINGS_DIR / "extra_inputs.yml")


=== extra_inputs.yml (from reference) ===
  Points to CSVs in extra_inputs/; must exist
  ✓ Wrote extra_inputs.yml (12 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/extra_inputs.yml


### Patch transmission costs CSV

The GUI CSV is missing `interconnect_annuity_mw`, `total_interconnect_annuity_mw`, and `dollar_year` — all required by PG v0.7's `transmission_tables()`. We add the annuity columns using a CRF derived from `wacc=0.044` and `investment_years=60` (≈0.0476, matches the reference file), and set `dollar_year=2018` per the reference. We also dedupe bidirectional pairs (the GUI lists each pair twice with identical costs) since Switch expects one direction per pair.

In [49]:
import pandas as pd

csv_path = GUI_SETTINGS_DIR.parent / "data" / transmission_file_name
df = pd.read_csv(csv_path)

# CRF from transmission.yml (wacc=0.044, investment_years=60)
# Matches the ~0.0476 implied by the reference CSV
wacc = 0.044
n = 60
crf = wacc * (1 + wacc)**n / ((1 + wacc)**n - 1)

df["interconnect_annuity_mw"] = df["interconnect_cost_mw"] * crf
df["total_interconnect_annuity_mw"] = df["total_interconnect_cost_mw"] * crf
df["dollar_year"] = 2018  # confirmed from reference CSV

# Deduplicate bidirectional pairs — Switch expects one direction per pair.
# Verified symmetric: cost/loss values match across directions.
df["_pair"] = df.apply(
    lambda r: tuple(sorted([r["start_region"], r["dest_region"]])), axis=1
)
before = len(df)
df = df.drop_duplicates(subset="_pair", keep="first").drop(columns="_pair")

df.to_csv(csv_path, index=False)

print(f"Patched {before} → {len(df)} rows (CRF={crf:.5f}, dollar_year=2018)")

Patched 93 → 93 rows (CRF=0.04759, dollar_year=2018)


## 14. `flexible_load.yml` — Demand response and load growth as flexible resources



PowerGenome can create "virtual generators" that represent flexible demand: loads that can be shifted in time or curtailed during tight hours. Two use cases live in this file:

1. **Demand response (DR) resources**: e.g. shiftable EV charging, smart thermostats, industrial interruptible loads
2. **Load growth treated as flexible**: extra load on top of historical profiles that the model can optionally interrupt

We use this file primarily for the second case. The ReEDS load profiles in `demand.yml` are frozen at 2023 levels, so we add ICF 2025–35 growth as a flexible resource in `flexible_load.yml`. That way the model can choose whether to serve the growth (business as usual) or curtail it (treating the growth as interruptible to reduce system cost).

#### How it works

PowerGenome reads `demand_response_fn` (a CSV with hourly profiles per resource, year, scenario, and region), filters rows to the scenario named by `demand_response`, and creates one virtual generator per region per entry in `flexible_demand_resources`. Each virtual generator gets:

- `Existing_Cap_MW = fraction_shiftable × profile.max()`
- `New_Build = -1` (always, then overridden by the `New_Build` tag in `resource_tags.yml`)
- `Fuel = "No_fuel"`
- Max hourly output equal to the profile, so it can represent curtailment

The flexible resources also need to be tagged `FLEX = 1` and `New_Build = -1` in `resource_tags.yml`, and their `gen_max_annual_availability` set there. The reference `resource_tags.yml` already handles this for `us_exports` and `load_growth`.

#### Parameters

| Key | What it does |
|---|---|
| `demand_response` | Scenario name matching a value in row 3 of `demand_response_fn`. We use `base`. |
| `demand_response_fn` | CSV file with hourly profiles. Must have 4 header rows: (1) resource name matching a key in `flexible_demand_resources`, (2) model year, (3) scenario name, (4) model region. Then one row per load hour with no explicit index. We use `load_adjustments.csv.zip`. |
| `flexible_demand_resources` | Nested dict: `{year: {resource_name: {fraction_shiftable, parameter_values}}}`. `fraction_shiftable` is the fraction of peak load that can be interrupted. `parameter_values` gets passed through to the virtual generator's row in `gen_info.csv`. |

#### What's in the reference

The reference version only defines `us_exports` for model years 2024, 2029, 2030, and 2035, all with `fraction_shiftable: 0.0`. The zero is deliberate: `us_exports` represents scheduled exports to Canada/Mexico, which are treated as a fixed load (not a flexible resource), but PowerGenome still wants a dummy virtual generator entry so it won't warn about missing definitions.

To add a real DR resource, copy the `us_exports` structure, change the name to something present in your `demand_response_fn`, set a nonzero `fraction_shiftable`, and add parameters like `Max_Flexible_Demand_Delay` and `Flexible_Demand_Energy_Eff` under `parameter_values`. See the commented-out example at the top of the original reference file for the full structure.

#### Gotchas

- **`flexible_demand_resources` must match `resource_tags.yml`.** If you add a DR resource here, make sure `resource_tags.yml > model_tag_values` has entries for `FLEX` and `New_Build` for it, or PowerGenome will reset those tags to 0.
- **Setting `gen_max_annual_availability` under `parameter_values` doesn't work as you'd expect.** It applies as a default to *all* generators, not just the one you're defining. Set it in `resource_tags.yml > model_tag_values > gen_max_annual_availability` using the specific resource name as the key instead.
- **Profile maximum matters.** `Existing_Cap_MW` is computed from `profile.max()`, so if the profile has a spike, the virtual generator will be oversized relative to the average shiftable load. Smooth out spikes in the source CSV if this causes trouble.

#### Coverage past 2030

The reference `demand_response_fn` (`load_adjustments.csv.zip`) only defines `us_exports` profiles through 2030. For studies that extend past 2030, we use an extended version (`load_adjustments_to2050.csv.zip`) generated by the commented-out streaming script in the cell below the next one. That script appends 2031–2050 us_exports columns by copying the 2023 block forward — safe
because each region's us_exports values are constant across 2023–2030 in the reference file. Run it once whenever the upstream `load_adjustments.csv` is regenerated. See the script's header comment for assumptions and edit `NEW_YEARS` to change the end year. For model years that still fall outside the data's coverage (or if you choose not to extend the file), the code below sets
`flexible_demand_resources[year] = None`, which PowerGenome handles by skipping DR for that year.

See [Switch-US-PG-ReEDS docs: flexible demand](https://github.com/switch-model/Switch-USA-PG-ReEDS/blob/main/pg/settings/flexible_load.yml).

In [114]:
# # ─────────────────────────────────────────────────────────────────────────────
# # Extend us_exports columns in extra_inputs/load_adjustments.csv from 2030 to 2050
# # ─────────────────────────────────────────────────────────────────────────────
# # What this does:
# #   Streams the (very large, ~500 MB) load_adjustments.csv line by line and
# #   writes a new file load_adjustments_to2050.csv(.zip) that adds 20 new years
# #   (2031–2050) to the us_exports section. For each new year it appends a copy
# #   of the year-2023 us_exports block at the end of every row. The load_growth
# #   columns are left completely untouched. A pandas-based sanity check at the
# #   bottom re-reads the new file and verifies that all 12 us_exports regions
# #   are still constant across 2023–2050.
# #
# # Why it exists:
# #   load_adjustments.csv only defines us_exports for 2023–2030. When extending
# #   a Switch-USA-PG-ReEDS study horizon past 2030 (e.g. out to 2050), this file
# #   needs corresponding entries for the later years or downstream steps fail.
# #   The file is too large to load into pandas comfortably, so this uses csv
# #   streaming with O(1) memory.
# #
# # When to use it:
# #   Run this once whenever you need a load_adjustments.csv that covers years
# #   beyond 2030. Edit NEW_YEARS if you want a different end year. Re-run from
# #   scratch if the upstream load_adjustments.csv is regenerated.
# #
# # Key assumptions (verified for the current file — re-check if inputs change):
# #   1. us_exports values for each of the 12 regions
# #      (p1, p11, p127, p129, p134, p18, p3, p36, p37, p42, p61, p65) are
# #      CONSTANT across 2023–2030, so copying the 2023 block forward is exact.
# #      If a future input file has time-varying us_exports, this script will
# #      silently flatten that variation — pick a different extension rule.
# #   2. load_growth columns do NOT need extension here (they are handled by
# #      whatever load path is configured under scenario_management.yml, or by
# #      a separate process). This script does not touch them.
# #   3. The new us_exports year-blocks are APPENDED to the end of each row
# #      rather than interleaved into the existing us_exports section. The
# #      get_region() reader (and any reader that matches by header metadata
# #      rather than column position) handles this fine.
# #   4. Header layout is 4 rows: metric (sparse / merged-cell style),
# #      year, scenario, region.
# # ─────────────────────────────────────────────────────────────────────────────
# import csv
# import os
# import zipfile
# from collections import OrderedDict

# INPUT      = "/Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/extra_inputs/load_adjustments.csv"
# OUTPUT_CSV = "/Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/extra_inputs/load_adjustments_to2050.csv"
# OUTPUT_ZIP = OUTPUT_CSV + ".zip"

# NEW_YEARS = list(range(2031, 2051))   # 2031..2050 inclusive (20 years)

# def ffill(row):
#     out, last = [], ""
#     for c in row:
#         if c.strip():
#             last = c.strip()
#         out.append(last)
#     return out

# with open(INPUT, "r", newline="") as f_in, \
#      open(OUTPUT_CSV, "w", newline="") as f_out:

#     reader = csv.reader(f_in)
#     writer = csv.writer(f_out)

#     h1 = next(reader)   # metric (load_growth / us_exports), sparse
#     h2 = next(reader)   # year
#     h3 = next(reader)   # scenario (base)
#     h4 = next(reader)   # region

#     h1_filled = ffill(h1)

#     # Find every us_exports column
#     us_cols = [i for i, m in enumerate(h1_filled) if m == "us_exports"]
#     print(f"Found {len(us_cols)} us_exports columns")

#     # Group by year so we can grab the first year's block as our template
#     by_year = OrderedDict()
#     for i in us_cols:
#         y = int(h2[i])
#         by_year.setdefault(y, []).append(i)
#     years_present = list(by_year.keys())
#     template_cols = by_year[years_present[0]]   # cols for year 2023
#     regions_in_order = [h4[i].strip() for i in template_cols]
#     print(f"Years present : {years_present}")
#     print(f"Regions ({len(regions_in_order)}): {regions_in_order}")
#     print(f"Template cols : {len(template_cols)} (should equal n_regions)")

#     n_regions = len(regions_in_order)
#     n_new = n_regions * len(NEW_YEARS)

#     # Build the new header cells to append (one block per new year)
#     new_h1 = ["us_exports"] * n_new
#     new_h2 = [str(y) for y in NEW_YEARS for _ in range(n_regions)]
#     new_h3 = [h3[us_cols[0]]] * n_new
#     new_h4 = regions_in_order * len(NEW_YEARS)

#     writer.writerow(h1 + new_h1)
#     writer.writerow(h2 + new_h2)
#     writer.writerow(h3 + new_h3)
#     writer.writerow(h4 + new_h4)

#     # Stream data rows; append 20 copies of the year-2023 us_exports block
#     for i, row in enumerate(reader, start=5):
#         block = [row[c] for c in template_cols]
#         writer.writerow(row + block * len(NEW_YEARS))
#         if i % 10000 == 0:
#             print(f"  processed {i:,} rows")

# print(f"Wrote {OUTPUT_CSV}")
# print("Zipping...")
# with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     zf.write(OUTPUT_CSV, arcname=os.path.basename(OUTPUT_CSV))
# print(f"Done -> {OUTPUT_ZIP}")

# # Optional: remove the unzipped csv to save space
# # os.remove(OUTPUT_CSV)
# import pandas as pd

# INPUT = "/Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/extra_inputs/load_adjustments_to2050.csv"

# def get_region(region, metric=None, nrows=10000, path=INPUT):
#     """
#     Return a DataFrame of values for a given region across all years.

#     Parameters
#     ----------
#     region : str         e.g. "p1"
#     metric : str or None e.g. "load_growth" / "us_exports". If None, returns both.
#     nrows  : int         number of data rows to read (default 1000)
#     """
#     # 1. Read just the 4 header rows to locate matching columns
#     header = pd.read_csv(path, header=None, nrows=4)
#     metrics   = header.iloc[0].ffill()   # forward-fill merged-cell style headers
#     years     = header.iloc[1]
#     scenarios = header.iloc[2]
#     regions   = header.iloc[3]

#     mask = (regions == region)
#     if metric is not None:
#         mask &= (metrics == metric)

#     col_positions = mask[mask].index.tolist()
#     if not col_positions:
#         raise ValueError(f"No columns found for region={region!r}, metric={metric!r}")

#     # 2. Read only those columns, skipping the 4 header rows
#     df = pd.read_csv(
#         path,
#         header=None,
#         skiprows=4,
#         usecols=col_positions,
#         nrows=nrows,
#     )

#     # 3. Label the columns
#     if metric is not None:
#         df.columns = [years[i] for i in col_positions]
#         df.columns.name = "year"
#     else:
#         df.columns = [f"{metrics[i]}_{years[i]}" for i in col_positions]

#     return df

# # Final Check
# for region in ['p1', 'p11', 'p127', 'p129', 'p134', 'p18', 'p3', 'p36', 'p37', 'p42', 'p61', 'p65']:
#     df_region = get_region(region, metric="us_exports")
#     all_identical = (df_region.nunique(axis=1) == 1).all()
#     print(f"{region}: all years identical? {all_identical}")
#     print(df_region.head())
#     print()

In [50]:
# flexible_load.yml is copied from the reference. It defines us_exports as a
# dummy virtual generator (fraction_shiftable=0) across all model years to
# suppress PowerGenome warnings. See the markdown above for how to add real
# demand-response resources.
fl_data = load_yml(REF_SETTINGS_DIR / "flexible_load.yml")

# Use the extended load_adjustments file that covers 2023–2050.
# The reference file only goes through 2030; the extended version is built by
# the commented-out streaming script below. See the markdown's "Coverage past
# 2030" subsection for details.
extended_fn = "load_adjustments_to2050.csv.zip"
extended_path = REF_SETTINGS_DIR.parent / "extra_inputs" / extended_fn
if not extended_path.exists():
    raise FileNotFoundError(
        f"Extended us_exports file not found: {extended_path}\n"
        f"Run the commented-out streaming script below to generate it from "
        f"load_adjustments.csv, or revert demand_response_fn to the reference "
        f"'load_adjustments.csv.zip' if you don't need years past 2030."
    )
fl_data["demand_response_fn"] = extended_fn

# Ensure every model year has a us_exports dummy entry in flexible_demand_resources.
# Without this, PowerGenome crashes with KeyError in create_demand_response_gen_rows
# when a model year is missing from this dict. us_exports is a dummy resource
# (fraction_shiftable=0) that exists only to suppress PowerGenome warnings; the
# extended load_adjustments_to2050.csv.zip provides the underlying profiles
# through 2050.
default_flex_entry = {
    "us_exports": {
        "fraction_shiftable": 0.0,
        "parameter_values": {"New_Build": -1},
    }
}

for y in md_settings["model_year"]:
    if y not in fl_data["flexible_demand_resources"] or fl_data["flexible_demand_resources"][y] is None:
        fl_data["flexible_demand_resources"][y] = copy.deepcopy(default_flex_entry)
        
# Sort by year for readability
fl_data["flexible_demand_resources"] = dict(
    sorted(fl_data["flexible_demand_resources"].items())
)

print("\n=== flexible_load.yml (from reference) ===")
print(f"demand_response_fn: {fl_data['demand_response_fn']}")
print(f"flexible_demand_resources years: {list(fl_data['flexible_demand_resources'])}")
save_yml(fl_data, OUTPUT_SETTINGS_DIR / "flexible_load.yml")


=== flexible_load.yml (from reference) ===
demand_response_fn: load_adjustments_to2050.csv.zip
flexible_demand_resources years: [2024, 2029, 2030, 2035, 2050]
  ✓ Wrote flexible_load.yml (3 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/flexible_load.yml


## 15. `regional_resource_tags.yml` — State policy eligibility (disabled in this notebook)



**You should not hand-edit this file.** Most of this markdown is so you know what it is when you stumble across it.

#### What it would contain

In the lab's reference setup, this is a 19,000-line auto-generated artifact with a single top-level key, `regional_tag_values`, defining per-region overrides for state RPS, CES, and offshore wind eligibility tags (`ESR_*`, `MinCapTag_*`). The reference file's header says it explicitly: "managed by `make_emission_policies.py` — the marked sections will be overwritten each time that is run." The contents are derived from ReEDS state-policy data.

#### What this notebook writes instead

A one-line minimal version: `regional_tag_values: null`. This disables all state-level tag overrides without removing the `ESR_*` and `MinCapTag_*` tags from `model_tag_names` (they fall back to their defaults of 0 everywhere).

We do this because Switch capacity expansion runs in this lab generally don't engage with state Energy Share Requirements. ESR policies are common in GenX-style modeling. Copying a 19,000-line file just to never read its contents is wasteful, and `pg_to_switch.py` explicitly supports the disabled state: it writes an empty `rps_generators.csv` and issues a warning, which is the intended behavior.

#### Why hand-editing the full file is hard even if you wanted to

The reference file's keys are raw p-regions (`p1`, `p2`, …, `p134`). When you aggregate p-regions into model regions via `region_aggregations`, the policy assignments would need to be remapped — and the remap is genuinely hard, because state RPS/CES eligibility is *state-specific*. An aggregated region spanning multiple states has no clean answer to "which state's policy applies to a generator here?"

#### To enable state policies for your study

If you really do need state policies active:

1. Copy the reference file manually: `cp pg/settings/regional_resource_tags.yml pg/my_settings/`
2. Decide how to assign each aggregated region to a single state's policy regime (this is a judgment call, not a mechanical translation)
3. Remap the p-region keys to your aggregated model region names, or regenerate the file with `make_emission_policies.py`

This is really outside what this notebook does automatically.

#### A note on the GUI

The PowerGenome GUI generates ESR data in a completely different format: it uses generic `ESR_1` through `ESR_N` tag names with a trading-zone lookup from `rectable.csv` and `allowed_techs.csv`, embeds `regional_tag_values` directly inside `resource_tags.yml` (not as a separate file), and drops an `emission_policies.csv` into `extra_inputs/`. The notebook's resource_tags merge loop absorbs the GUI's `regional_tag_values` into `my_settings/resource_tags.yml`.

See [PG docs: regional tag values](https://powergenome.github.io/PowerGenome/v0.8-beta/reference/settings/resource-tags/#regional_tag_values).

In [51]:
# regional_resource_tags.yml in the lab's reference is a 19,000-line auto-
# generated file from make_emission_policies.py defining per-region state
# RPS/CES/MinCapTag eligibility. Switch studies in this lab don't engage
# with state ESR policies, so we write a minimal version that disables all
# state-level tag overrides instead of copying the full reference.
#
# pg_to_switch.py explicitly handles this case: it writes an empty
# rps_generators.csv and issues a warning, which is the intended behavior.
#
# To enable state policies for your study, copy the reference file manually
# and remap the p-region keys to your aggregated model regions. See markdown
# above for the full procedure.
rrt_data = {"regional_tag_values": None}

print("\n=== regional_resource_tags.yml (minimal: state policies disabled) ===")
save_yml(rrt_data, OUTPUT_SETTINGS_DIR / "regional_resource_tags.yml")


=== regional_resource_tags.yml (minimal: state policies disabled) ===
  ✓ Wrote regional_resource_tags.yml (1 keys) → file:///Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS/pg/settings_10weeks_7days_PROFILE_CLUSTERS/regional_resource_tags.yml


---

## Final verification

Check that `my_settings/` has all the files it needs and compare key counts against the reference.

In [52]:
print("=== my_settings/ contents ===")
out_files = sorted(OUTPUT_SETTINGS_DIR.glob("*.yml"))
for f in out_files:
    data = load_yml(f)
    print(f"  {f.name:35s} {len(data):>3d} keys")

print(f"\nTotal: {len(out_files)} files")

# Compare against reference
ref_files = sorted(REF_SETTINGS_DIR.glob("*.yml"))
ref_names = {f.name for f in ref_files}
out_names = {f.name for f in out_files}

missing = ref_names - out_names
extra = out_names - ref_names

if missing:
    print(f"\n⚠ Missing from output: {sorted(missing)}")
if extra:
    print(f"\n⚠ Extra in output (not in ref): {sorted(extra)}")
if not missing and not extra:
    print(f"\n✓ Output has same files as reference")

# Key count comparison
print(f"\n=== Key count comparison ===")
out_all = load_all_keys(OUTPUT_SETTINGS_DIR)
ref_all = load_all_keys(REF_SETTINGS_DIR)

out_k = set(out_all.keys())
ref_k = set(ref_all.keys())

missing_keys = ref_k - out_k
extra_keys = out_k - ref_k

print(f"  Reference: {len(ref_k)} keys")
print(f"  Output:    {len(out_k)} keys")
if missing_keys:
    print(f"  ⚠ Missing keys ({len(missing_keys)}):")
    for k in sorted(missing_keys):
        fname, _ = ref_all[k]
        print(f"      [{fname}] {k}")
if extra_keys:
    print(f"  Extra keys ({len(extra_keys)}): {sorted(extra_keys)}")
if not missing_keys:
    print(f"  ✓ All reference keys present")

=== my_settings/ contents ===
  demand.yml                            6 keys
  distributed_gen.yml                   4 keys
  env.yml                               6 keys
  extra_inputs.yml                     12 keys
  flexible_load.yml                     3 keys
  fuels.yml                            15 keys
  model_definition.yml                 10 keys
  regional_resource_tags.yml            1 keys
  resource_tags.yml                     3 keys
  resources.yml                        45 keys
  scenario_management.yml               1 keys
  startup_costs.yml                     8 keys
  switch.yml                            6 keys
  time_clustering.yml                   5 keys
  transmission.yml                      4 keys

Total: 15 files

✓ Output has same files as reference

=== Key count comparison ===
  Reference: 124 keys
  Output:    129 keys
  Extra keys (5): ['extra_outputs', 'interconnect_capex_mw', 'retirement_ages', 'user_fuel_price', 'user_fuel_usd_year']
  ✓ All referen

---

## What to do next: running the pipeline



With `my_settings/` populated and verified above, you're ready to run the PowerGenome → Switch pipeline. The conversion happens via `pg_to_switch.py` from the Switch-USA-PG-ReEDS repo.

### Running pg_to_switch.py

From the repo root:

```bash
cd <REPO_ROOT>
python pg_to_switch.py pg/my_settings switch_inputs --case-id <your_case>
```

The first positional argument is the settings folder (or a single YAML file), the second is where results go. Use `--case-id <id>` once per case you want to prepare. To prepare multiple cases, repeat the flag:

```bash
python pg_to_switch.py pg/my_settings switch_inputs \
    --case-id case1 \
    --case-id case2 \
    --year 2030
```

`--year` similarly takes one value per flag and lets you build inputs for only specific years rather than every year in `model_year`.

Other flags worth knowing about:

- `--myopic` builds separate single-year models that chain forward (one per planning year), instead of a single multi-year foresight model.
- `--case-index N` selects the Nth case from the available list. Useful for parallel jobs.

### If pg_to_switch.py errors out

Common failure modes, roughly in order of frequency:

1. **Missing or wrong paths in `env.yml`** — PG can't find PUDL, the PG database, or the resource group profiles. Re-check section 1.
2. **`KeyError` in `startup_fuel_use` → `eia_atb_tech_map`** — A new-build technology in `startup_fuel_use` isn't in `eia_atb_tech_map`. Pass 4 of section 3 (`resources.yml`) auto-handles known patterns; if an unknown tech shows up, the cell prints a list of unresolved techs at the bottom of its output. Add them manually to the GUI's `resources.yml` `eia_atb_tech_map` before re-running this notebook.
3. **Region name mismatches** — Something in `scenario_management.yml`, `extra_inputs/`, or `regional_resource_tags.yml` references a p-region (`p1`, `p2`, ...) that doesn't appear in your aggregated regions. The error message usually names the offending key.
4. **Planning year mismatches** — `model_year` in `model_definition.yml` doesn't line up with hard-coded years in `flexible_load.yml`'s `flexible_demand_resources` block, or in scenario CSVs in `extra_inputs/`. See the warning in section 2 about planning year synchronization.

### After Switch runs

The output directory (`switch_inputs/<case_id>/`) will contain `gen_info.csv`, `transmission_lines.csv`, `loads.csv`, `fuel_cost.csv`, `financials.csv`, `planning_reserve_requirements.csv`, and the rest of Switch's expected input set. From there you can run Switch directly with:

```bash
switch solve --inputs-dir switch_inputs/<case_id> --outputs-dir switch_outputs/<case_id>
```

Refer to the [Switch model documentation](http://switch-model.org/) for solver configuration, post-processing, and result analysis.

In [53]:
# Print an example pg_to_switch.py command using values from this notebook run.
# This is a convenience — see the markdown below for full CLI documentation.

import shlex

repo_root_str = str(REPO_ROOT)
settings_path = OUTPUT_SETTINGS_DIR.relative_to(REPO_ROOT)

# Decide which case_id and years to feature in the example command.
# Three possibilities:
#   1. The auto-add cell ran and APPLY_CHANGES was True → use new_case + missing_years
#   2. The auto-add cell ran but APPLY_CHANGES was False → use new_case but warn
#   3. No new case was created (missing_years was empty) → use any existing case from CSV
example_case = None
example_years = []
warning = None

try:
    if missing_years and APPLY_CHANGES:
        example_case = new_case
        example_years = missing_years
    elif missing_years and not APPLY_CHANGES:
        example_case = new_case
        example_years = missing_years
        warning = (
            "⚠ The auto-add cell ran in DRY RUN mode (APPLY_CHANGES = False).\n"
            f"  '{new_case}' is NOT in scenario_inputs.csv yet — the command\n"
            "  below will fail until you set APPLY_CHANGES = True and re-run\n"
            "  the auto-add cell."
        )
    else:
        # No new case; pick any existing case_id from the CSV as an example
        import pandas as pd
        ei_data = load_yml(REF_SETTINGS_DIR / "extra_inputs.yml")
        scen_inputs_path = (
            REF_SETTINGS_DIR.parent / ei_data["input_folder"] / ei_data["scenario_definitions_fn"]
        )
        scen_df = pd.read_csv(scen_inputs_path)
        # Pick the case_id with the most year rows — that's usually a "real" research case
        case_counts = scen_df["case_id"].value_counts()
        example_case = case_counts.index[0]
        example_years = sorted(scen_df.loc[scen_df["case_id"] == example_case, "year"].unique().tolist())
except NameError:
    # missing_years / new_case / APPLY_CHANGES don't exist yet
    # (auto-add cell wasn't run). Fall back to a fully generic example.
    example_case = "<your_case_id>"
    example_years = []

# Build the command
year_flags = " ".join(f"--year {y}" for y in example_years)
results_dir = "switch/in_46"  # convention; adjust if your study uses a different layout

cmd_parts = [
    "python pg_to_switch.py",
    str(settings_path),
    results_dir,
    f"--case-id {example_case}",
]
if year_flags:
    cmd_parts.append(year_flags)
cmd = " ".join(cmd_parts)

print("=" * 70)
print("Example command for your build")
print("=" * 70)
if warning:
    print(warning)
    print()
print(f"From {repo_root_str}, run:")
print()
print(f"  cd {repo_root_str}")
print(f"  {cmd} --myopic")
# print("    ")
print()
print("Add --myopic to build separate single-year models that chain forward.")
print("See the markdown below for the full set of flags.")

Example command for your build
From /Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS, run:

  cd /Users/melek/Documents/GitHub/Switch-USA-PG-ReEDS
  python pg_to_switch.py pg/settings_10weeks_7days_PROFILE_CLUSTERS switch/in_46 --case-id s4x1 --year 2024 --year 2029 --year 2030 --year 2035 --myopic

Add --myopic to build separate single-year models that chain forward.
See the markdown below for the full set of flags.
